# 📋 РУКОВОДСТВО ПО ИСПОЛЬЗОВАНИЮ ШАБЛОНА MODELLING

**Версия 9.0** | **С изоляцией циклов** | **Универсальный шаблон для бинарной классификации**

---

## 📊 Просмотр результатов

**Ноутбук уже выполнен.**

### Входные данные:

**Источник:** EDA-ноутбук того же проекта  
**Файлы:** `X_train_scaled.csv`, `y_train.csv`, `X_val_scaled.csv`, `y_val.csv`  
**Метаданные:** `production_metadata.json` (из EDA)

> **Примечание:** Данные не включены в репозиторий. Для самостоятельного запуска необходимо сначала выполнить ноутбуки Preprocessing и EDA.

---

## ОСНОВНЫЕ НАСТРОЙКИ [(блок 1.4)](#)

### Medical Cost

| Переменная | Описание | Пример |
|------------|----------|--------|
| `COST_FP` | Штраф за ложную тревогу (FP) - фиксированный | `1` |
| `GRID_FN_VALUES` | Значения штрафа за пропуск инсульта (FN) — перебираются в сетке | `[18, 30, 45]` |

>  Штраф за FN перебирается через сетку, штраф за FP фиксирован.

---

### 🔬 Сетка экспериментов

| Переменная | Описание | Пример |
|------------|----------|--------|
| `GRID_FN_VALUES` | Значения штрафа за FN (перебор) | `[18, 30, 45]` |
| `GRID_PENALTIES` | Типы регуляризации | `['l1', 'l2']` |
| `GRID_C_VALUES` | Сила регуляризации | `[0.1, 1, 10]` |

---

## ФИЛЬТРЫ (3 УРОВНЯ)


### Уровень 1. Технический (отсев заведомо плохих моделей)

| Что проверяем | Порог | Почему это важно |
|---------------|-------|------------------|
| **Brier Skill Score** | `> 0.1` | Модель должна быть лучше наивного прогноза |
| **Сходимость** | `✅` | Несходящаяся модель даёт нестабильные предсказания |
| **Stability Gap** | `≤ 0.10` | Train не должен быть намного лучше Val (защита от переобучения) |

---

### Уровень 2. Расширенный технический (качество и стабильность)

| Что проверяем | Порог | Почему это важно |
|---------------|-------|------------------|
| **PR-AUC (CV)** | `≥ 0.2` | Модель должна хоть как-то разделять классы |
| **CV Stability (std)** | `≤ 0.08` | Оптимальный порог должен быть стабилен по фолдам |
| **MCE** | `≤ 0.15` | Максимальная ошибка калибровки не должна быть большой |
| **ECE** | `≤ 0.05` | Средняя ошибка калибровки — модель должна быть хорошо откалибрована |
| **Reliability Slope** | `0.8 — 1.2` | Наклон калибровочной кривой должен быть близок к 1 |
| **Medical Cost CV Coef** | `≤ 0.30` | Стоимость должна быть стабильна (коэфф. вариации <30%) |

---

### Уровень 3. Клинический (бизнес-требования)

| Что проверяем | Порог | Почему это важно |
|---------------|-------|------------------|
| **Recall (VAL)** | `≥ 0.80` | Пропускать более 20% инсультов — клинически неприемлемо |
| **Recall (нижняя граница CI)** | `≥ 0.70` | Даже в худшем случае Recall не должен падать ниже 70% |
| **Precision (VAL)** | `≥ 0.05` | Хотя бы 1 спасённый из 20 тревог |
| **Selection Rate** | `≤ 40%` | Не более 40% пациентов направляем на дообследование |
| **NNI** | `≤ 12` | Чтобы найти 1 инсульт, обследуем не более 12 пациентов |
| **True Positives (VAL)** | `≥ 10` | Достаточно примеров для стабильной оценки метрик |

---

## СТАТУСЫ ЦИКЛА

| `cycle_status` | Описание |
|----------------|----------|
| `'VALID'` | Есть победитель, модель сохранена |
| `'NO_MODEL'` | Ни одна модель не прошла фильтры |

---

## 📁 РЕЗУЛЬТАТЫ

После выполнения создаётся структура:

<pre>
📁 {PROJECT_NAME}/
└── 📁 cycle_{CYCLE_NUMBER}/
    └── 📁 modelling/
        ├── 📄 final_model.joblib           # модель с калибровкой
        ├── 📄 final_metadata.json          # метрики, порог, random_state
        ├── 📄 eda_recommendations.json     # рекомендации для следующего цикла
        ├── 📄 scalers.pkl                  # копия из EDA
        ├── 📁 reports/
        │   ├── 📄 experiments_comparison.csv
        │   ├── 📄 threshold_analysis.csv
        │   └── 📄 ...
        └── 📁 plots/
            ├── 📄 confusion_matrix.png
            ├── 📄 medical_cost_curve.png
            ├── 📄 calibration_curve.png
            ├── 📄 risk_stratification.png
            └── 📄 ...

</pre>


## ВАЖНО

- Все метрики на валидации рассчитаны **до финального обучения**
- Финальная модель обучается на **train+val**
- Порог сохраняется как атрибут модели `model.threshold_` (в `final_model.joblib`)
- `random_state` автоматически подхватывается из `production_metadata.json` (EDA)

---

## СВЯЗАННЫЕ НОУТБУКИ

| Ноутбук | Назначение |
|---------|------------|
| **Preprocessing** | Загрузка, очистка, разбиение на train/val/test |
| **EDA** | Feature engineering, кодирование, масштабирование |
| **Modelling** | Обучение, эксперименты, выбор модели |
| **Cycle Comparison** | Сравнение циклов, финальное тестирование |

---

✅ **После завершения Modelling** переходим в ноутбук **Cycle_Comparison**

# СТРУКТУРА ШАБЛОНА MODELLING

**Версия 9.1** (с изоляцией циклов)

---

## ЧАСТЬ 1: НАСТРОЙКА И ЗАГРУЗКА

| Блок | Название |
|------|----------|
| 1.1 | Установка библиотек |
| 1.2 | Импорт библиотек |
| 1.3 | Определение окружения и версий |
| 1.4 | Настройки проекта |
| 1.5 | Монтирование Google Drive |
| 1.6 | Настройка путей для modelling |
| 1.7 | Загрузка метаданных из EDA |
| 1.8 | Загрузка данных (train + val) |
| 1.9 | Проверка мультиколлинеарности (VIF) |
| 1.10 | Финальная валидация данных перед обучением |

---

## ЧАСТЬ 2: ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ

| Блок | Название |
|------|----------|
| 2.1 | Функции для расчёта метрик и порогов |
| 2.2 | Функции для калибровки |
| 2.3 | Фабрика моделей |
| 2.4 | Функции визуализации |

---

## ЧАСТЬ 3: ФУНКЦИИ ДЛЯ ЭКСПЕРИМЕНТОВ

| Блок | Название |
|------|----------|
| 3.1 | Функции для кросс-валидации и стабильности |
| 3.2 | Анализ ошибок и стратификация риска |
| 3.3 | Запуск одного эксперимента с CV-оптимизацией |
| 3.4 | Автоматический выбор лучшей модели (3 уровня фильтров) |
| 3.5 | Статистическая значимость коэффициентов |
| 3.6 | Генерация сетки и запуск всех экспериментов |
| 3.7 | Форматирование вывода |

---

## ЧАСТЬ 4: ЗАПУСК ЭКСПЕРИМЕНТОВ

| Блок | Название |
|------|----------|
| 4.1 | Запуск экспериментов |
| 4.2 | Автоматический выбор лучшей модели |
| 4.3 | Сравнение победителя с Dummy |
| 4.4 | Визуализация победителя (валидация) |
| 4.5 | Проверка стабильности финалиста |
| 4.6 | Сохранение модели и метаданных |

---

## ЧАСТЬ 5: АНАЛИЗ ЛУЧШЕЙ МОДЕЛИ

| Блок | Название |
|------|----------|
| 5.1 | Статистическая значимость коэффициентов |
| 5.2 | SHAP-анализ |
| 5.3 | Анализ ошибок (FN vs TP и FP) |
| 5.4 | Распределение вероятностей |
| 5.5 | Рекомендации для EDA |
| 5.6 | Sensitivity analysis (устойчивость к удалению данных) |
| 5.7 | Permutation importance |
| 5.8 | Learning curve |

---

## РЕЗУЛЬТАТЫ

**Папка `reports/` (внутри `cycle_N/modelling/`):**
- `experiments_comparison.csv` - результаты всех экспериментов
- `risk_stratification.csv` - таблица стратификации риска
- `finalist_stability.csv` - стабильность финалиста
- `sensitivity_analysis.csv` - результаты sensitivity analysis
- `permutation_importance.csv` - важность признаков
- `learning_curve.csv` - кривая обучения
- `coefficients_stats.csv` - статистика коэффициентов
- `error_profiles.csv` - профили ошибок
- `failed_experiments.json` - список неудачных экспериментов

**Папка `plots/` (внутри `cycle_N/modelling/`):**
- `confusion_matrix.png` - матрица ошибок
- `medical_cost_curve.png` - кривая стоимости
- `recall_vs_threshold.png` - зависимость Recall от порога
- `calibration_curve.png` - калибровочная кривая
- `risk_stratification.png` - стратификация риска
- `probability_histogram.png` - гистограмма вероятностей
- `comparison_with_dummy.png` - сравнение с Dummy
- `finalist_stability.png` - график стабильности
- `learning_curve.png` - кривая обучения
- `forest_plot.png` - Forest Plot (Odds Ratios)
- `coefficient_importance.png` - важность коэффициентов
- `shap_summary.png` - SHAP summary plot
- `shap_importance.png` - SHAP feature importance

**Корень `modelling/` (внутри `cycle_N/modelling/`):**
- `final_model.joblib` - обученная модель
- `scalers.pkl` - копия скейлеров из EDA
- `final_metadata.json` - метаданные финальной модели
- `eda_recommendations.json` - рекомендации для EDA


# БЛОК 1. ЗАГРУЗКА ОКРУЖЕНИЯ

In [1]:
# =============================================================================
# 1.1 УСТАНОВКА БИБЛИОТЕК
# =============================================================================

import sys
import subprocess

required_packages = [
    ('shap', 'shap'),
    ('statsmodels', 'statsmodels'),
]

for package, import_name in required_packages:
    try:
        __import__(import_name)
        print(f"✅ {package} уже установлен")
    except ImportError:
        print(f"📦 Установка {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
        print(f"   ✅ {package} установлен")

print("✅ Все необходимые библиотеки установлены")


# =============================================================================
# 1.2 ИМПОРТЫ БИБЛИОТЕК
# =============================================================================

# СИСТЕМНЫЕ И БАЗОВЫЕ
import os
import json
import shutil
import time
import joblib
import hashlib
import warnings
import re
from datetime import datetime

# ДАННЫЕ И ВИЗУАЛИЗАЦИЯ
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import matplotlib as mpl

# Настройки визуализации
plt.rcParams['figure.figsize'] = (12, 6)
sns.set_style('whitegrid')
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 6)

# МОДЕЛИ
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier

# КРОСС-ВАЛИДАЦИЯ
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict

# МЕТРИКИ
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    fbeta_score,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    confusion_matrix,
    brier_score_loss,
    ConfusionMatrixDisplay
)

# КАЛИБРОВКА
from sklearn.calibration import (
    calibration_curve,
    CalibratedClassifierCV
)

# СТАТИСТИКА
from scipy import stats
from scipy.stats import beta
import statsmodels.api as sm

# ИНТЕРПРЕТАЦИЯ МОДЕЛЕЙ
import shap

# НАСТРОЙКИ ПРЕДУПРЕЖДЕНИЙ
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings('ignore', category=ConvergenceWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

✅ shap уже установлен
✅ statsmodels уже установлен
✅ Все необходимые библиотеки установлены


In [2]:
# =============================================================================
# 1.3 ОПРЕДЕЛЕНИЕ ОКРУЖЕНИЯ И ВЕРСИЙ (РАБОЧАЯ ВЕРСИЯ)
# =============================================================================

import sys
import os
from datetime import datetime

# ОПРЕДЕЛЕНИЕ ОКРУЖЕНИЯ
# -----------------------------------------------------------------------------
try:
    from google.colab import drive
    IN_COLAB = True
    ENV_NAME = "Google Colab"
except ImportError:
    IN_COLAB = False
    # Проверка на Kaggle
    if os.environ.get('KAGGLE_KERNEL_RUN_TYPE'):
        ENV_NAME = "Kaggle"
    else:
        ENV_NAME = "Локально"

print(f"📡 Окружение: {ENV_NAME}")

# ВЕРСИИ БИБЛИОТЕК (безопасное получение)
# -----------------------------------------------------------------------------
def safe_version(lib, name):
    """Безопасно получает версию библиотеки"""
    try:
        return lib.__version__
    except:
        return f"{name} не импортирован"

VERSIONS = {
    'python': sys.version.split()[0],
}

# Добавляем версии только импортированных библиотек
try:
    import numpy as np
    VERSIONS['numpy'] = np.__version__
except:
    pass

try:
    import pandas as pd
    VERSIONS['pandas'] = pd.__version__
except:
    pass

try:
    import sklearn
    VERSIONS['sklearn'] = sklearn.__version__
except:
    pass

try:
    import matplotlib as mpl
    VERSIONS['matplotlib'] = mpl.__version__
except:
    pass

try:
    import seaborn as sns
    VERSIONS['seaborn'] = sns.__version__
except:
    pass

try:
    import scipy
    VERSIONS['scipy'] = scipy.__version__
except:
    pass

try:
    import shap
    VERSIONS['shap'] = shap.__version__
except:
    VERSIONS['shap'] = 'unknown'

try:
    import joblib
    VERSIONS['joblib'] = joblib.__version__
except:
    pass

try:
    import statsmodels.api as sm
    VERSIONS['statsmodels'] = sm.__version__
except:
    pass

print("\n📚 Версии библиотек:")
for name, version in VERSIONS.items():
    print(f"   {name}: {version}")

print(f"\n🕐 Запуск: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# ДОПОЛНИТЕЛЬНАЯ ИНФОРМАЦИЯ
# -----------------------------------------------------------------------------
print(f"\n💻 Система: {sys.platform}")
print(f"   CPU count: {os.cpu_count() or 'N/A'}")

# Для Colab дополнительная информация
if IN_COLAB:
    try:
        from google.colab import output
        print(f"   Colab GPU: {output.eval_js('google.colab.kernel.kernel.getGpuStatus') if hasattr(output, 'eval_js') else 'не определено'}")
    except:
        pass

print("=" * 60)

📡 Окружение: Google Colab

📚 Версии библиотек:
   python: 3.12.13
   numpy: 2.0.2
   pandas: 2.2.2
   sklearn: 1.6.1
   matplotlib: 3.10.0
   seaborn: 0.13.2
   scipy: 1.16.3
   shap: 0.51.0
   joblib: 1.5.3
   statsmodels: 0.14.6

🕐 Запуск: 2026-05-02 20:31:57

💻 Система: linux
   CPU count: 2


# Настройки проекта

In [3]:
# =============================================================================
# 1.4 НАСТРОЙКИ ПРОЕКТА
# =============================================================================

# --------------------------------------------
# 1. ПРОЕКТ И ЦИКЛ (ХАРДКОД + ЗАЩИТА ОТ ПЕРЕЗАПИСИ)
# --------------------------------------------
PROJECT_NAME = 'stroke'           # Название проекта (папка в datasets)
CYCLE_NUMBER = 5                      # Номер цикла (хардкод, меняем вручную при новом цикле)

if IN_COLAB:
    BASE_PATH = '/content/drive/MyDrive/ml_learning'  # Colab: Google Drive
else:
    BASE_PATH = os.path.join(os.getcwd(), 'ml_learning')  # Локально: папка ml_learning

# --------------------------------------------
# 2. КЛАССЫ И ЦЕЛЕВАЯ ПЕРЕМЕННАЯ (для вывода и графиков)
# --------------------------------------------
POSITIVE_CLASS_NAME = 'Stroke'       # Название целевого класса (инсульт)
NEGATIVE_CLASS_NAME = 'No Stroke'    # Название отрицательного класса
TARGET_LABEL = 'инсульт'             # Для подписей в графиках (русский)

# --------------------------------------------
# 3. СТОИМОСТЬ ОШИБОК (MEDICAL COST)
# --------------------------------------------
COST_FN = 18      # используется, если GRID_COST_VALUES пуст
COST_FP = 1       # Штраф за ложную тревогу

# --------------------------------------------
# 4. СЕТКА ЭКСПЕРИМЕНТОВ (перебор гиперпараметров)
# --------------------------------------------
GRID_COST_VALUES = [16, 17, 18, 19, 20]    # Перебираем штраф за FN (от базового)
GRID_PENALTIES = ['l2']                    # L1 = Lasso (отбор признаков), L2 = Ridge
GRID_C_VALUES = [0.01, 0.1, 1]             # Сила регуляризации

# --------------------------------------------
# 5. БАЗОВАЯ КОНФИГУРАЦИЯ МОДЕЛИ
# --------------------------------------------
BASE_CONFIG = {
    'class_weight': 'balanced',      # Автоматически учитываем дисбаланс классов
    'max_iter': 5000,                # Максимум итераций для сходимости
    'tol': 1e-4,                     # Точность остановки (чем меньше, тем дольше)
    'random_state': None,            # Заполнится из EDA метаданных (для воспроизводимости)
}

# --------------------------------------------
# 6. КРОСС-ВАЛИДАЦИЯ И КАЛИБРОВКА
# --------------------------------------------
CV_FOLDS = 5                         # Количество фолдов для CV (выбор порога)
CALIBRATION_ENABLED = True           # Включить калибровку вероятностей
CALIBRATION_METHOD = 'sigmoid'       # sigmoid (платт) или isotonic
CALIBRATION_CV = 5                   # Фолды для калибровки (может отличаться от CV_FOLDS)
# Примечание: Минимальный размер фолда проверяется в ячейке 1.9
# (должен быть хотя бы 2 положительных в каждом фолде)

# --------------------------------------------
# 6.1 ROBUST OPTIMIZATION (стабильность порога)
# --------------------------------------------
ROBUST_ALPHA = 1.5      # Коэффициент риска (1.5 = 85% доверительный интервал)
ROBUST_ENABLED = True   # Включить поиск по верхней границе стоимости

# --------------------------------------------
# 7. ЗАПАС СТАБИЛЬНОСТИ (используется в фильтрах)
# --------------------------------------------
STABILITY_MARGIN = 0.05              # 5% запас для жёстких фильтров (например, Recall)

# --------------------------------------------
# 8. ФИЛЬТРЫ УРОВНЯ 1 (технические) — отсекаем заведомо плохие модели
# --------------------------------------------
FILTER_MIN_BSS = 0.1                 # Минимальный Brier Skill Score (чем выше, тем лучше)
FILTER_REQUIRE_CONVERGED = True      # Требуем, чтобы модель сошлась (закончила итерации)
FILTER_MAX_STABILITY_GAP = 0.10      # PR-AUC(train) - PR-AUC(valid) не больше 0.10

# --------------------------------------------
# 9. ФИЛЬТРЫ УРОВНЯ 2 (расширенные) — качество модели и стабильность
# --------------------------------------------
FILTER_MAX_CV_STABILITY_STD = 0.08           # Стабильность порога по фолдам (чем меньше, тем лучше)
FILTER_MAX_MCE = 0.15                        # Maximum Calibration Error (чем меньше, тем лучше)
FILTER_MAX_ECE = 0.05                        # Expected Calibration Error (чем меньше, тем лучше)
FILTER_RELIABILITY_SLOPE_MIN = 0.8           # Наклон калибровочной кривой (минимальный)
FILTER_RELIABILITY_SLOPE_MAX = 1.2           # Наклон калибровочной кривой (максимальный)
FILTER_MIN_PR_AUC = 0.2                      # Минимальный PR-AUC (отсекаем совсем плохие)
FILTER_MAX_MEDICAL_COST_CV_COEF = 0.30       # Коэффициент вариации стоимости (стабильность)

# --------------------------------------------
# 10. ФИЛЬТРЫ УРОВНЯ 3 (клинические) — бизнес-требования
# --------------------------------------------
FILTER_MIN_RECALL_VAL = 0.75 + STABILITY_MARGIN   # Минимальный Recall (0.90 с запасом)
FILTER_MIN_PRECISION_VAL = 0.05                   # Минимальная Precision (низкий порог)
FILTER_MAX_SELECTION_RATE_VAL = 0.40              # Максимум срабатываний (45% пациентов)
FILTER_MAX_NNI_VAL = 12                           # NNI (сколько обследовать на 1 найденный)
FILTER_MIN_TP_VAL = 10                             # Минимум TP в валидации (для стабильности)
FILTER_MIN_RECALL_LOWER_CI = 0.70                 # Нижняя граница доверительного интервала Recall

# --------------------------------------------
# 11. СТРАТИФИКАЦИЯ РИСКА (для таблицы и графика)
# --------------------------------------------
RISK_BINS = [0, 0.02, 0.05, 0.10, 0.20, 1.0]     # Границы групп риска
RISK_LABELS = ['Очень низкий (<2%)', 'Низкий (2-5%)', 'Средний (5-10%)',
               'Высокий (10-20%)', 'Критический (>20%)']

# --------------------------------------------
# 12. СОХРАНЕНИЕ АРТЕФАКТОВ
# --------------------------------------------
SAVE_BEST_MODEL = True       # Сохранять финальную модель
SAVE_PLOTS = True            # Сохранять все графики

# --------------------------------------------
# ВЫВОД НАСТРОЕК (для проверки)
# --------------------------------------------
print("=" * 70)
print("📋 НАСТРОЙКИ ПРОЕКТА")
print("=" * 70)
print(f"📂 BASE_PATH: {BASE_PATH}")
print(f"   PROJECT_NAME: {PROJECT_NAME}")
print(f"   CYCLE_NUMBER: {CYCLE_NUMBER}")
print(f"\n💰 Medical Cost: FN×{COST_FN} + FP×{COST_FP}")
print(f"\n🔬 Сетка: COST={GRID_COST_VALUES}, penalty={GRID_PENALTIES}, C={GRID_C_VALUES}")
print(f"\n🔧 Базовая конфигурация модели:")
for key, value in BASE_CONFIG.items():
    print(f"   {key}: {value if value is not None else 'будет из метаданных'}")
print(f"\n📏 Запас стабильности: {STABILITY_MARGIN*100:.0f}%")
print(f"\n🛡️ Фильтр Recall: ≥{FILTER_MIN_RECALL_VAL}")
print(f"🛡️ Фильтр Selection Rate: ≤{FILTER_MAX_SELECTION_RATE_VAL*100:.0f}%")
print(f"🛡️ Фильтр PR-AUC: ≥{FILTER_MIN_PR_AUC}")
print("=" * 70)

📋 НАСТРОЙКИ ПРОЕКТА
📂 BASE_PATH: /content/drive/MyDrive/ml_learning
   PROJECT_NAME: stroke
   CYCLE_NUMBER: 5

💰 Medical Cost: FN×18 + FP×1

🔬 Сетка: COST=[16, 17, 18, 19, 20], penalty=['l2'], C=[0.01, 0.1, 1]

🔧 Базовая конфигурация модели:
   class_weight: balanced
   max_iter: 5000
   tol: 0.0001
   random_state: будет из метаданных

📏 Запас стабильности: 5%

🛡️ Фильтр Recall: ≥0.8
🛡️ Фильтр Selection Rate: ≤40%
🛡️ Фильтр PR-AUC: ≥0.2


In [4]:
# =============================================================================
# 1.5 МОНТИРОВАНИЕ GOOGLE DRIVE
# =============================================================================

print("📂 МОНТИРОВАНИЕ GOOGLE DRIVE")
print("="*50)

if IN_COLAB:
    if BASE_PATH.startswith('/content/drive'):
        from google.colab import drive

        if os.path.ismount('/content/drive'):
            print("✓ Google Drive уже смонтирован")
        else:
            print("📂 Монтирование Google Drive...")

            # Если есть папка /content/drive, но она НЕ точка монтирования
            if os.path.exists('/content/drive') and not os.path.ismount('/content/drive'):
                backup_name = f'/content/drive_backup_{datetime.now().strftime("%Y%m%d_%H%M%S")}'
                print(f"   ⚠️ Обнаружена папка /content/drive (не точка монтирования)")
                print(f"   Перемещаем в {backup_name}")
                shutil.move('/content/drive', backup_name)

            # Монтируем Google Drive с обработкой ошибок
            try:
                drive.mount('/content/drive')
                print("✓ Google Drive смонтирован")
            except Exception as e:
                print(f"\n❌ ОШИБКА МОНТИРОВАНИЯ: {e}")
                print("   Возможные причины:")
                print("   • Нет подключения к интернету")
                print("   • Нет прав доступа к Google Drive")
                print("   • Проблемы с авторизацией")
                print("\n   💡 Решение:")
                print("   • Проверьте интернет-соединение")
                print("   • Перезапустите среду выполнения (Runtime → Restart runtime)")
                print("   • При монтировании нажмите на ссылку и скопируйте код авторизации")
                raise  # Останавливаем выполнение, так как без Drive дальше не продолжить
    else:
        print("✓ BASE_PATH не на Google Drive, монтирование не требуется")
else:
    print("✓ Локальная среда, монтирование не требуется")

print("="*50 + "\n")

📂 МОНТИРОВАНИЕ GOOGLE DRIVE
✓ Google Drive уже смонтирован



In [5]:
# =============================================================================
# 1.6 НАСТРОЙКА ПУТЕЙ ДЛЯ MODELLING (ХАРДКОД + ЗАЩИТА ОТ ПЕРЕЗАПИСИ)
# =============================================================================

print("\n" + "="*70)
print("📂 НАСТРОЙКА ПУТЕЙ ДЛЯ MODELLING")
print("="*70)

# --------------------------------------------
# ПРОВЕРКА НАСТРОЕК
# --------------------------------------------
if 'BASE_PATH' not in globals():
    raise NameError("❌ BASE_PATH не определён. Выполните блок 1.4")

if 'PROJECT_NAME' not in globals() or PROJECT_NAME is None:
    raise ValueError("❌ Укажите PROJECT_NAME в блоке 1.4")

if 'CYCLE_NUMBER' not in globals() or CYCLE_NUMBER is None:
    raise ValueError("❌ Укажите CYCLE_NUMBER в блоке 1.4")

if not isinstance(CYCLE_NUMBER, int) or CYCLE_NUMBER <= 0:
    raise ValueError(f"❌ CYCLE_NUMBER должен быть положительным целым числом, получено: {CYCLE_NUMBER}")

# --------------------------------------------
# ФОРМИРУЕМ ПУТИ
# --------------------------------------------
PROJECT_PATH = os.path.join(BASE_PATH, 'datasets', PROJECT_NAME)
CYCLE_PATH = os.path.join(PROJECT_PATH, f'cycle_{CYCLE_NUMBER}')
EDA_PATH = os.path.join(CYCLE_PATH, 'eda')
EDA_DATA_PATH = os.path.join(EDA_PATH, 'data')
EDA_TRANSFORMERS_PATH = os.path.join(EDA_PATH, 'transformers')
PRODUCTION_METADATA_PATH = os.path.join(EDA_PATH, 'production_metadata.json')

MODELLING_PATH = os.path.join(CYCLE_PATH, 'modelling')
MODELLING_REPORTS_PATH = os.path.join(MODELLING_PATH, 'reports')
MODELLING_PLOTS_PATH = os.path.join(MODELLING_PATH, 'plots')
MODELLING_SPLITS_PATH = os.path.join(MODELLING_PATH, 'splits')

# --------------------------------------------
# ЗАЩИТА ОТ ПЕРЕЗАПИСИ (удаляем старую папку при подтверждении)
# --------------------------------------------
if os.path.exists(MODELLING_PATH):
    print(f"\n⚠️ ВНИМАНИЕ: Папка modelling для цикла {CYCLE_NUMBER} уже существует!")
    print(f"   {MODELLING_PATH}")

    existing_files = os.listdir(MODELLING_PATH)
    if existing_files:
        print(f"   Обнаружены существующие файлы: {existing_files[:5]}...")
        if 'final_model.joblib' in existing_files:
            print("   ⚠️ ВНИМАНИЕ: Обнаружена финальная модель! Перезапись удалит её.")

    print(f"\n   Что делать:")
    print(f"   • Если хотите СОЗДАТЬ НОВЫЙ ЦИКЛ — измените CYCLE_NUMBER на {CYCLE_NUMBER + 1} в блоке 1.4")
    print(f"   • Если хотите ПЕРЕЗАПИСАТЬ текущий modelling — введите 'y'")

    response = input(f"\n   Продолжить с перезаписью? (y/N): ")
    if response.lower() != 'y':
        print("❌ Операция отменена. Измените CYCLE_NUMBER и запустите заново.")
        raise SystemExit(0)
    else:
        print(f"   ⚠️ Продолжаем с перезаписью modelling для cycle_{CYCLE_NUMBER}")
        shutil.rmtree(MODELLING_PATH)
        print(f"   🗑️ Старая папка modelling удалена")

# --------------------------------------------
# СОЗДАНИЕ СТРУКТУРЫ ПАПОК MODELLING
# --------------------------------------------
for path in [MODELLING_PATH, MODELLING_REPORTS_PATH, MODELLING_PLOTS_PATH, MODELLING_SPLITS_PATH]:
    os.makedirs(path, exist_ok=True)

print(f"\n📂 ПУТИ НАСТРОЕНЫ:")
print(f"   MODELLING_PATH: {MODELLING_PATH}")
print(f"   MODELLING_SPLITS_PATH: {MODELLING_SPLITS_PATH}")

print("\n✅ Пути настроены, защита от перезаписи активна")
print("="*70 + "\n")


📂 НАСТРОЙКА ПУТЕЙ ДЛЯ MODELLING

📂 ПУТИ НАСТРОЕНЫ:
   MODELLING_PATH: /content/drive/MyDrive/ml_learning/datasets/stroke/cycle_5/modelling
   MODELLING_SPLITS_PATH: /content/drive/MyDrive/ml_learning/datasets/stroke/cycle_5/modelling/splits

✅ Пути настроены, защита от перезаписи активна



In [6]:
# =============================================================================
# 1.7 ЗАГРУЗКА МЕТАДАННЫХ ИЗ EDA (С ПРОВЕРКАМИ)
# =============================================================================

print("\n" + "=" * 70)
print("📋 ЗАГРУЗКА МЕТАДАННЫХ ИЗ EDA")
print("=" * 70)

# --------------------------------------------
# ПРОВЕРКА СУЩЕСТВОВАНИЯ EDA-ФАЙЛОВ
# --------------------------------------------
if not os.path.exists(EDA_PATH):
    raise FileNotFoundError(
        f"\n❌ ОШИБКА: Папка EDA для цикла {CYCLE_NUMBER} не найдена!\n"
        f"   Искали: {EDA_PATH}\n"
        f"   Убедитесь, что:\n"
        f"   1. Preprocessing и EDA для цикла {CYCLE_NUMBER} выполнены\n"
        f"   2. CYCLE_NUMBER указан правильно\n"
        f"   3. Пути настроены корректно"
    )

if not os.path.exists(EDA_DATA_PATH):
    raise FileNotFoundError(
        f"❌ Папка EDA data не найдена: {EDA_DATA_PATH}\n"
        f"   Сначала выполните EDA ноутбук для цикла {CYCLE_NUMBER}"
    )

if not os.path.exists(EDA_TRANSFORMERS_PATH):
    raise FileNotFoundError(
        f"❌ Папка EDA transformers не найдена: {EDA_TRANSFORMERS_PATH}\n"
        f"   Сначала выполните EДА ноутбук для цикла {CYCLE_NUMBER}"
    )

if not os.path.exists(PRODUCTION_METADATA_PATH):
    raise FileNotFoundError(
        f"❌ production_metadata.json не найден: {PRODUCTION_METADATA_PATH}\n"
        f"   Сначала выполните EDA ноутбук для цикла {CYCLE_NUMBER}"
    )

# --------------------------------------------
# ЗАГРУЗКА МЕТАДАННЫХ
# --------------------------------------------
with open(PRODUCTION_METADATA_PATH, 'r', encoding='utf-8') as f:
    prod_metadata = json.load(f)

# --------------------------------------------
# ИЗВЛЕЧЕНИЕ ПАРАМЕТРОВ (НЕ ПЕРЕЗАПИСЫВАЕМ CYCLE_NUMBER!)
# --------------------------------------------
eda_cycle_number = prod_metadata.get('cycle_number', None)
if eda_cycle_number is not None and eda_cycle_number != CYCLE_NUMBER:
    print(f"\n⚠️ ПРЕДУПРЕЖДЕНИЕ: Несоответствие номеров циклов!")
    print(f"   Хардкодный CYCLE_NUMBER (блок 1.4): {CYCLE_NUMBER}")
    print(f"   EDA цикл в метаданных: {eda_cycle_number}")
    print(f"   Продолжаем с хардкодным значением {CYCLE_NUMBER}")

RANDOM_STATE = prod_metadata.get('random_state', 42)
TASK_TYPE = prod_metadata.get('task_type', 'classification')
FEATURE_COLUMNS = prod_metadata.get('features', {}).get('all_features', [])
TARGET_COLUMN = prod_metadata.get('target_column', 'target')

# --------------------------------------------
# ПРОВЕРКА НА ПУСТОЙ FEATURE_COLUMNS
# --------------------------------------------
if not FEATURE_COLUMNS:
    raise ValueError(
        f"❌ FEATURE_COLUMNS пуст или не загружен из metadata!\n"
        f"   Проверьте, что EDA сохранил список признаков в production_metadata.json\n"
        f"   Ожидается структура: {{'features': {{'all_features': [...]}} }}"
    )

FINAL_COUNT = len(FEATURE_COLUMNS)

# --------------------------------------------
# ОБНОВЛЯЕМ BASE_CONFIG (только random_state!)
# --------------------------------------------
BASE_CONFIG['random_state'] = RANDOM_STATE

# --------------------------------------------
# ПРОВЕРКИ КОРРЕКТНОСТИ
# --------------------------------------------
if TASK_TYPE != 'classification':
    raise ValueError(
        f"❌ Этот ноутбук предназначен для классификации.\n"
        f"   Обнаружен тип задачи: '{TASK_TYPE}'."
    )

if FINAL_COUNT == 0:
    raise ValueError("❌ Список признаков пуст. Проверьте ноутбук EDA.")

# --------------------------------------------
# ВЫВОД МЕТАДАННЫХ
# --------------------------------------------
print("\n📋 МЕТАДАННЫЕ ПРОЕКТА (из EDA):")
print(f"   Проект: {PROJECT_NAME}")
print(f"   Хардкодный CYCLE_NUMBER: {CYCLE_NUMBER}")
print(f"   EDA cycle в метаданных: {eda_cycle_number}")
print(f"   Тип задачи: {TASK_TYPE}")
print(f"   Random state: {RANDOM_STATE} (загружен из EDA)")
print(f"   Целевая колонка: {TARGET_COLUMN}")
print(f"   Признаков: {FINAL_COUNT}")

max_display = 15
print(f"   Список признаков (первые {min(FINAL_COUNT, max_display)}):")
for i, col in enumerate(FEATURE_COLUMNS[:max_display], 1):
    print(f"      {i:2d}. {col}")
if FINAL_COUNT > max_display:
    print(f"      ... и ещё {FINAL_COUNT - max_display} признаков")

# --------------------------------------------
# ДОПОЛНИТЕЛЬНАЯ ПРОВЕРКА: целевая колонка не должна быть в признаках
# --------------------------------------------
if TARGET_COLUMN in FEATURE_COLUMNS:
    raise ValueError(
        f"❌ Целевая колонка '{TARGET_COLUMN}' найдена в списке признаков!\n"
        f"   Это утечка данных. Проверьте EDA ноутбук."
)

print("\n✅ Метаданные загружены, проверки пройдены")
print("=" * 70 + "\n")


📋 ЗАГРУЗКА МЕТАДАННЫХ ИЗ EDA

📋 МЕТАДАННЫЕ ПРОЕКТА (из EDA):
   Проект: stroke
   Хардкодный CYCLE_NUMBER: 5
   EDA cycle в метаданных: 5
   Тип задачи: classification
   Random state: 99 (загружен из EDA)
   Целевая колонка: stroke
   Признаков: 10
   Список признаков (первые 10):
       1. age
       2. avg_glucose_level
       3. cardio_risk
       4. stable_old_age
       5. smoking_age_impact
       6. bmi_missing_flag
       7. work_type_govt_job
       8. work_type_never_worked
       9. work_type_private
      10. work_type_self-employed

✅ Метаданные загружены, проверки пройдены



In [7]:
# =============================================================================
# 1.8 ЗАГРУЗКА ДАННЫХ (ТОЛЬКО TRAIN И VAL)
# =============================================================================

print("\n" + "=" * 70)
print("📂 ЗАГРУЗКА ДАННЫХ (TRAIN + VAL)")
print("=" * 70)

# Фиксированные имена файлов (данные уже подготовлены в EDA)
train_file = 'X_train_scaled.csv'
val_file = 'X_val_scaled.csv'
y_train_file = 'y_train.csv'
y_val_file = 'y_val.csv'

print(f"   X_train: {train_file}")
print(f"   X_val:   {val_file}")
print(f"   y_train: {y_train_file}")
print(f"   y_val:   {y_val_file}")
print("   ⚠️ TEST НЕ ЗАГРУЖАЕТСЯ — будет использован в cycle_comparison")

# --------------------------------------------
# ПРОВЕРКА НАЛИЧИЯ ФАЙЛОВ
# --------------------------------------------
missing = []
for f in [train_file, val_file, y_train_file, y_val_file]:
    if not os.path.exists(os.path.join(EDA_DATA_PATH, f)):
        missing.append(f)

if missing:
    raise FileNotFoundError(
        f"❌ Не найдены файлы в {EDA_DATA_PATH}:\n"
        f"   {', '.join(missing)}\n"
        f"   Убедитесь, что ноутбук EDA выполнен для цикла {CYCLE_NUMBER}."
    )

# --------------------------------------------
# ЗАГРУЗКА ДАННЫХ
# --------------------------------------------
X_train = pd.read_csv(os.path.join(EDA_DATA_PATH, train_file))
X_val = pd.read_csv(os.path.join(EDA_DATA_PATH, val_file))

# Загрузка целевой переменной с проверкой имени колонки
y_train_df = pd.read_csv(os.path.join(EDA_DATA_PATH, y_train_file))
y_val_df = pd.read_csv(os.path.join(EDA_DATA_PATH, y_val_file))

# ПРОВЕРКА: колонка с целевой переменной должна существовать
if TARGET_COLUMN not in y_train_df.columns:
    raise ValueError(
        f"❌ Колонка '{TARGET_COLUMN}' не найдена в y_train.csv!\n"
        f"   Доступные колонки: {list(y_train_df.columns)}\n"
        f"   Ожидается ровно одна колонка с именем '{TARGET_COLUMN}'.\n"
        f"   Проверьте EDA ноутбук для цикла {CYCLE_NUMBER}."
    )

if TARGET_COLUMN not in y_val_df.columns:
    raise ValueError(
        f"❌ Колонка '{TARGET_COLUMN}' не найдена в y_val.csv!\n"
        f"   Доступные колонки: {list(y_val_df.columns)}\n"
        f"   Ожидается ровно одна колонка с именем '{TARGET_COLUMN}'.\n"
        f"   Проверьте EDA ноутбук для цикла {CYCLE_NUMBER}."
    )

y_train = y_train_df[TARGET_COLUMN]
y_val = y_val_df[TARGET_COLUMN]

# Приведение к int (защита от строковых значений)
y_train = y_train.astype(int)
y_val = y_val.astype(int)

# --------------------------------------------
# БАЗОВЫЕ ПРОВЕРКИ
# --------------------------------------------
if len(X_train) != len(y_train):
    raise ValueError(f"❌ Несоответствие размеров: X_train={len(X_train)}, y_train={len(y_train)}")

if len(X_val) != len(y_val):
    raise ValueError(f"❌ Несоответствие размеров: X_val={len(X_val)}, y_val={len(y_val)}")

print(f"\n📊 РАЗМЕРЫ ВЫБОРОК:")
print(f"   Train: {X_train.shape}, y={y_train.shape}")
print(f"   Val:   {X_val.shape}, y={y_val.shape}")

# --------------------------------------------
# Проверка наличия scaler
# --------------------------------------------
scaler_path = os.path.join(EDA_TRANSFORMERS_PATH, 'scalers.pkl')
if not os.path.exists(scaler_path):
    print("\n⚠️ ВНИМАНИЕ: scalers.pkl не найден!")
    print(f"   Искали: {scaler_path}")
    print("   Данные могут НЕ быть масштабированы или масштабированы другим способом.")
    print("   Убедитесь, что EDA ноутбук выполнил масштабирование и сохранил scalers.pkl.\n")
else:
    print(f"\n✅ Scalers найден: {scaler_path}")

# --------------------------------------------
# ПРОВЕРКА СООТВЕТСТВИЯ ПРИЗНАКОВ МЕТАДАННЫМ
# --------------------------------------------
if FEATURE_COLUMNS:
    available_features = [col for col in FEATURE_COLUMNS if col in X_train.columns]
    missing_in_data = set(FEATURE_COLUMNS) - set(available_features)

    if missing_in_data:
        print(f"\n⚠️ ВНИМАНИЕ: В данных отсутствуют признаки из метаданных:")
        for col in missing_in_data:
            print(f"      - {col}")

    extra_in_data = set(X_train.columns) - set(FEATURE_COLUMNS)
    if extra_in_data:
        print(f"\n⚠️ ВНИМАНИЕ: В данных есть признаки, отсутствующие в метаданных:")
        for col in extra_in_data:
            print(f"      - {col}")

    X_train = X_train[available_features]
    X_val = X_val[available_features]

    print(f"\n🔒 Оставлено {len(available_features)} признаков из {len(FEATURE_COLUMNS)}")
else:
    available_features = X_train.columns.tolist()
    print(f"\n⚠️ FEATURE_COLUMNS не найдена в метаданных, используем все {len(available_features)} признаков")

feature_names = X_train.columns.tolist()

print(f"\n📊 ФИНАЛЬНЫЕ РАЗМЕРЫ:")
print(f"   Train: {X_train.shape}")
print(f"   Val:   {X_val.shape}")
print(f"   Признаков: {len(feature_names)}")

# --------------------------------------------
# СТАТИСТИКА ПО КЛАССАМ
# --------------------------------------------
train_pos = y_train.sum()
val_pos = y_val.sum()
train_neg = len(y_train) - train_pos
val_neg = len(y_val) - val_pos

print(f"\n🎯 ЦЕЛЕВАЯ ПЕРЕМЕННАЯ ({TARGET_LABEL}):")
print(f"   Train: {train_pos} положительных ({train_pos/len(y_train)*100:.2f}%), {train_neg} отрицательных")
print(f"   Val:   {val_pos} положительных ({val_pos/len(y_val)*100:.2f}%), {val_neg} отрицательных")

if train_pos == 0 or train_neg == 0:
    raise ValueError(f"❌ В train есть пустой класс: положительных={train_pos}, отрицательных={train_neg}")
if val_pos == 0 or val_neg == 0:
    raise ValueError(f"❌ В val есть пустой класс: положительных={val_pos}, отрицательных={val_neg}")

# --------------------------------------------
# ПРОВЕРКА НАЛИЧИЯ TEST (ЗАЩИТА ОТ СЛУЧАЙНОЙ ЗАГРУЗКИ)
# --------------------------------------------
test_file = os.path.join(EDA_DATA_PATH, 'X_test_scaled.csv')
if os.path.exists(test_file):
    print(f"\n✅ TEST файл существует, НО НЕ ЗАГРУЖАЕТСЯ (как и требуется)")
    print(f"   Путь: {test_file}")
else:
    print(f"\n⚠️ TEST файл не найден в {EDA_DATA_PATH} — возможно, EDA не создал test сплит")

print("\n✅ Данные загружены")
print("=" * 70 + "\n")


📂 ЗАГРУЗКА ДАННЫХ (TRAIN + VAL)
   X_train: X_train_scaled.csv
   X_val:   X_val_scaled.csv
   y_train: y_train.csv
   y_val:   y_val.csv
   ⚠️ TEST НЕ ЗАГРУЖАЕТСЯ — будет использован в cycle_comparison

📊 РАЗМЕРЫ ВЫБОРОК:
   Train: (3031, 10), y=(3031,)
   Val:   (1011, 10), y=(1011,)

✅ Scalers найден: /content/drive/MyDrive/ml_learning/datasets/stroke/cycle_5/eda/transformers/scalers.pkl

🔒 Оставлено 10 признаков из 10

📊 ФИНАЛЬНЫЕ РАЗМЕРЫ:
   Train: (3031, 10)
   Val:   (1011, 10)
   Признаков: 10

🎯 ЦЕЛЕВАЯ ПЕРЕМЕННАЯ (инсульт):
   Train: 149 положительных (4.92%), 2882 отрицательных
   Val:   50 положительных (4.95%), 961 отрицательных

✅ TEST файл существует, НО НЕ ЗАГРУЖАЕТСЯ (как и требуется)
   Путь: /content/drive/MyDrive/ml_learning/datasets/stroke/cycle_5/eda/data/X_test_scaled.csv

✅ Данные загружены



In [8]:
# --------------------------------------------
# 1.9 ПРОВЕРКА НА МУЛЬТИКОЛЛИНЕАРНОСТЬ (VIF) — только предупреждение
# --------------------------------------------
print("\n📊 8. ПРОВЕРКА НА МУЛЬТИКОЛЛИНЕАРНОСТЬ (VIF):")

try:
    from statsmodels.stats.outliers_influence import variance_inflation_factor
    import statsmodels.api as sm

    # Добавляем константу для VIF
    X_with_const = sm.add_constant(X_train)

    vif_data = []
    for i in range(X_with_const.shape[1]):
        vif = variance_inflation_factor(X_with_const.values, i)
        vif_data.append({
            'feature': X_with_const.columns[i],
            'VIF': vif
        })

    df_vif = pd.DataFrame(vif_data)
    df_vif = df_vif[df_vif['feature'] != 'const'].sort_values('VIF', ascending=False)

    high_vif = df_vif[df_vif['VIF'] > 10]

    if len(high_vif) > 0:
        print(f"   ⚠️ ВНИМАНИЕ: Обнаружены признаки с VIF > 10 (сильная мультиколлинеарность):")
        for _, row in high_vif.head(5).iterrows():
            print(f"      • {row['feature']}: VIF = {row['VIF']:.1f}")
        if len(high_vif) > 5:
            print(f"      ... и ещё {len(high_vif) - 5} признаков")
        print(f"   💡 Рекомендация: Вернитесь в EDA и проверьте эти признаки.")
        print(f"   🔧 Обучение продолжается, но модель может быть нестабильной.")

        # Сохраняем VIF отчёт
        vif_path = os.path.join(MODELLING_REPORTS_PATH, 'vif_report.csv')
        df_vif.to_csv(vif_path, index=False)
        print(f"   💾 Полный отчёт VIF сохранён: {vif_path}")
    else:
        print(f"   ✅ Все признаки имеют VIF ≤ 10 (мультиколлинеарность в норме)")

except Exception as e:
    print(f"   ⚠️ Не удалось рассчитать VIF: {e}")
    print(f"      (возможно, слишком мало данных или признаки нечисловые)")


📊 8. ПРОВЕРКА НА МУЛЬТИКОЛЛИНЕАРНОСТЬ (VIF):
   ✅ Все признаки имеют VIF ≤ 10 (мультиколлинеарность в норме)


In [9]:
# =============================================================================
# 1.10 ФИНАЛЬНАЯ ВАЛИДАЦИЯ ДАННЫХ ПЕРЕД ОБУЧЕНИЕМ
# =============================================================================

print("\n" + "="*70)
print("🛡️ ФИНАЛЬНАЯ ВАЛИДАЦИЯ ДАННЫХ ПЕРЕД ОБУЧЕНИЕМ")
print("="*70)

validation_passed = True
warnings_list = []

# --------------------------------------------
# 1. IDENTICAL COLUMNS
# --------------------------------------------
print("\n📋 1. ПРОВЕРКА КОЛОНОК:")
print(f"   Train: {list(X_train.columns[:5])}... ({len(X_train.columns)} признаков)")
print(f"   Val:   {list(X_val.columns[:5])}... ({len(X_val.columns)} признаков)")

if list(X_train.columns) != list(X_val.columns):
    validation_passed = False
    print(f"   ❌ КРИТИЧЕСКАЯ ОШИБКА: Колонки Train и Val различаются!")
    print(f"      Только в Train: {set(X_train.columns) - set(X_val.columns)}")
    print(f"      Только в Val: {set(X_val.columns) - set(X_train.columns)}")
else:
    print(f"   ✅ Колонки идентичны")

# --------------------------------------------
# 2. NaN И INF
# --------------------------------------------
print("\n🔍 2. ПРОВЕРКА НА NaN И Inf:")

for name, X_df in [('Train', X_train), ('Val', X_val)]:
    nan_count = X_df.isnull().sum().sum()
    if nan_count > 0:
        validation_passed = False
        print(f"   ❌ {name}: обнаружены NaN ({nan_count} пропусков)")
    else:
        print(f"   ✅ {name}: нет NaN")

    inf_cols = X_df.columns[np.isinf(X_df).any()].tolist()
    if inf_cols:
        validation_passed = False
        print(f"   ❌ {name}: обнаружены Inf в колонках: {inf_cols}")
    else:
        print(f"   ✅ {name}: нет Inf")

# --------------------------------------------
# 3. CONSTANT FEATURES
# --------------------------------------------
print("\n⚠️ 3. ПРОВЕРКА КОНСТАНТНЫХ ПРИЗНАКОВ:")

constant_cols_train = []
constant_cols_val = []

for col in X_train.columns:
    if X_train[col].nunique() == 1:
        constant_cols_train.append(col)

for col in X_val.columns:
    if X_val[col].nunique() == 1:
        constant_cols_val.append(col)

if constant_cols_train:
    warnings_list.append(f"Train: константные признаки {constant_cols_train}")
    print(f"   ⚠️ Train: константные признаки: {constant_cols_train[:5]}{'...' if len(constant_cols_train) > 5 else ''}")
else:
    print(f"   ✅ Train: все признаки вариативны")

if constant_cols_val:
    warnings_list.append(f"Val: константные признаки {constant_cols_val}")
    print(f"   ⚠️ Val: константные признаки: {constant_cols_val[:5]}{'...' if len(constant_cols_val) > 5 else ''}")
else:
    print(f"   ✅ Val: все признаки вариативны")

# Проверка: одинаковые ли значения у константных признаков в train и val
for col in constant_cols_train:
    if col in constant_cols_val:
        train_val = X_train[col].iloc[0]
        val_val = X_val[col].iloc[0]
        if train_val != val_val:
            warnings_list.append(f"Признак '{col}' константен в train ({train_val}) и val ({val_val}) — разные значения!")
            print(f"   ⚠️ Признак '{col}': train={train_val}, val={val_val} (разные константы)")

# --------------------------------------------
# 4. ЭКСТРЕМАЛЬНЫЕ ЗНАЧЕНИЯ (|z| > 5)
# --------------------------------------------
print("\n🔍 4. ПРОВЕРКА НА ЭКСТРЕМАЛЬНЫЕ ЗНАЧЕНИЯ (|z| > 5):")

zscore_threshold = 5
extreme_train = []
extreme_val = []

for col in X_train.columns:
    if (np.abs(X_train[col]) > zscore_threshold).any():
        extreme_train.append(col)

for col in X_val.columns:
    if (np.abs(X_val[col]) > zscore_threshold).any():
        extreme_val.append(col)

if extreme_train:
    warnings_list.append(f"Экстремальные значения в train: {extreme_train}")
    print(f"   ⚠️ Train: экстремальные значения в {extreme_train}")
else:
    print(f"   ✅ Train: нет экстремальных значений")

if extreme_val:
    warnings_list.append(f"Экстремальные значения в val: {extreme_val}")
    print(f"   ⚠️ Val: экстремальные значения в {extreme_val}")
else:
    print(f"   ✅ Val: нет экстремальных значений")

# --------------------------------------------
# 5. DATA TYPE CONSISTENCY
# --------------------------------------------
print("\n📊 5. ПРОВЕРКА ТИПОВ ДАННЫХ:")

dtype_mismatches = []
for col in X_train.columns:
    if col in X_val.columns:
        if X_train[col].dtype != X_val[col].dtype:
            dtype_mismatches.append(f"{col}: Train={X_train[col].dtype}, Val={X_val[col].dtype}")

if dtype_mismatches:
    warnings_list.append(f"Несоответствие типов данных: {len(dtype_mismatches)} признаков")
    print(f"   ⚠️ Несоответствие типов данных:")
    for mismatch in dtype_mismatches[:5]:
        print(f"      - {mismatch}")
    if len(dtype_mismatches) > 5:
        print(f"      ... и ещё {len(dtype_mismatches) - 5}")
else:
    print(f"   ✅ Типы данных согласованы")

# --------------------------------------------
# 6. ЦЕЛЕВАЯ ПЕРЕМЕННАЯ (0/1 и распределение)
# --------------------------------------------
print("\n🎯 6. ЦЕЛЕВАЯ ПЕРЕМЕННАЯ:")

# Проверка, что y содержит только 0 и 1
unique_train = set(y_train.unique())
unique_val = set(y_val.unique())

if not unique_train.issubset({0, 1}):
    raise ValueError(f"❌ y_train содержит недопустимые значения: {unique_train - {0, 1}}")

if not unique_val.issubset({0, 1}):
    raise ValueError(f"❌ y_val содержит недопустимые значения: {unique_val - {0, 1}}")

for name, y_series in [('Train', y_train), ('Val', y_val)]:
    pos = y_series.sum()
    neg = len(y_series) - pos
    pos_pct = pos / len(y_series) * 100
    print(f"   {name}: 0={neg}, 1={pos} ({pos_pct:.2f}%)")

val_pos = y_val.sum()
if val_pos < FILTER_MIN_TP_VAL:
    warnings_list.append(f"В Val всего {val_pos} положительных (минимум {FILTER_MIN_TP_VAL} для стабильности метрик)")
    print(f"\n⚠️ ПРЕДУПРЕЖДЕНИЕ: В Val очень мало положительных случаев ({val_pos} < {FILTER_MIN_TP_VAL})")
    print(f"   → Метрики могут быть нестабильными. Рекомендуется увеличить долю val или использовать стратификацию.")

# --------------------------------------------
# 7. ПРОВЕРКА РАЗМЕРОВ ДЛЯ КАЛИБРОВКИ И CV
# --------------------------------------------
print("\n🔧 7. ПРОВЕРКА РАЗМЕРОВ ДЛЯ КАЛИБРОВКИ:")

n_samples = len(X_train)
n_positives = y_train.sum()
n_negatives = n_samples - n_positives
positive_rate = y_train.mean()

print(f"   📊 Train: {n_samples} объектов, положительных: {n_positives} ({positive_rate*100:.2f}%)")

# РАССЧИТЫВАЕМ МИНИМАЛЬНЫЙ РАЗМЕР ФОЛДА
# ------------------------------------------------------------
# Исходим из того, что в каждом фолде должен быть хотя бы 1 положительный
# Желательно 2-3 для стабильности метрик

if n_positives < 10:
    print(f"   ⚠️ КРИТИЧЕСКИ МАЛО положительных ({n_positives})!")
    print(f"      Рекомендуется собрать больше данных или использовать стратификацию.")

# Минимальный размер фолда = как минимум 2 положительных в самом маленьком фолде
min_positives_per_fold = 2
min_fold_size_from_positives = int(np.ceil(min_positives_per_fold / positive_rate))

# Нижняя граница: хотя бы 10 объектов для статистики
min_fold_size = max(10, min_fold_size_from_positives)

print(f"   📏 Рассчитанный минимальный размер фолда: {min_fold_size} объектов")
print(f"      (исходя из {min_positives_per_fold}+ положительных в каждом фолде)")

# ПРОВЕРКА ДЛЯ CV_FOLDS
# ------------------------------------------------------------
n_folds_cv = CV_FOLDS
expected_fold_size = n_samples // n_folds_cv
expected_positives_per_fold = int(expected_fold_size * positive_rate)

print(f"\n   🔄 CV_FOLDS = {n_folds_cv}:")
print(f"      Ожидаемый размер фолда: ~{expected_fold_size} объектов")
print(f"      Ожидаемое число положительных в фолде: ~{expected_positives_per_fold}")

if expected_fold_size < min_fold_size:
    recommended_cv = max(2, int(n_samples / min_fold_size))
    print(f"      ⚠️ СЛИШКОМ МАЛО! Рекомендуется уменьшить CV_FOLDS до {recommended_cv}")
elif expected_positives_per_fold < min_positives_per_fold:
    print(f"      ⚠️ МАЛО ПОЛОЖИТЕЛЬНЫХ! Рассмотрите увеличение данных или уменьшение CV_FOLDS")
else:
    print(f"      ✅ Достаточно объектов для стабильной CV")

# ПРОВЕРКА ДЛЯ CALIBRATION_CV
# ------------------------------------------------------------
if CALIBRATION_ENABLED:
    n_folds_calib = CALIBRATION_CV
    expected_fold_size_calib = n_samples // n_folds_calib
    expected_positives_per_fold_calib = int(expected_fold_size_calib * positive_rate)

    print(f"\n   🔧 CALIBRATION_CV = {n_folds_calib}:")
    print(f"      Ожидаемый размер фолда: ~{expected_fold_size_calib} объектов")
    print(f"      Ожидаемое число положительных: ~{expected_positives_per_fold_calib}")

    if expected_fold_size_calib < min_fold_size:
        recommended_calib = max(2, int(n_samples / min_fold_size))
        print(f"      ⚠️ СЛИШКОМ МАЛО! Рекомендуется уменьшить CALIBRATION_CV до {recommended_calib}")
    elif expected_positives_per_fold_calib < min_positives_per_fold:
        print(f"      ⚠️ МАЛО ПОЛОЖИТЕЛЬНЫХ! Калибровка может быть нестабильной")
    else:
        print(f"      ✅ Достаточно объектов для стабильной калибровки")
else:
    print(f"\n   ℹ️ Калибровка отключена (CALIBRATION_ENABLED=False)")

# --------------------------------------------
# ИТОГ
# --------------------------------------------
print("\n" + "="*70)

if validation_passed:
    print("✅ ФИНАЛЬНАЯ ВАЛИДАЦИЯ ПРОЙДЕНА УСПЕШНО")
else:
    print("❌ ОБНАРУЖЕНЫ КРИТИЧЕСКИЕ ПРОБЛЕМЫ!")
    raise ValueError("Исправьте ошибки перед обучением модели")

if warnings_list:
    print(f"\n⚠️ ПРЕДУПРЕЖДЕНИЯ ({len(warnings_list)}):")
    for w in warnings_list:
        print(f"   - {w}")

print("="*70 + "\n")


🛡️ ФИНАЛЬНАЯ ВАЛИДАЦИЯ ДАННЫХ ПЕРЕД ОБУЧЕНИЕМ

📋 1. ПРОВЕРКА КОЛОНОК:
   Train: ['age', 'avg_glucose_level', 'cardio_risk', 'stable_old_age', 'smoking_age_impact']... (10 признаков)
   Val:   ['age', 'avg_glucose_level', 'cardio_risk', 'stable_old_age', 'smoking_age_impact']... (10 признаков)
   ✅ Колонки идентичны

🔍 2. ПРОВЕРКА НА NaN И Inf:
   ✅ Train: нет NaN
   ✅ Train: нет Inf
   ✅ Val: нет NaN
   ✅ Val: нет Inf

⚠️ 3. ПРОВЕРКА КОНСТАНТНЫХ ПРИЗНАКОВ:
   ✅ Train: все признаки вариативны
   ✅ Val: все признаки вариативны

🔍 4. ПРОВЕРКА НА ЭКСТРЕМАЛЬНЫЕ ЗНАЧЕНИЯ (|z| > 5):
   ✅ Train: нет экстремальных значений
   ✅ Val: нет экстремальных значений

📊 5. ПРОВЕРКА ТИПОВ ДАННЫХ:
   ✅ Типы данных согласованы

🎯 6. ЦЕЛЕВАЯ ПЕРЕМЕННАЯ:
   Train: 0=2882, 1=149 (4.92%)
   Val: 0=961, 1=50 (4.95%)

🔧 7. ПРОВЕРКА РАЗМЕРОВ ДЛЯ КАЛИБРОВКИ:
   📊 Train: 3031 объектов, положительных: 149 (4.92%)
   📏 Рассчитанный минимальный размер фолда: 41 объектов
      (исходя из 2+ положительных в каждом фол

# БЛОК 2. ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ

In [10]:
# =============================================================================
# 2.1 ФУНКЦИИ ДЛЯ РАСЧЁТА МЕТРИК И ПОРОГОВ
# =============================================================================

def get_metrics_at_threshold(y_true, y_probs, threshold, cost_fn=None, cost_fp=None):
    """
    Возвращает все метрики для конкретного порога, включая Medical Cost.

    Параметры:
    ----------
    y_true : array-like
        Истинные метки
    y_probs : array-like
        Предсказанные вероятности
    threshold : float
        Порог классификации
    cost_fn : float, optional
        Штраф за пропуск (False Negative)
    cost_fp : float, optional
        Штраф за ложную тревогу (False Positive)

    Возвращает:
    -----------
    dict : Словарь со всеми метриками
    """
    if cost_fn is None:
        cost_fn = COST_FN
    if cost_fp is None:
        cost_fp = COST_FP

    y_pred = (y_probs >= threshold).astype(int)

    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred)

    # Проверка, что оба класса присутствуют
    if cm.shape != (2, 2):
        raise ValueError(
            f"❌ Confusion matrix имеет форму {cm.shape}, ожидается (2, 2).\n"
            f"   Проверьте, что y_true содержит оба класса (0 и 1).\n"
            f"   Уникальные значения y_true: {np.unique(y_true)}"
        )

    tn, fp, fn, tp = cm.ravel()

    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    npv = tn / (tn + fn) if (tn + fn) > 0 else 0

    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    f2 = (5 * precision * recall) / (4 * precision + recall) if (4 * precision + recall) > 0 else 0

    selection_rate = (tp + fp) / len(y_true) if len(y_true) > 0 else 0
    nni = 1 / precision if precision > 0 else float('inf')
    medical_cost = fn * cost_fn + fp * cost_fp

    return {
        'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn,
        'recall': recall,
        'precision': precision,
        'specificity': specificity,
        'npv': npv,
        'f1': f1,
        'f2': f2,
        'selection_rate': selection_rate,
        'nni': nni,
        'medical_cost': medical_cost
    }


def calculate_medical_cost(fn, fp, cost_fn=None, cost_fp=None):
    """
    Рассчитывает Medical Cost = FN * cost_fn + FP * cost_fp

    Параметры:
    ----------
    fn : int
        Количество False Negative (пропущенных положительных)
    fp : int
        Количество False Positive (ложных тревог)
    cost_fn : float, optional
        Штраф за пропуск (по умолчанию COST_FN)
    cost_fp : float, optional
        Штраф за ложную тревогу (по умолчанию COST_FP)
    """
    if cost_fn is None:
        cost_fn = COST_FN
    if cost_fp is None:
        cost_fp = COST_FP
    return fn * cost_fn + fp * cost_fp


def calculate_nni(precision):
    """
    Number Needed to Inspect = 1 / Precision.
    NNI показывает, сколько пациентов нужно обследовать,
    чтобы найти одного с инсультом.
    """
    if precision > 0:
        return 1 / precision
    return float('inf')


def recall_confidence_interval(tp, fn, confidence=0.95):
    """
    Доверительный интервал для Recall (пропорции) через бета-распределение.
    Использует Jeffreys interval, подходит для малых выборок.

    Параметры:
    ----------
    tp : int
        True Positive (пойманные положительные)
    fn : int
        False Negative (пропущенные положительные)
    confidence : float, default=0.95
        Уровень доверия (должен быть между 0 и 1)

    Возвращает:
    -----------
    (lower, upper) : tuple
        Нижняя и верхняя границы доверительного интервала
    """
    if not 0 < confidence < 1:
        raise ValueError(f"confidence должен быть между 0 и 1, получено {confidence}")

    n = tp + fn
    if n == 0:
        return 0.0, 0.0

    alpha = tp + 1
    beta_param = fn + 1

    lower = beta.ppf((1 - confidence) / 2, alpha, beta_param)
    upper = beta.ppf(1 - (1 - confidence) / 2, alpha, beta_param)

    return lower, upper


def find_optimal_threshold_by_cost(y_true, y_probs, cost_fn=None, cost_fp=None):
    """
    Находит порог, минимизирующий Medical Cost = FN * cost_fn + FP * cost_fp.

    ВНИМАНИЕ: Эта функция НЕ использует кросс-валидацию.
    Для фиксации порога используйте CV-подход в run_single_experiment_cv.

    Возвращает:
    -----------
    best_threshold : float
        Оптимальный порог
    metrics : dict
        Метрики при оптимальном пороге
    """
    if cost_fn is None:
        cost_fn = COST_FN
    if cost_fp is None:
        cost_fp = COST_FP

    # Используем стандартную precision_recall_curve
    precision, recall, thresholds = precision_recall_curve(y_true, y_probs)

    if len(thresholds) == 0:
        return 0.5, get_metrics_at_threshold(y_true, y_probs, 0.5, cost_fn, cost_fp)

    best_cost = np.inf
    best_idx = None

    for i, th in enumerate(thresholds):
        y_pred = (y_probs >= th).astype(int)
        cm = confusion_matrix(y_true, y_pred)

        if cm.shape != (2, 2):
            continue

        tn, fp, fn, tp = cm.ravel()
        cost = calculate_medical_cost(fn, fp, cost_fn, cost_fp)

        if cost < best_cost:
            best_cost = cost
            best_idx = i

    if best_idx is None:
        return 0.5, get_metrics_at_threshold(y_true, y_probs, 0.5, cost_fn, cost_fp)

    best_threshold = thresholds[best_idx]
    metrics = get_metrics_at_threshold(y_true, y_probs, best_threshold, cost_fn, cost_fp)

    return best_threshold, metrics


def find_robust_threshold_by_cost(y_true, y_probs, cost_fn=None, cost_fp=None, alpha=1.5):
    """
    Находит порог, минимизирующий ВЕРХНЮЮ ГРАНИЦУ стоимости (Mean + alpha * Std).

    Это защита от выбора "везучего" порога, который может рухнуть на тесте.

    Параметры:
    ----------
    y_true : array-like
        Истинные метки
    y_probs : array-like
        Предсказанные вероятности
    cost_fn : float, optional
        Штраф за FN
    cost_fp : float, optional
        Штраф за FP
    alpha : float, default=1.5
        Коэффициент риска (1.5 = ~85% доверительный интервал)

    Возвращает:
    -----------
    best_threshold : float
        Оптимальный порог (с учётом стабильности)
    metrics : dict
        Метрики при оптимальном пороге
    """
    if cost_fn is None:
        cost_fn = COST_FN
    if cost_fp is None:
        cost_fp = COST_FP

    precision, recall, thresholds = precision_recall_curve(y_true, y_probs)

    if len(thresholds) == 0:
        return 0.5, get_metrics_at_threshold(y_true, y_probs, 0.5, cost_fn, cost_fp)

    # Рассчитываем стоимость для каждого порога
    costs = []
    for th in thresholds:
        y_pred = (y_probs >= th).astype(int)
        cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
        if cm.shape != (2, 2):
            continue
        tn, fp, fn, tp = cm.ravel()
        cost = fn * cost_fn + fp * cost_fp
        costs.append(cost)

    if len(costs) == 0:
        return 0.5, get_metrics_at_threshold(y_true, y_probs, 0.5, cost_fn, cost_fp)

    costs = np.array(costs)

    # Скользящее окно для расчёта mean и std (чтобы сгладить шум)
    window = max(3, len(costs) // 20)
    window = min(window, len(costs))

    mean_costs = np.convolve(costs, np.ones(window)/window, mode='same')

    std_costs = np.array([
        np.std(costs[max(0, i-window):min(len(costs), i+window+1)])
        for i in range(len(costs))
    ])

    # Робастная стоимость = верхняя граница доверительного интервала
    robust_costs = mean_costs + alpha * std_costs

    best_idx = np.argmin(robust_costs)
    best_threshold = thresholds[best_idx]

    metrics = get_metrics_at_threshold(y_true, y_probs, best_threshold, cost_fn, cost_fp)

    return best_threshold, metrics


print("✅ Robust threshold functions загружены")

def precision_confidence_interval(tp, fp, confidence=0.95):
    """
    Доверительный интервал для Precision через бета-распределение.

    Параметры:
    ----------
    tp : int
        True Positive
    fp : int
        False Positive
    confidence : float, default=0.95
        Уровень доверия

    Возвращает:
    -----------
    (lower, upper) : tuple
        Нижняя и верхняя границы доверительного интервала
    """
    from scipy.stats import beta

    if not 0 < confidence < 1:
        raise ValueError(f"confidence должен быть между 0 и 1, получено {confidence}")

    n = tp + fp
    if n == 0:
        return 0.0, 0.0

    alpha_param = tp + 1
    beta_param = fp + 1

    lower = beta.ppf((1 - confidence) / 2, alpha_param, beta_param)
    upper = beta.ppf(1 - (1 - confidence) / 2, alpha_param, beta_param)

    return lower, upper


print("✅ Функции метрик и confidence intervals загружены")


print("✅ Функции метрик загружены")

✅ Robust threshold functions загружены
✅ Функции метрик и confidence intervals загружены
✅ Функции метрик загружены


In [11]:
# =============================================================================
# 2.2 ФУНКЦИИ ДЛЯ КАЛИБРОВКИ
# =============================================================================

def brier_skill_score(y_true, y_probs):
    """
    Brier Skill Score (BSS)

    BSS = 1 - (BS_model / BS_climatology)

    Интерпретация:
    - BSS > 0 → модель лучше "глупого" прогноза (climatology)
    - BSS = 0 → модель на уровне "глупого" прогноза
    - BSS < 0 → модель хуже "глупого" прогноза

    Параметры:
    ----------
    y_true : array-like
        Истинные метки (0/1)
    y_probs : array-like
        Предсказанные вероятности

    Возвращает:
    -----------
    float : Brier Skill Score
    """
    bs_model = brier_score_loss(y_true, y_probs)
    p_mean = np.mean(y_true)
    bs_climatology = brier_score_loss(y_true, np.full_like(y_true, p_mean))

    if bs_climatology == 0:
        return 0.0

    return 1 - (bs_model / bs_climatology)


def get_reliability_slope(y_true, y_probs, n_bins=10, return_intercept=False):
    """
    Наклон калибровочной кривой (идеально = 1).

    Интерпретация:
    - slope < 1 → модель переоценивает риск (предсказания слишком высокие)
    - slope > 1 → модель недооценивает риск (предсказания слишком низкие)

    Параметры:
    ----------
    y_true : array-like
        Истинные метки
    y_probs : array-like
        Предсказанные вероятности
    n_bins : int, default=10
        Количество бинов для калибровочной кривой
    return_intercept : bool, default=False
        Возвращать ли интерсепт вместе со slope

    Возвращает:
    -----------
    slope : float
        Наклон калибровочной кривой
    intercept : float (если return_intercept=True)
        Интерсепт (свободный член)
    """
    prob_true, prob_pred = calibration_curve(y_true, y_probs, n_bins=n_bins, strategy='quantile')

    if len(prob_pred) < 2:
        if return_intercept:
            return 1.0, 0.0
        return 1.0

    slope, intercept = np.polyfit(prob_pred, prob_true, 1)

    if return_intercept:
        return slope, intercept
    return slope


def calculate_ece(y_true, y_probs, n_bins=10):
    """
    Expected Calibration Error (ECE)

    ECE = Σ (n_bin / n_total) * |true_prob_bin - pred_prob_bin|

    Чем меньше, тем лучше. Идеал = 0.

    Параметры:
    ----------
    y_true : array-like
        Истинные метки
    y_probs : array-like
        Предсказанные вероятности
    n_bins : int, default=10
        Количество бинов для калибровки

    Возвращает:
    -----------
    ece : float
        Expected Calibration Error (взвешенный по количеству объектов)
    """
    total = len(y_probs)

    # Создаём бины на основе процентилей (равное количество объектов в каждом бине)
    bins = np.percentile(y_probs, np.linspace(0, 100, n_bins + 1))
    bins[0] = -np.inf
    bins[-1] = np.inf

    # Разбиваем объекты по бинам
    bin_indices = np.digitize(y_probs, bins) - 1  # -1 т.к. digitize возвращает 1..n+1
    bin_indices = np.clip(bin_indices, 0, n_bins - 1)

    ece = 0.0

    for i in range(n_bins):
        mask = bin_indices == i
        n_bin = mask.sum()

        if n_bin == 0:
            continue

        pred_mean = y_probs[mask].mean()
        true_mean = y_true[mask].mean()
        weight = n_bin / total

        ece += weight * abs(true_mean - pred_mean)

    return ece


def calculate_mce(y_true, y_probs, n_bins=10):
    """
    Maximum Calibration Error (MCE)

    MCE = max |true_prob_bin - pred_prob_bin|

    Чем меньше, тем лучше. Идеал = 0.

    Параметры:
    ----------
    y_true : array-like
        Истинные метки
    y_probs : array-like
        Предсказанные вероятности
    n_bins : int, default=10
        Количество бинов для калибровки

    Возвращает:
    -----------
    mce : float
        Maximum Calibration Error (максимальная ошибка по бинам)
    """
    n_bins = min(n_bins, len(np.unique(y_probs)))

    # Создаём бины на основе процентилей
    bins = np.percentile(y_probs, np.linspace(0, 100, n_bins + 1))
    bins[0] = -np.inf
    bins[-1] = np.inf

    bin_indices = np.digitize(y_probs, bins) - 1
    bin_indices = np.clip(bin_indices, 0, n_bins - 1)

    max_error = 0.0

    for i in range(n_bins):
        mask = bin_indices == i
        if mask.sum() == 0:
            continue

        pred_mean = y_probs[mask].mean()
        true_mean = y_true[mask].mean()
        error = abs(true_mean - pred_mean)

        if error > max_error:
            max_error = error

    return max_error


def calculate_ece_mce(y_true, y_probs, n_bins=10):
    """
    Удобная функция, возвращающая оба значения (ECE и MCE).

    Возвращает:
    -----------
    (ece, mce) : tuple
        Expected Calibration Error и Maximum Calibration Error
    """
    ece = calculate_ece(y_true, y_probs, n_bins)
    mce = calculate_mce(y_true, y_probs, n_bins)
    return ece, mce


def get_calibration_data(y_true, y_probs, n_bins=10):
    """
    Возвращает данные для построения калибровочной кривой с весами.

    Параметры:
    ----------
    y_true : array-like
        Истинные метки
    y_probs : array-like
        Предсказанные вероятности
    n_bins : int, default=10
        Количество бинов

    Возвращает:
    -----------
    dict : Содержит prob_true, prob_pred, bin_counts, bin_frac_pos, bin_avg_prob
    """
    prob_true, prob_pred = calibration_curve(y_true, y_probs, n_bins=n_bins, strategy='quantile')

    # Рассчитываем количество объектов в каждом бине
    bins = np.percentile(y_probs, np.linspace(0, 100, n_bins + 1))
    bins[0] = -np.inf
    bins[-1] = np.inf

    bin_indices = np.digitize(y_probs, bins) - 1
    bin_indices = np.clip(bin_indices, 0, n_bins - 1)
    bin_counts = np.bincount(bin_indices, minlength=n_bins)

    # Доля положительных в каждом бине (альтернативный расчёт)
    bin_frac_pos = []
    bin_avg_prob = []

    for i in range(n_bins):
        mask = bin_indices == i
        if mask.sum() > 0:
            bin_frac_pos.append(y_true[mask].mean())
            bin_avg_prob.append(y_probs[mask].mean())
        else:
            bin_frac_pos.append(0.0)
            bin_avg_prob.append(0.0)

    return {
        'prob_true': prob_true,
        'prob_pred': prob_pred,
        'bin_counts': bin_counts,
        'bin_frac_pos': np.array(bin_frac_pos),
        'bin_avg_prob': np.array(bin_avg_prob),
        'n_bins': n_bins
    }


print("✅ Функции калибровки загружены")

✅ Функции калибровки загружены


In [12]:
# =============================================================================
# 2.3 ФАБРИКА МОДЕЛЕЙ
# =============================================================================

def build_model(exp_config, base_config=None):
    """
    Фабрика моделей. Поддерживает LogisticRegression и Dummy.

    Параметры:
    ----------
    exp_config : dict
        Конфигурация эксперимента. Должна содержать:

        Для LogisticRegression:
        - 'penalty': str ('l1', 'l2', 'elasticnet', 'none')
        - 'C': float (сила регуляризации, обратный параметр)
        - 'solver': str (опционально, определяется автоматически если не указан)

        Для Dummy:
        - 'model_type': 'dummy'
        - 'strategy': str ('stratified', 'most_frequent', 'prior', 'uniform')

    base_config : dict, optional
        Базовая конфигурация (class_weight, random_state, max_iter, tol)

    Возвращает:
    -----------
    model : sklearn estimator
        Необученная модель

    Примеры:
    --------
    >>> exp = {'penalty': 'l1', 'C': 0.1}
    >>> model = build_model(exp)

    >>> exp = {'model_type': 'dummy', 'strategy': 'stratified'}
    >>> model = build_model(exp)
    """
    if base_config is None:
        base_config = BASE_CONFIG.copy()

    # Dummy модель
    if exp_config.get('model_type') == 'dummy':
        strategy = exp_config.get('strategy', 'stratified')
        return DummyClassifier(
            strategy=strategy,
            random_state=base_config.get('random_state', 42)
        )

    # LogisticRegression
    model_params = base_config.copy()
    model_params['penalty'] = exp_config.get('penalty', 'l2')
    model_params['C'] = exp_config.get('C', 1.0)

    # Определяем solver, если не указан явно
    solver = exp_config.get('solver', None)
    if solver is None:
        # Автоматический выбор solver на основе penalty
        penalty = model_params['penalty']
        if penalty == 'l1':
            solver = 'liblinear'
        elif penalty == 'elasticnet':
            solver = 'saga'
        else:  # l2, none
            solver = 'lbfgs'
    model_params['solver'] = solver

    # Корректировка параметров для совместимости
    if model_params['solver'] == 'liblinear':
        # Для liblinear penalty l2 тоже работает, но нужно указать dual
        model_params['dual'] = False
        # liblinear не поддерживает max_iter в том же формате, но accepts
    elif model_params['solver'] == 'saga':
        # saga поддерживает все penalty
        pass
    elif model_params['solver'] == 'lbfgs':
        # lbfgs не поддерживает penalty='l1'
        if model_params['penalty'] == 'l1':
            print(f"   ⚠️ ВНИМАНИЕ: lbfgs не поддерживает penalty='l1', меняем на 'l2'")
            model_params['penalty'] = 'l2'
        # lbfgs не поддерживает penalty='elasticnet'
        if model_params['penalty'] == 'elasticnet':
            print(f"   ⚠️ ВНИМАНИЕ: lbfgs не поддерживает penalty='elasticnet', меняем на 'l2'")
            model_params['penalty'] = 'l2'

    return LogisticRegression(**model_params)


def is_calibration_needed(model):
    """
    Определяет, нужна ли калибровка для данной модели.

    DummyClassifier не нужно калибровать.
    LogisticRegression с sigmoid калибровкой калибровать не нужно (уже калибрована),
    но может потребоваться для других методов (isotonic).

    Параметры:
    ----------
    model : sklearn estimator
        Модель для проверки

    Returns:
    --------
    bool : True если калибровка нужна, False если нет
    """
    # Dummy модель не требует калибровки
    if isinstance(model, DummyClassifier):
        return False

    # LogisticRegression с sigmoid калибровкой уже близка к калиброванной
    # Но для isotonic калибровка может быть полезна
    # Возвращаем True, так как калибровка может улучшить, но не навредит
    return True


def get_model_info(model):
    """
    Возвращает информацию о модели для логирования.

    Параметры:
    ----------
    model : sklearn estimator
        Обученная или необученная модель

    Returns:
    --------
    dict : Информация о модели
    """
    info = {
        'model_type': type(model).__name__
    }

    if isinstance(model, LogisticRegression):
        info.update({
            'penalty': model.penalty,
            'solver': model.solver,
            'C': model.C,
            'class_weight': model.class_weight,
            'max_iter': model.max_iter,
        })
    elif isinstance(model, DummyClassifier):
        info.update({
            'strategy': model.strategy,
        })

    return info


print("✅ build_model() и вспомогательные функции загружены")

✅ build_model() и вспомогательные функции загружены


In [13]:
# =============================================================================
# 2.4 ФУНКЦИИ ВИЗУАЛИЗАЦИИ
# =============================================================================

def plot_confusion_matrix_at_threshold(y_true, y_probs, threshold, title="", save_path=None, ax=None):
    """
    Построить confusion matrix для конкретного порога.

    Параметры:
    ----------
    y_true : array-like
        Истинные метки
    y_probs : array-like
        Предсказанные вероятности
    threshold : float
        Порог классификации
    title : str, default=""
        Заголовок графика
    save_path : str, optional
        Путь для сохранения графика
    ax : matplotlib.axes.Axes, optional
        Axes для рисования (если None, создаётся новая фигура)

    Возвращает:
    -----------
    fig, ax : tuple
        Figure и Axes объекты
    """
    y_pred = (y_probs >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred)

    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 5))
    else:
        fig = ax.get_figure()

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=[NEGATIVE_CLASS_NAME, POSITIVE_CLASS_NAME],
                yticklabels=[NEGATIVE_CLASS_NAME, POSITIVE_CLASS_NAME])
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_title(f'{title} (threshold={threshold:.3f})' if title else f'Threshold = {threshold:.3f}')

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')

    return fig, ax


def plot_risk_stratification(strat_table, model_name="", save_path=None):
    """
    Визуализация таблицы стратификации риска.

    Параметры:
    ----------
    strat_table : pd.DataFrame
        Таблица стратификации из get_risk_stratification_table
    model_name : str, default=""
        Название модели для заголовка
    save_path : str, optional
        Путь для сохранения графика

    Возвращает:
    -----------
    fig : matplotlib.figure.Figure
        Figure объект
    """
    strat_table = strat_table.sort_values('Actual_Risk_Rate').reset_index(drop=True)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # График 1: Actual Risk Rate по группам
    groups = strat_table['Risk Group'].tolist()
    risks = strat_table['Actual_Risk_Rate'].tolist()

    axes[0].barh(groups, risks, color='crimson', alpha=0.7)
    axes[0].set_xlabel('Actual Risk Rate (доля положительных)')
    axes[0].set_title(f'{model_name} - Risk by Group')
    axes[0].axvline(x=np.mean(risks), color='gray', linestyle='--', label='Average Risk')
    axes[0].legend()

    # График 2: Доля целевого класса vs доля пациентов
    patients_pct = strat_table['%_of_All_Patients'].tolist()
    target_pct = strat_table['%_of_All_Target'].tolist()

    x = np.arange(len(groups))
    width = 0.35

    axes[1].bar(x - width/2, patients_pct, width, label='% of Patients', alpha=0.7)
    axes[1].bar(x + width/2, target_pct, width, label=f'% of {TARGET_LABEL.title()}', alpha=0.7)
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(groups, rotation=45, ha='right')
    axes[1].set_ylabel('Percentage of total (%)')
    axes[1].set_title(f'{model_name} - Target Concentration')
    axes[1].legend()

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')

    return fig


def plot_recall_vs_threshold(y_true, y_probs, model_name="", target_recall=None, save_path=None):
    """
    Кривая Recall vs Threshold.

    Параметры:
    ----------
    y_true : array-like
        Истинные метки
    y_probs : array-like
        Предсказанные вероятности
    model_name : str, default=""
        Название модели для заголовка
    target_recall : float, optional
        Целевой Recall для отображения горизонтальной линии
    save_path : str, optional
        Путь для сохранения графика

    Возвращает:
    -----------
    fig : matplotlib.figure.Figure
        Figure объект
    """
    if target_recall is None:
        target_recall = FILTER_MIN_RECALL_VAL

    precision, recall, thresholds = precision_recall_curve(y_true, y_probs)
    recall = recall[:-1]

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(thresholds, recall, 'b-', linewidth=2)
    ax.set_xlabel('Threshold')
    ax.set_ylabel('Recall')
    ax.set_title(f'{model_name} - Recall vs Threshold')
    ax.grid(alpha=0.3)
    ax.axhline(y=target_recall, color='r', linestyle='--',
               label=f'Min Recall = {target_recall:.0%}')
    ax.legend()

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')

    return fig


def plot_probability_histogram(y_true, y_probs, model_name="", save_path=None):
    """
    Гистограмма распределения вероятностей для классов.

    Параметры:
    ----------
    y_true : array-like
        Истинные метки
    y_probs : array-like
        Предсказанные вероятности
    model_name : str, default=""
        Название модели для заголовка
    save_path : str, optional
        Путь для сохранения графика

    Возвращает:
    -----------
    fig : matplotlib.figure.Figure
        Figure объект
    """
    fig, ax = plt.subplots(figsize=(10, 5))

    ax.hist(y_probs[y_true == 0], bins=50, alpha=0.5, label=NEGATIVE_CLASS_NAME,
            density=False, color='steelblue', weights=np.ones_like(y_probs[y_true == 0]) / len(y_probs) * 100)
    ax.hist(y_probs[y_true == 1], bins=50, alpha=0.5, label=POSITIVE_CLASS_NAME,
            density=False, color='crimson', weights=np.ones_like(y_probs[y_true == 1]) / len(y_probs) * 100)

    ax.set_xlabel('Predicted Probability')
    ax.set_ylabel('Percentage of All Patients (%)')
    ax.set_title(f'{model_name} - Probability Distribution')
    ax.legend()

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')

    return fig


def plot_calibration_curve(y_true, y_probs, model_name="", n_bins=10, save_path=None):
    """
    Калибровочная кривая (Reliability Diagram) с ECE/MCE.

    Параметры:
    ----------
    y_true : array-like
        Истинные метки
    y_probs : array-like
        Предсказанные вероятности
    model_name : str, default=""
        Название модели для заголовка
    n_bins : int, default=10
        Количество бинов
    save_path : str, optional
        Путь для сохранения графика

    Возвращает:
    -----------
    fig : matplotlib.figure.Figure
        Figure объект
    """
    prob_true, prob_pred = calibration_curve(y_true, y_probs, n_bins=n_bins, strategy='quantile')

    ece, mce = calculate_ece_mce(y_true, y_probs, n_bins)

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.plot(prob_pred, prob_true, marker='o', linewidth=2, markersize=8,
            color='crimson', label=f'{model_name}')
    ax.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Perfect calibration')
    ax.set_xlabel('Mean Predicted Probability')
    ax.set_ylabel('Fraction of Positives')
    ax.set_title(f'Calibration Curve - {model_name}\nECE = {ece:.4f}, MCE = {mce:.4f}')
    ax.legend()
    ax.grid(alpha=0.3)

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')

    return fig


def plot_medical_cost_curve(y_true, y_probs, cost_fn=None, cost_fp=None,
                            model_name="", save_path=None):
    """
    Кривая Medical Cost vs Threshold + FN/FP vs Threshold.

    ВНИМАНИЕ: Порог, возвращаемый этой функцией, НЕ ИСПОЛЬЗУЕТСЯ в production.
    Для фиксации порога используется CV-подход в run_single_experiment_cv.

    Параметры:
    ----------
    y_true : array-like
        Истинные метки
    y_probs : array-like
        Предсказанные вероятности
    cost_fn : float, optional
        Штраф за FN
    cost_fp : float, optional
        Штраф за FP
    model_name : str, default=""
        Название модели для заголовка
    save_path : str, optional
        Путь для сохранения графика

    Возвращает:
    -----------
    fig : matplotlib.figure.Figure
        Figure объект
    best_th : float
        Оптимальный порог (только для визуализации!)
    best_cost : float
        Минимальная стоимость
    best_fn : int
        FN при оптимальном пороге
    best_fp : int
        FP при оптимальном пороге
    """
    if cost_fn is None:
        cost_fn = COST_FN
    if cost_fp is None:
        cost_fp = COST_FP

    precision, recall, thresholds = precision_recall_curve(y_true, y_probs)

    costs = []
    fn_counts = []
    fp_counts = []

    for th in thresholds:
        y_pred = (y_probs >= th).astype(int)
        cm = confusion_matrix(y_true, y_pred)
        if cm.shape == (2, 2):
            tn, fp, fn, tp = cm.ravel()
        else:
            continue
        costs.append(calculate_medical_cost(fn, fp, cost_fn, cost_fp))
        fn_counts.append(fn)
        fp_counts.append(fp)

    if len(thresholds) == 0 or len(costs) == 0:
        print("⚠️ Не удалось построить Medical Cost Curve: нет порогов")
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        axes[0].text(0.5, 0.5, 'No thresholds available', ha='center', va='center')
        axes[1].text(0.5, 0.5, 'No thresholds available', ha='center', va='center')
        if save_path:
            plt.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.close(fig)
        return fig, 0.5, 0, 0, 0

    best_idx = np.argmin(costs)
    best_th = thresholds[best_idx]
    best_cost = costs[best_idx]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # График 1: Medical Cost
    axes[0].plot(thresholds, costs, 'b-', linewidth=2, label='Medical Cost')
    axes[0].axvline(x=best_th, color='crimson', linestyle='--', linewidth=2,
                    label=f'Optimal = {best_th:.3f}\nCost = {best_cost:.0f}')
    axes[0].set_xlabel('Threshold')
    axes[0].set_ylabel(f'Medical Cost (FN×{cost_fn} + FP×{cost_fp})')
    axes[0].set_title(f'Medical Cost vs Threshold - {model_name}')
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    # График 2: FN и FP
    axes[1].plot(thresholds, fn_counts, 'r-', linewidth=2, label='FN')
    axes[1].plot(thresholds, fp_counts, 'orange', linewidth=2, label='FP')
    axes[1].axvline(x=best_th, color='crimson', linestyle='--', linewidth=2)
    axes[1].set_xlabel('Threshold')
    axes[1].set_ylabel('Count')
    axes[1].set_title(f'FN and FP vs Threshold - {model_name}')
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')

    return fig, best_th, best_cost, fn_counts[best_idx], fp_counts[best_idx]


print("✅ Функции визуализации загружены")

✅ Функции визуализации загружены


# БЛОК 3. ФУНКЦИИ ДЛЯ ЭКСПЕРИМЕНТОВ

In [14]:
# =============================================================================
# 3.1 ФУНКЦИИ ДЛЯ КРОСС-ВАЛИДАЦИИ И СТАБИЛЬНОСТИ
# =============================================================================

def get_pr_auc_cv(model, X, y, cv=CV_FOLDS, random_state=None):
    """
    PR-AUC с кросс-валидацией.

    Параметры:
    ----------
    model : sklearn estimator
        Необученная модель
    X : pd.DataFrame
        Признаки
    y : pd.Series
        Целевая переменная
    cv : int, default=CV_FOLDS
        Количество фолдов
    random_state : int, optional
        random_state для воспроизводимости (если None, используется RANDOM_STATE)

    Возвращает:
    -----------
    (mean, std) : tuple
        Среднее и стандартное отклонение PR-AUC по фолдам
    """
    if random_state is None:
        random_state = RANDOM_STATE

    cv_split = StratifiedKFold(n_splits=cv, shuffle=True, random_state=random_state)
    scores = cross_val_score(model, X, y, cv=cv_split, scoring='average_precision')
    return scores.mean(), scores.std()


def get_stability_gap(model, X_train, y_train, X_val, y_val):
    """
    Детектор переобучения: разница PR-AUC между train и val.

    ВНИМАНИЕ: Модель должна быть уже обучена!
    Функция НЕ вызывает model.fit() сама.

    Положительный gap = модель переобучена (train лучше val).

    Параметры:
    ----------
    model : sklearn estimator
        УЖЕ ОБУЧЕННАЯ модель
    X_train : pd.DataFrame
        Train признаки
    y_train : pd.Series
        Train целевая переменная
    X_val : pd.DataFrame
        Val признаки
    y_val : pd.Series

    Возвращает:
    -----------
    float : Разница PR-AUC (train - val)
    """
    # Проверяем, что модель обучена
    if not hasattr(model, 'classes_'):
        raise ValueError("Модель не обучена! Вызовите model.fit() перед вызовом get_stability_gap")

    if hasattr(model, 'predict_proba'):
        train_probs = model.predict_proba(X_train)[:, 1]
        val_probs = model.predict_proba(X_val)[:, 1]
    else:
        raise ValueError("Модель не имеет метода predict_proba")

    train_pr_auc = average_precision_score(y_train, train_probs)
    val_pr_auc = average_precision_score(y_val, val_probs)

    return train_pr_auc - val_pr_auc


def get_threshold_cv_stability(model, X, y, cost_fn, cost_fp, cv=CV_FOLDS, random_state=None):
    """
    Вычисляет стабильность оптимального порога по фолдам CV.
    Используется для фильтра FILTER_MAX_CV_STABILITY_STD.

    ВНИМАНИЕ: Внутри CV используется калибровка (если включена),
    чтобы пороги были сопоставимы с production.

    Параметры:
    ----------
    model : sklearn estimator
        Необученная модель (будет клонироваться для каждого фолда)
    X : pd.DataFrame
        Признаки
    y : pd.Series
        Целевая переменная
    cost_fn : float
        Штраф за FN
    cost_fp : float
        Штраф за FP
    cv : int, default=CV_FOLDS
        Количество фолдов
    random_state : int, optional
        random_state для воспроизводимости

    Возвращает:
    -----------
    dict : Словарь с threshold_mean, threshold_std, thresholds, costs
    """
    if random_state is None:
        random_state = RANDOM_STATE

    cv_split = StratifiedKFold(n_splits=cv, shuffle=True, random_state=random_state)

    thresholds = []
    costs = []

    for train_idx, val_idx in cv_split.split(X, y):
        X_fold_train = X.iloc[train_idx]
        X_fold_val = X.iloc[val_idx]
        y_fold_train = y.iloc[train_idx]
        y_fold_val = y.iloc[val_idx]

        from sklearn.base import clone
        fold_model = clone(model)
        fold_model.fit(X_fold_train, y_fold_train)

        # Калибровка (если включена)
        if CALIBRATION_ENABLED and is_calibration_needed(fold_model):
            fold_calib = CalibratedClassifierCV(fold_model, cv=CALIBRATION_CV, method=CALIBRATION_METHOD)
            fold_calib.fit(X_fold_train, y_fold_train)
            fold_probs = fold_calib.predict_proba(X_fold_val)[:, 1]
        else:
            fold_probs = fold_model.predict_proba(X_fold_val)[:, 1]

        threshold, metrics = find_optimal_threshold_by_cost(
            y_fold_val, fold_probs, cost_fn, cost_fp
        )

        thresholds.append(threshold)
        costs.append(metrics['medical_cost'])

    threshold_mean = np.mean(thresholds)
    threshold_std = np.std(thresholds)
    cost_mean = np.mean(costs)
    cost_std = np.std(costs)

    return {
        'threshold_mean': threshold_mean,
        'threshold_std': threshold_std,
        'cost_mean': cost_mean,
        'cost_std': cost_std,
        'thresholds': thresholds,
        'costs': costs
    }


def get_top_odds_ratio(model, feature_names):
    """
    Возвращает Odds Ratio самого сильного признака (наибольшее отклонение от 1).

    Параметры:
    ----------
    model : sklearn estimator
        Обученная модель с coef_ (LogisticRegression)
    feature_names : list
        Список названий признаков

    Возвращает:
    -----------
    str : Описание признака и его Odds Ratio
    """
    if not hasattr(model, 'coef_'):
        return 'N/A'

    coefs = model.coef_[0]
    odds_ratios = np.exp(coefs)

    or_deviation = np.abs(odds_ratios - 1)
    top_idx = np.argmax(or_deviation)

    top_feature = feature_names[top_idx]
    top_or = odds_ratios[top_idx]

    if top_or > 1:
        return f"{top_feature}: {top_or:.2f} (↑ риск)"
    else:
        return f"{top_feature}: {top_or:.2f} (↓ риск)"


def get_bottom_odds_ratio(model, feature_names):
    """
    Возвращает Odds Ratio самого слабого признака (ближайший к 1).

    Параметры:
    ----------
    model : sklearn estimator
        Обученная модель с coef_ (LogisticRegression)
    feature_names : list
        Список названий признаков

    Возвращает:
    -----------
    str : Описание признака и его Odds Ratio
    """
    if not hasattr(model, 'coef_'):
        return 'N/A'

    coefs = model.coef_[0]
    odds_ratios = np.exp(coefs)

    or_deviation = np.abs(odds_ratios - 1)
    bottom_idx = np.argmin(or_deviation)

    bottom_feature = feature_names[bottom_idx]
    bottom_or = odds_ratios[bottom_idx]

    if abs(bottom_or - 1) < 0.05:
        return f"{bottom_feature}: {bottom_or:.2f} (не влияет)"
    elif bottom_or > 1:
        return f"{bottom_feature}: {bottom_or:.2f} (слабый ↑)"
    else:
        return f"{bottom_feature}: {bottom_or:.2f} (слабый ↓)"


def check_convergence(model):
    """
    Проверяет, сошлась ли модель.

    Для LogisticRegression: n_iter_ < max_iter
    Для Dummy: всегда True

    Параметры:
    ----------
    model : sklearn estimator
        Обученная модель

    Возвращает:
    -----------
    bool : True если модель сошлась, False если нет
    """
    if hasattr(model, 'n_iter_'):
        return model.n_iter_[0] < model.max_iter
    return True


print("✅ Функции кросс-валидации загружены")

✅ Функции кросс-валидации загружены


In [15]:
# =============================================================================
# 3.2 АНАЛИЗ ОШИБОК И СТРАТИФИКАЦИЯ РИСКА
# =============================================================================

def analyze_error_profiles(X_val, y_true, y_probs, threshold, feature_names,
                           model_name="", target_label="целевого класса", save_path=None):
    """
    Сравнение FN (пропущенных) и TP (пойманных) пациентов на VAL.
    Используется только для анализа победителя.

    Параметры:
    ----------
    X_val : pd.DataFrame
        Валидационные признаки
    y_true : array-like
        Истинные метки
    y_probs : array-like
        Предсказанные вероятности
    threshold : float
        Порог классификации
    feature_names : list
        Список названий признаков
    model_name : str, default=""
        Название модели для вывода
    target_label : str, default="целевого класса"
        Название целевого класса для вывода
    save_path : str, optional
        Путь для сохранения результатов в CSV

    Возвращает:
    -----------
    df_errors : pd.DataFrame or None
        DataFrame с профилями ошибок или None (если нет FN или TP)
    """

    # ВНИМАНИЕ: Сравнение абсолютных разниц чувствительно к масштабу признаков.
    # Все признаки должны быть масштабированы (это проверено в EDA).

    # Проверка длины
    if len(y_true) != len(y_probs):
        raise ValueError(f"Длина y_true ({len(y_true)}) и y_probs ({len(y_probs)}) не совпадают")

    y_pred = (y_probs >= threshold).astype(int)

    fn_idx = (y_true == 1) & (y_pred == 0)
    tp_idx = (y_true == 1) & (y_pred == 1)
    fp_idx = (y_true == 0) & (y_pred == 1)

    print(f"\n{'='*70}")
    print(f"📋 ERROR PROFILES: Кого модель пропускает? ({model_name})")
    print(f"{'='*70}")

    if fn_idx.sum() == 0:
        print("✅ Нет пропущенных (FN) пациентов!")
        return None

    if tp_idx.sum() == 0:
        print("⚠️ Нет пойманных (TP) пациентов — модель не работает")
        return None

    total_positives = fn_idx.sum() + tp_idx.sum()
    total_negatives = fp_idx.sum() + (y_true == 0).sum() - fp_idx.sum()

    fn_pct = (fn_idx.sum() / total_positives) * 100
    tp_pct = (tp_idx.sum() / total_positives) * 100
    fp_pct = (fp_idx.sum() / total_negatives) * 100 if total_negatives > 0 else 0

    print(f"📊 СТАТИСТИКА ОШИБОК:")
    print(f"   FN (пропущенные):     {fn_idx.sum():3d} пациентов ({fn_pct:.1f}% от всех положительных)")
    print(f"   TP (пойманные):       {tp_idx.sum():3d} пациентов ({tp_pct:.1f}% от всех положительных)")
    print(f"   FP (ложные тревоги):  {fp_idx.sum():3d} пациентов ({fp_pct:.1f}% от всех отрицательных)")
    print(f"   Всего положительных:  {total_positives}")
    print(f"   Всего отрицательных:  {total_negatives}")
    print()

    # Расчёт средних профилей
    fn_profile = X_val[fn_idx].mean()
    tp_profile = X_val[tp_idx].mean()
    fp_profile = X_val[fp_idx].mean() if fp_idx.sum() > 0 else None

    df_errors = pd.DataFrame({
        'feature': feature_names,
        'FN_mean': fn_profile.values,
        'TP_mean': tp_profile.values,
        'Difference_FN_TP': fn_profile.values - tp_profile.values,
        'Abs_Difference_FN_TP': np.abs(fn_profile.values - tp_profile.values)
    })

    if fp_profile is not None:
        df_errors['FP_mean'] = fp_profile.values
        df_errors['Difference_FP_TP'] = fp_profile.values - tp_profile.values
        df_errors['Abs_Difference_FP_TP'] = np.abs(fp_profile.values - tp_profile.values)

    df_errors = df_errors.sort_values('Abs_Difference_FN_TP', ascending=False)

    print(f"\n⚠️ ТОП-5 отличий FN от TP:")
    for _, row in df_errors.head(5).iterrows():
        direction = "↑ выше" if row['Difference_FN_TP'] > 0 else "↓ ниже"
        print(f"   • {row['feature']}: FN {direction} на {abs(row['Difference_FN_TP']):.3f}")

    # Анализ FP (если есть)
    if fp_idx.sum() > 0:
        print(f"\n⚠️ ТОП-5 отличий FP от TP (ложные тревоги):")
        df_fp_analysis = df_errors.sort_values('Abs_Difference_FP_TP', ascending=False)
        for _, row in df_fp_analysis.head(5).iterrows():
            if pd.notna(row.get('Difference_FP_TP', None)):
                direction = "↑ выше" if row['Difference_FP_TP'] > 0 else "↓ ниже"
                print(f"   • {row['feature']}: FP {direction} на {abs(row['Difference_FP_TP']):.3f}")

    # Сохранение результатов
    if save_path:
        df_errors.to_csv(save_path, index=False)
        print(f"\n💾 Error profiles сохранены: {save_path}")

    return df_errors


def get_risk_stratification_table(y_true, y_probs, model_name="",
                                   bins=None, labels=None, show_lift=True):
    """
    Таблица стратификации риска на VAL.

    Параметры:
    ----------
    y_true : array-like
        Истинные метки
    y_probs : array-like
        Предсказанные вероятности
    model_name : str, default=""
        Название модели (для вывода)
    bins : list, optional
        Границы бинов (по умолчанию RISK_BINS)
    labels : list, optional
        Названия бинов (по умолчанию RISK_LABELS)
    show_lift : bool, default=True
        Добавлять ли колонку Lift

    Возвращает:
    -----------
    strat_table : pd.DataFrame
        Таблица стратификации риска
    """
    # Проверка длины
    if len(y_true) != len(y_probs):
        raise ValueError(f"Длина y_true ({len(y_true)}) и y_probs ({len(y_probs)}) не совпадают")

    if bins is None:
        bins = RISK_BINS
    if labels is None:
        labels = RISK_LABELS

    if len(bins) != len(labels) + 1:
        raise ValueError(f"Количество границ bins ({len(bins)}) должно быть на 1 больше, чем labels ({len(labels)})")

    df = pd.DataFrame({'true': y_true, 'prob': y_probs})
    df['Risk Group'] = pd.cut(df['prob'], bins=bins, labels=labels, include_lowest=True)

    strat_table = df.groupby('Risk Group', observed=True).agg(
        Total_Patients=('true', 'count'),
        Actual_Target=('true', 'sum'),
        Predicted_Prob_Mean=('prob', 'mean')
    ).reset_index()

    # Добавляем все бины, даже пустые
    all_groups = pd.DataFrame({'Risk Group': labels})
    strat_table = all_groups.merge(strat_table, on='Risk Group', how='left').fillna(0)

    # Сортируем в порядке возрастания риска
    strat_table['Risk Group'] = pd.Categorical(strat_table['Risk Group'], categories=labels, ordered=True)
    strat_table = strat_table.sort_values('Risk Group').reset_index(drop=True)

    strat_table['Actual_Risk_Rate'] = (strat_table['Actual_Target'] / strat_table['Total_Patients']).round(4)
    strat_table['%_of_All_Patients'] = (strat_table['Total_Patients'] / len(df) * 100).round(1)

    total_target = strat_table['Actual_Target'].sum()
    if total_target > 0:
        strat_table['%_of_All_Target'] = (strat_table['Actual_Target'] / total_target * 100).round(1)
        strat_table['Cumulative_Target_Capture'] = strat_table['%_of_All_Target'].cumsum().round(1)
    else:
        strat_table['%_of_All_Target'] = 0
        strat_table['Cumulative_Target_Capture'] = 0

    if show_lift:
        avg_risk = total_target / len(df) if len(df) > 0 else 0
        if avg_risk > 0:
            strat_table['Lift'] = (strat_table['Actual_Risk_Rate'] / avg_risk).round(1)
        else:
            strat_table['Lift'] = 0

    return strat_table


def print_risk_stratification_table(strat_table, model_name=""):
    """
    Красивый вывод таблицы стратификации риска в консоль.

    Параметры:
    ----------
    strat_table : pd.DataFrame
        Таблица стратификации из get_risk_stratification_table
    model_name : str, default=""
        Название модели для заголовка
    """
    print(f"\n{'='*80}")
    print(f"📊 СТРАТИФИКАЦИЯ РИСКА - {model_name}")
    print(f"{'='*80}")

    display_cols = ['Risk Group', 'Total_Patients', 'Actual_Target',
                    'Actual_Risk_Rate', '%_of_All_Patients', '%_of_All_Target']

    if 'Lift' in strat_table.columns:
        display_cols.append('Lift')

    df_display = strat_table[display_cols].copy()
    df_display['Actual_Risk_Rate'] = df_display['Actual_Risk_Rate'].apply(lambda x: f"{x:.2%}")

    print(df_display.to_string(index=False))
    print(f"{'='*80}\n")


print("✅ Функции анализа ошибок и стратификации загружены")

✅ Функции анализа ошибок и стратификации загружены


In [16]:
# =============================================================================
# 3.3 ЗАПУСК ОДНОГО ЭКСПЕРИМЕНТА С CV-ОПТИМИЗАЦИЕЙ (БЕЗ УТЕЧЕК)
# =============================================================================

def run_single_experiment_cv(model, model_name, X_train, y_train, X_val, y_val,
                              exp_config=None, feature_names=None, cv_folds=5,
                              random_state=42, cost_fn=None, cost_fp=None):
    """
    Запускает эксперимент с ЧЕСТНОЙ кросс-валидацией:

    1. CV на train → поиск порога по MIN Medical Cost (НА КАЖДОМ ФОЛДЕ)
    2. Усреднение порога по фолдам (mean)
    3. Оценка на Val с усреднённым порогом
    4. Калибровка на всём train
    5. Сбор всех метрик

    ВАЖНО: НЕТ УТЕЧЕК ДАННЫХ!
    - Порог ищется на каждом фолде отдельно
    - Калибровка выполняется после определения порога
    - Внутри CV используется CALIBRATION_CV (не хардкод!)
    """
    # =========================================================================
    # ПРОВЕРКИ
    # =========================================================================
    assert cv_folds >= 2, f"❌ cv_folds должно быть >= 2, получено {cv_folds}"

    if cost_fn is None:
        cost_fn = COST_FN
    if cost_fp is None:
        cost_fp = COST_FP

    results = {'Experiment': model_name, '_grid_cost': cost_fn}

    # =========================================================================
    # 1. КРОСС-ВАЛИДАЦИЯ ДЛЯ ПОИСКА ОПТИМАЛЬНОГО ПОРОГА (ЧЕСТНО, БЕЗ УТЕЧЕК)
    # =========================================================================
    cv = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=random_state)

    cv_thresholds = []
    cv_medical_costs = []
    cv_pr_auc_scores = []

    print(f"   CV поиск порога...", end='', flush=True)

    for train_idx, val_idx in cv.split(X_train, y_train):
        X_fold_train = X_train.iloc[train_idx]
        X_fold_val = X_train.iloc[val_idx]
        y_fold_train = y_train.iloc[train_idx]
        y_fold_val = y_train.iloc[val_idx]

        fold_model = build_model(exp_config) if exp_config else clone(model)
        fold_model.fit(X_fold_train, y_fold_train)

        # КАЛИБРУЕМ на фолде (используем CALIBRATION_CV, не хардкод!)
        if CALIBRATION_ENABLED and is_calibration_needed(fold_model):
            fold_calib = CalibratedClassifierCV(fold_model, cv=CALIBRATION_CV, method=CALIBRATION_METHOD)
            fold_calib.fit(X_fold_train, y_fold_train)
            fold_probs_calib = fold_calib.predict_proba(X_fold_val)[:, 1]
        else:
            fold_probs_calib = fold_model.predict_proba(X_fold_val)[:, 1]

        if ROBUST_ENABLED:
            threshold, threshold_metrics = find_robust_threshold_by_cost(
                y_fold_val, fold_probs_calib, cost_fn, cost_fp, alpha=ROBUST_ALPHA
            )
        else:
            threshold, threshold_metrics = find_optimal_threshold_by_cost(
                y_fold_val, fold_probs_calib, cost_fn, cost_fp
            )

        cv_thresholds.append(threshold)
        cv_medical_costs.append(threshold_metrics['medical_cost'])
        cv_pr_auc_scores.append(average_precision_score(y_fold_val, fold_probs_calib))

    print(f" готово")

    # Усредняем порог по фолдам с отсевом выбросов (robust mean)
    cv_thresholds_sorted = sorted(cv_thresholds)
    if len(cv_thresholds_sorted) > 3:
        # Отбрасываем минимальный и максимальный порог (защита от выбросов)
        best_threshold_cv = np.mean(cv_thresholds_sorted[1:-1])
    else:
        best_threshold_cv = np.mean(cv_thresholds)

    cv_threshold_std = np.std(cv_thresholds)
    medical_cost_cv_mean = np.mean(cv_medical_costs)
    medical_cost_cv_std = np.std(cv_medical_costs)
    medical_cost_cv_coef = medical_cost_cv_std / medical_cost_cv_mean if medical_cost_cv_mean > 0 else 1.0

    pr_auc_mean = np.mean(cv_pr_auc_scores)
    pr_auc_std = np.std(cv_pr_auc_scores)

    # =========================================================================
    # 2. ОБУЧЕНИЕ НА ВСЁМ TRAIN
    # =========================================================================
    model.fit(X_train, y_train)
    converged = check_convergence(model)

    train_probs = model.predict_proba(X_train)[:, 1]
    train_pr_auc = average_precision_score(y_train, train_probs)

    # Train Medical Cost (для оценки переобучения по бизнес-метрике)
    train_metrics_at_threshold = get_metrics_at_threshold(
        y_train, train_probs, best_threshold_cv, cost_fn, cost_fp
    )
    train_medical_cost = train_metrics_at_threshold['medical_cost']
    medical_cost_stability_gap = train_medical_cost - medical_cost_cv_mean

    # ВНИМАНИЕ: Это приближённая оценка, НЕ прямое сравнение!
    # Сравнивает PR-AUC на train (некалиброванные вероятности) с PR-AUC на CV (калиброванные по фолдам).
    # Использовать ТОЛЬКО для detection переобучения, НЕ для сравнения моделей между собой.
    cv_train_gap_approx = train_pr_auc - pr_auc_mean

    if feature_names is not None and hasattr(model, 'coef_'):
        top_feature_or = get_top_odds_ratio(model, feature_names)
        bottom_feature_or = get_bottom_odds_ratio(model, feature_names)
    else:
        top_feature_or = 'N/A'
        bottom_feature_or = 'N/A'

    # =========================================================================
    # 3. КАЛИБРОВКА (если включена и нужна)
    # =========================================================================
    if CALIBRATION_ENABLED and is_calibration_needed(model):
        calibrated = CalibratedClassifierCV(model, cv=CALIBRATION_CV, method=CALIBRATION_METHOD)
        calibrated.fit(X_train, y_train)
        probs_val_calib = calibrated.predict_proba(X_val)[:, 1]
        calibrated_model = calibrated
    else:
        probs_val_calib = model.predict_proba(X_val)[:, 1]
        calibrated_model = None

    # =========================================================================
    # 4. МЕТРИКИ КАЛИБРОВКИ
    # =========================================================================
    probs_val_base = model.predict_proba(X_val)[:, 1]

    bss_before = brier_skill_score(y_val, probs_val_base)
    bss_after = brier_skill_score(y_val, probs_val_calib) if calibrated_model is not None else bss_before
    calibration_gain = bss_after - bss_before

    reliability_slope = get_reliability_slope(y_val, probs_val_calib)
    ece, mce = calculate_ece_mce(y_val, probs_val_calib, n_bins=10)

    # =========================================================================
    # 5. МЕТРИКИ НА VAL С CV-ПОРОГОМ
    # =========================================================================
    val_metrics = get_metrics_at_threshold(y_val, probs_val_calib, best_threshold_cv, cost_fn, cost_fp)

    recall_lower_ci, recall_upper_ci = recall_confidence_interval(
        val_metrics['tp'], val_metrics['fn']
    )

    selection_rate_pct = val_metrics['selection_rate'] * 100

    # =========================================================================
    # 6. СБОР РЕЗУЛЬТАТОВ (НЕ СОХРАНЯЕМ probs_calib И y_val ДЛЯ ЭКОНОМИИ ПАМЯТИ)
    # =========================================================================
    results.update({
        'Converged': '✅' if converged else '❌',
        'PR_AUC_mean': pr_auc_mean,
        'PR_AUC_std': pr_auc_std,
        'Stability_Gap': cv_train_gap_approx,
        'CV_Stability_std': cv_threshold_std,
        'Top_Feature_OR': top_feature_or,
        'Bottom_Feature_OR': bottom_feature_or,

        'BSS_Before': bss_before,
        'BSS_After': bss_after,
        'Calibration_Gain': calibration_gain,
        'Reliability_Slope': reliability_slope,
        'ECE': ece,
        'MCE': mce,

        'Threshold_CV': best_threshold_cv,
        'Medical_Cost_CV': medical_cost_cv_mean,
        'Medical_Cost_CV_std': medical_cost_cv_std,
        'Medical_Cost_CV_coef': medical_cost_cv_coef,

        'Train_Medical_Cost': train_medical_cost,
        'Medical_Cost_Stability_Gap': medical_cost_stability_gap,

        'Val_Recall': val_metrics['recall'],
        'Val_Precision': val_metrics['precision'],
        'Val_FN': val_metrics['fn'],
        'Val_FP': val_metrics['fp'],
        'Val_TP': val_metrics['tp'],
        'Val_TN': val_metrics['tn'],
        'Val_NNI': val_metrics['nni'],
        'Medical_Cost': val_metrics['medical_cost'],
        'Selection_Rate': selection_rate_pct,
        'Recall_Lower_CI': recall_lower_ci,
        'Recall_Upper_CI': recall_upper_ci,

        'base_model': model,
        'calibrated_model': calibrated_model,
        'threshold_cv_std': cv_threshold_std,

        # ПРИМЕЧАНИЕ: probs_calib и y_val НЕ сохраняются для экономии памяти
        # При необходимости их можно восстановить из base_model/calibrated_model
    })

    return results


print("✅ run_single_experiment_cv() загружена (исправленная версия)")

✅ run_single_experiment_cv() загружена (исправленная версия)


In [17]:
# =============================================================================
# 3.4 АВТОМАТИЧЕСКИЙ ВЫБОР ЛУЧШЕЙ МОДЕЛИ (ТРИ УРОВНЯ ФИЛЬТРОВ)
# =============================================================================

def apply_filters(df_results, filter_settings=None):
    """
    Применяет три уровня фильтров к результатам экспериментов.

    Параметры:
    ----------
    df_results : pd.DataFrame
        DataFrame с результатами экспериментов
    filter_settings : dict, optional
        Словарь с настройками фильтров (если None, используются глобальные)

    Возвращает:
    -----------
    df_filtered : pd.DataFrame
        Отфильтрованный DataFrame
    stats : dict
        Статистика по количеству моделей после каждого уровня
    filter_log : dict
        Детальный лог, сколько моделей отсек каждый фильтр
    """
    if filter_settings is None:
        filter_settings = {
            # Уровень 1
            'min_bss': FILTER_MIN_BSS,
            'require_converged': FILTER_REQUIRE_CONVERGED,
            'max_stability_gap': FILTER_MAX_STABILITY_GAP,
            # Уровень 2
            'max_cv_stability_std': FILTER_MAX_CV_STABILITY_STD,
            'max_mce': FILTER_MAX_MCE,
            'max_ece': FILTER_MAX_ECE,
            'min_reliability_slope': FILTER_RELIABILITY_SLOPE_MIN,
            'max_reliability_slope': FILTER_RELIABILITY_SLOPE_MAX,
            'min_pr_auc': FILTER_MIN_PR_AUC,
            'max_medical_cost_cv_coef': FILTER_MAX_MEDICAL_COST_CV_COEF,
            # Уровень 3
            'min_recall_val': FILTER_MIN_RECALL_VAL,
            'min_precision_val': FILTER_MIN_PRECISION_VAL,
            'max_selection_rate_val': FILTER_MAX_SELECTION_RATE_VAL,
            'max_nni_val': FILTER_MAX_NNI_VAL,
            'min_tp_val': FILTER_MIN_TP_VAL,
            'min_recall_lower_ci': FILTER_MIN_RECALL_LOWER_CI,
        }

    # Проверка наличия необходимых колонок
    required_cols = ['BSS_After', 'Converged', 'Stability_Gap', 'CV_Stability_std',
                     'MCE', 'ECE', 'Reliability_Slope', 'PR_AUC_mean', 'Val_Recall',
                     'Val_Precision', 'Selection_Rate', 'Val_NNI', 'Val_TP', 'Recall_Lower_CI',
                     'Medical_Cost_CV_coef']  # ← теперь обязательный

    missing_cols = [col for col in required_cols if col not in df_results.columns]
    if missing_cols:
        raise ValueError(f"В df_results отсутствуют необходимые колонки: {missing_cols}")

    df = df_results.copy()
    initial_count = len(df)
    filter_log = {name: 0 for name in [
        'BSS_After', 'Converged', 'Stability_Gap',
        'CV_Stability_std', 'MCE', 'ECE', 'Reliability_Slope_low',
        'Reliability_Slope_high', 'PR_AUC_mean', 'Medical_Cost_CV_coef',
        'Val_Recall', 'Val_Precision', 'Selection_Rate',
        'Val_NNI', 'Val_TP', 'Recall_Lower_CI'
    ]}

    # =========================================================================
    # Уровень 1: Базовые технические фильтры
    # =========================================================================
    before = len(df)
    df = df[df['BSS_After'] > filter_settings['min_bss']]
    filter_log['BSS_After'] = before - len(df)

    if filter_settings['require_converged']:
        before = len(df)
        df = df[df['Converged'] == '✅']
        filter_log['Converged'] = before - len(df)

    before = len(df)
    df = df[df['Stability_Gap'] <= filter_settings['max_stability_gap']]
    filter_log['Stability_Gap'] = before - len(df)
    level1_count = len(df)

    # =========================================================================
    # Уровень 2: Расширенные технические фильтры
    # =========================================================================
    before = len(df)
    df = df[df['CV_Stability_std'] <= filter_settings['max_cv_stability_std']]
    filter_log['CV_Stability_std'] = before - len(df)

    before = len(df)
    df = df[df['MCE'] <= filter_settings['max_mce']]
    filter_log['MCE'] = before - len(df)

    before = len(df)
    df = df[df['ECE'] <= filter_settings['max_ece']]
    filter_log['ECE'] = before - len(df)

    before = len(df)
    df = df[df['Reliability_Slope'] >= filter_settings['min_reliability_slope']]
    filter_log['Reliability_Slope_low'] = before - len(df)

    before = len(df)
    df = df[df['Reliability_Slope'] <= filter_settings['max_reliability_slope']]
    filter_log['Reliability_Slope_high'] = before - len(df)

    before = len(df)
    df = df[df['PR_AUC_mean'] >= filter_settings['min_pr_auc']]
    filter_log['PR_AUC_mean'] = before - len(df)

    before = len(df)
    df = df[df['Medical_Cost_CV_coef'] <= filter_settings['max_medical_cost_cv_coef']]
    filter_log['Medical_Cost_CV_coef'] = before - len(df)

    level2_count = len(df)

    # =========================================================================
    # Уровень 3: Клинические фильтры
    # =========================================================================
    before = len(df)
    df = df[df['Val_Recall'] >= filter_settings['min_recall_val']]
    filter_log['Val_Recall'] = before - len(df)

    before = len(df)
    df = df[df['Val_Precision'] >= filter_settings['min_precision_val']]
    filter_log['Val_Precision'] = before - len(df)

    before = len(df)
    df = df[df['Selection_Rate'] <= filter_settings['max_selection_rate_val'] * 100]
    filter_log['Selection_Rate'] = before - len(df)

    before = len(df)
    df = df[df['Val_NNI'] <= filter_settings['max_nni_val']]
    filter_log['Val_NNI'] = before - len(df)

    before = len(df)
    df = df[df['Val_TP'] >= filter_settings['min_tp_val']]
    filter_log['Val_TP'] = before - len(df)

    before = len(df)
    df = df[df['Recall_Lower_CI'] >= filter_settings['min_recall_lower_ci']]
    filter_log['Recall_Lower_CI'] = before - len(df)
    level3_count = len(df)

    stats = {
        'initial': initial_count,
        'after_level1': level1_count,
        'after_level2': level2_count,
        'after_level3': level3_count
    }

    return df, stats, filter_log


def auto_select_best_model(df_results):
    """
    Автоматический выбор лучшей модели:
    1. Применяет три уровня фильтров
    2. Если нет прошедших — возвращает None и статистику
    3. Среди прошедших выбирает модель с минимальным Medical_Cost_CV (CV-стоимость)

    Параметры:
    ----------
    df_results : pd.DataFrame
        DataFrame с результатами экспериментов

    Возвращает:
    -----------
    best_row : pd.Series or None
        Строка с лучшей моделью
    selection_details : dict or None
        Детали выбора
    """
    print("\n" + "=" * 70)
    print("🤖 АВТОМАТИЧЕСКИЙ ВЫБОР МОДЕЛИ")
    print("=" * 70)

    filtered_df, stats, filter_log = apply_filters(df_results)

    print(f"\n📊 СТАТИСТИКА ФИЛЬТРАЦИИ:")
    print(f"   Всего экспериментов: {stats['initial']}")
    print(f"   После Уровня 1 (технический): {stats['after_level1']}")
    print(f"   После Уровня 2 (расширенный): {stats['after_level2']}")
    print(f"   После Уровня 3 (клинический): {stats['after_level3']}")

    print(f"\n📋 ФИЛЬТРЫ, ОТСЕКШИЕ БОЛЬШЕ ВСЕГО:")
    sorted_filters = sorted(filter_log.items(), key=lambda x: x[1], reverse=True)
    for name, count in sorted_filters[:5]:
        if count > 0:
            print(f"   • {name}: отсечено {count} моделей")

    if len(filtered_df) == 0:
        print("\n❌ НИ ОДНА МОДЕЛЬ НЕ ПРОШЛА ВСЕ ФИЛЬТРЫ!")
        return None, None

    # ВАЖНО: сортировка по Medical_Cost_CV (средняя стоимость на CV), а не по Medical_Cost
    # Это более робастная метрика, устойчивая к переобучению под конкретную валидацию
    filtered_df = filtered_df.sort_values('Medical_Cost_CV', ascending=True)

    best_row = filtered_df.iloc[0]
    model_name = best_row['Experiment']

    if '_grid_cost' in best_row and not pd.isna(best_row['_grid_cost']):
        selected_cost = int(best_row['_grid_cost'])
    else:
        import re
        match = re.search(r'FN(\d+)', model_name)
        selected_cost = int(match.group(1)) if match else COST_FN

    print(f"\n🏆 ПОБЕДИТЕЛЬ:")
    print(f"   Модель: {model_name}")
    print(f"   COST_FN: {selected_cost} (штраф за FN)")
    print(f"   Medical Cost (CV): {best_row['Medical_Cost_CV']:.0f}")
    print(f"   Medical Cost (VAL): {best_row['Medical_Cost']:.0f}")
    print(f"   Recall (VAL): {best_row['Val_Recall']:.3f}")
    print(f"   Precision (VAL): {best_row['Val_Precision']:.3f}")
    print(f"   Selection Rate: {best_row['Selection_Rate']:.1f}%")
    print(f"   Threshold: {best_row['Threshold_CV']:.4f}")

    if len(filtered_df) >= 3:
        print(f"\n📋 Топ-3 моделей (для справки):")
        for i, (_, row) in enumerate(filtered_df.head(3).iterrows(), 1):
            print(f"   {i}. {row['Experiment']}: Cost_CV={row['Medical_Cost_CV']:.0f}, "
                  f"Recall={row['Val_Recall']:.3f}")

    selection_details = {
        'mode': 'auto',
        'selected_model': model_name,
        'selected_cost_fn': selected_cost,
        'medical_cost_cv': float(best_row['Medical_Cost_CV']),
        'medical_cost_val': float(best_row['Medical_Cost']),
        'val_recall': float(best_row['Val_Recall']),
        'threshold': float(best_row['Threshold_CV']),
        'filters_passed': stats['after_level3'],
        'total_experiments': stats['initial']
    }

    return best_row, selection_details


print("✅ auto_select_best_model() загружена (исправленная версия)")

✅ auto_select_best_model() загружена (исправленная версия)


In [18]:
# =============================================================================
# 3.5 СТАТИСТИЧЕСКАЯ ЗНАЧИМОСТЬ КОЭФФИЦИЕНТОВ
# =============================================================================

def get_coefficient_significance(model, X_train, y_train, feature_names, alpha=0.05):
    """
    Рассчитывает p-values и доверительные интервалы для коэффициентов
    логистической регрессии через Fisher Information Matrix.

    ВНИМАНИЕ:
    - Для L1-регуляризации (Lasso) p-values НЕВОЗМОЖНО рассчитать корректно.
      Возвращаются только коэффициенты и Odds Ratios.
    - Для L2-регуляризации (Ridge) p-values are approximate и могут быть неточны.
    - При сингулярной матрице p-values также невалидны.
    """
    from scipy import stats

    if not hasattr(model, 'coef_'):
        raise ValueError("Модель не имеет coef_. Этот метод работает только для LogisticRegression.")

    coefs = model.coef_[0]
    penalty = getattr(model, 'penalty', 'unknown')

    # ========================================================================
    # L1-регуляризация: p-values невозможны
    # ========================================================================
    if penalty == 'l1':
        print("\n" + "=" * 70)
        print("⚠️ ВНИМАНИЕ: L1-регуляризация (Lasso) - p-values статистически НЕВАЛИДНЫ.")
        print("   Для оценки важности признаков используйте абсолютные значения коэффициентов.")
        print("   Доверительные интервалы не рассчитываются.")
        print("=" * 70 + "\n")

        odds_ratios = np.exp(coefs)
        interpretations = []
        for or_val in odds_ratios:
            if or_val > 1:
                interpretations.append(f"Увеличивает шансы на {((or_val-1)*100):.1f}%")
            elif or_val < 1:
                interpretations.append(f"Снижает шансы на {((1-or_val)*100):.1f}%")
            else:
                interpretations.append("Не влияет")

        df_coef = pd.DataFrame({
            'Feature': feature_names,
            'Coefficient': coefs,
            'OR': odds_ratios,
            'Interpretation': interpretations,
            'Signif': '— (L1: p-value не определён)'
        })
        df_coef = df_coef.sort_values('Coefficient', key=lambda x: x.abs(), ascending=False).reset_index(drop=True)
        return df_coef

    # ========================================================================
    # L2 и другие типы регуляризации (приближённые p-values)
    # ========================================================================
    if penalty in ['l2', 'none']:
        print("\n" + "=" * 70)
        print(f"ℹ️ Penalty='{penalty}'. p-values рассчитываются через Fisher Information.")
        print("   ВНИМАНИЕ: Это ПРИБЛИЖЁННЫЕ значения. Для регуляризованных моделей")
        print("   p-values НЕ являются строго статистически корректными.")
        print("   Используйте их только для ОРИЕНТИРОВОЧНОЙ оценки важности признаков.")
        print("=" * 70 + "\n")

    # Рассчитываем стандартные ошибки через Fisher Information
    X_with_intercept = np.column_stack([np.ones(len(X_train)), X_train.values])
    probs = model.predict_proba(X_train)[:, 1]

    W = probs * (1 - probs)
    XW = X_with_intercept * W[:, np.newaxis]
    fisher_info = XW.T @ X_with_intercept

    try:
        cov_matrix = np.linalg.inv(fisher_info)
    except np.linalg.LinAlgError:
        cov_matrix = np.linalg.pinv(fisher_info)
        print("   ⚠️ ВНИМАНИЕ: Fisher Information matrix сингулярна, использована псевдообратная матрица.")
        print("   p-values и доверительные интервалы могут быть невалидны.\n")

    standard_errors = np.sqrt(np.diag(cov_matrix))
    se_coefs = standard_errors[1:len(coefs)+1]

    z_scores = coefs / se_coefs
    p_values = 2 * (1 - stats.norm.cdf(np.abs(z_scores)))

    z_critical = stats.norm.ppf(1 - alpha/2)
    ci_lower_coef = coefs - z_critical * se_coefs
    ci_upper_coef = coefs + z_critical * se_coefs

    odds_ratios = np.exp(coefs)
    ci_lower_or = np.exp(ci_lower_coef)
    ci_upper_or = np.exp(ci_upper_coef)

    significance = []
    for p in p_values:
        if p < 0.001:
            significance.append("***")
        elif p < 0.01:
            significance.append("**")
        elif p < 0.05:
            significance.append("*")
        else:
            significance.append("—")

    interpretations = []
    for or_val in odds_ratios:
        if or_val > 1:
            interpretations.append(f"Увеличивает шансы на {((or_val-1)*100):.1f}%")
        elif or_val < 1:
            interpretations.append(f"Снижает шансы на {((1-or_val)*100):.1f}%")
        else:
            interpretations.append("Не влияет")

    df_coef = pd.DataFrame({
        'Feature': feature_names,
        'Coefficient': coefs,
        'Std_Error': se_coefs,
        'z_score': z_scores,
        'p_value': p_values,
        'Signif': significance,
        'OR': odds_ratios,
        'CI_lower_95': ci_lower_or,
        'CI_upper_95': ci_upper_or,
        'Interpretation': interpretations
    })

    df_coef = df_coef.sort_values('Coefficient', key=lambda x: x.abs(), ascending=False).reset_index(drop=True)

    return df_coef


def plot_forest_plot(df_coef, title="Odds Ratios with 95% CI", save_path=None, figsize=None):
    """
    Строит Forest Plot для визуализации Odds Ratio.

    Параметры:
    ----------
    df_coef : pd.DataFrame
        DataFrame из get_coefficient_significance
    title : str, default="Odds Ratios with 95% CI"
        Заголовок графика
    save_path : str, optional
        Путь для сохранения
    figsize : tuple, optional
        Размер фигуры (по умолчанию подбирается автоматически)

    Возвращает:
    -----------
    fig : matplotlib.figure.Figure
        Figure объект
    """
    n_features = len(df_coef)
    if figsize is None:
        figsize = (10, max(6, n_features * 0.4))

    fig, ax = plt.subplots(figsize=figsize)

    features = df_coef['Feature'].tolist()
    or_values = df_coef['OR'].tolist()
    ci_lower = df_coef['CI_lower_95'].tolist() if 'CI_lower_95' in df_coef.columns else [None] * n_features
    ci_upper = df_coef['CI_upper_95'].tolist() if 'CI_upper_95' in df_coef.columns else [None] * n_features
    significance = df_coef['Signif'].tolist()

    colors = ['crimson' if or_val > 1 else 'steelblue' if or_val < 1 else 'gray'
              for or_val in or_values]

    for i, (feat, or_val, ci_low, ci_up, color) in enumerate(zip(features, or_values, ci_lower, ci_upper, colors)):
        if ci_low is not None and ci_up is not None:
            ax.hlines(y=i, xmin=ci_low, xmax=ci_up, color=color, linewidth=2)
        ax.scatter(or_val, i, color=color, s=80, zorder=5, edgecolors='black')

        signif_star = significance[i]
        if signif_star not in ["—", "— (L1: p-value не определён)"]:
            ax.annotate(signif_star, xy=(or_val, i), xytext=(5, 5),
                       textcoords='offset points', fontsize=10, color='darkgreen')

    ax.axvline(x=1, color='gray', linestyle='--', linewidth=1, label='OR = 1 (нет эффекта)')
    ax.set_yticks(range(len(features)))
    ax.set_yticklabels(features)
    ax.set_xlabel('Odds Ratio (логарифмическая шкала)')
    ax.set_title(title)
    ax.legend(loc='lower right')
    ax.grid(axis='x', alpha=0.3)
    ax.set_xscale('log')

    ax.annotate('* p < 0.05, ** p < 0.01, *** p < 0.001\nПоложительный OR → фактор риска, Отрицательный → защитный',
                xy=(0.02, 0.02), xycoords='axes fraction', fontsize=8, color='gray')

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')

    return fig


def plot_coefficient_importance(df_coef, top_n=15, save_path=None):
    """
    Визуализация важности коэффициентов (абсолютные значения коэффициентов).

    Параметры:
    ----------
    df_coef : pd.DataFrame
        DataFrame из get_coefficient_significance
    top_n : int, default=15
        Количество признаков для отображения
    save_path : str, optional
        Путь для сохранения графика

    Возвращает:
    -----------
    fig : matplotlib.figure.Figure
        Figure объект
    """
    # Берём топ по АБСОЛЮТНОМУ значению коэффициента
    plot_df = df_coef.iloc[df_coef['Coefficient'].abs().nlargest(top_n).index]
    plot_df = plot_df.sort_values('Coefficient', ascending=True)  # Для горизонтального barh

    colors = ['crimson' if coef > 0 else 'steelblue' for coef in plot_df['Coefficient']]

    fig, ax = plt.subplots(figsize=(10, max(6, len(plot_df) * 0.4)))
    bars = ax.barh(plot_df['Feature'], plot_df['Coefficient'], color=colors, alpha=0.7)

    for i, (idx, row) in enumerate(plot_df.iterrows()):
        if row['Signif'] not in ["—", "— (L1: p-value не определён)"]:
            ax.annotate(row['Signif'], xy=(row['Coefficient'], i),
                        xytext=(5, 0), textcoords='offset points',
                        fontsize=10, color='darkgreen')

    ax.axvline(x=0, color='gray', linestyle='-', linewidth=0.5)
    ax.set_xlabel('Coefficient (логарифм Odds Ratio)')
    ax.set_ylabel('Feature')
    ax.set_title(f'Feature Importance (топ-{top_n} по |coefficient|)')
    ax.grid(axis='x', alpha=0.3)

    ax.annotate('* p < 0.05, ** p < 0.01, *** p < 0.001\nПоложительный = фактор риска, Отрицательный = защитный фактор',
                xy=(0.02, 0.02), xycoords='axes fraction', fontsize=8, color='gray')

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')

    return fig


print("✅ Функции статистической значимости загружены")

✅ Функции статистической значимости загружены


In [19]:
# =============================================================================
# 3.6 ГЕНЕРАЦИЯ СЕТКИ И ЗАПУСК ВСЕХ ЭКСПЕРИМЕНТОВ
# =============================================================================

def generate_grid_experiments(base_cost_fn=None, base_cost_fp=None):
    """
    Генерирует сетку экспериментов:
    - Все комбинации COST × penalty × C
    - Плюс Dummy baseline

    ВАЖНО: Не изменяет глобальные переменные.
    Стоимость передаётся в каждую конфигурацию эксперимента как параметр.

    Параметры:
    ----------
    base_cost_fn : float, optional
        Базовый штраф за FN (по умолчанию COST_FN)
    base_cost_fp : float, optional
        Базовый штраф за FP (по умолчанию COST_FP)

    Возвращает:
    -----------
    experiments : list of dict
        Список конфигураций экспериментов
    """
    if base_cost_fn is None:
        base_cost_fn = COST_FN
    if base_cost_fp is None:
        base_cost_fp = COST_FP

    experiments = []

    if base_cost_fn > base_cost_fp:
        cost_type = 'FN'
        grid_costs = GRID_COST_VALUES
    else:
        cost_type = 'FP'
        grid_costs = GRID_COST_VALUES

    for cost_val in grid_costs:
        for penalty in GRID_PENALTIES:
            for C in GRID_C_VALUES:
                name = f"{cost_type}{cost_val}_{penalty.upper()}_C={C}"

                experiments.append({
                    'name': name,
                    'model_type': 'logreg',
                    'penalty': penalty,
                    'C': C,
                    '_cost_fn': cost_val if cost_type == 'FN' else base_cost_fn,
                    '_cost_fp': cost_val if cost_type == 'FP' else base_cost_fp,
                    '_cost_used': cost_val,
                    '_cost_type': cost_type
                })

    experiments.append({
        'name': 'Dummy_stratified',
        'model_type': 'dummy',
        'strategy': 'stratified',
        '_cost_fn': base_cost_fn,
        '_cost_fp': base_cost_fp,
        '_cost_used': base_cost_fn if base_cost_fn > base_cost_fp else base_cost_fp,
        '_cost_type': cost_type
    })

    return experiments


def run_dummy_experiment_cv(model, model_name, X_train, y_train, X_val, y_val,
                            cost_fn=None, cost_fp=None, cv_folds=5, random_state=42):
    """
    Запускает Dummy baseline с ЧЕСТНОЙ CV-оптимизацией порога (как у реальных моделей).

    ВНИМАНИЕ: Dummy не требует CV для обучения, но порог выбирается через CV
    для честного сравнения с другими моделями.
    """
    from sklearn.base import clone

    if cost_fn is None:
        cost_fn = COST_FN
    if cost_fp is None:
        cost_fp = COST_FP

    # =========================================================================
    # 1. КРОСС-ВАЛИДАЦИЯ ДЛЯ ПОИСКА ОПТИМАЛЬНОГО ПОРОГА
    # =========================================================================
    cv = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=random_state)

    cv_thresholds = []
    cv_medical_costs = []

    print(f"   CV поиск порога для Dummy...", end='', flush=True)

    for train_idx, val_idx in cv.split(X_train, y_train):
        X_fold_train = X_train.iloc[train_idx]
        X_fold_val = X_train.iloc[val_idx]
        y_fold_train = y_train.iloc[train_idx]
        y_fold_val = y_train.iloc[val_idx]

        # Dummy обучается на фолде (но Dummy не обучается в классическом смысле)
        fold_model = clone(model)
        fold_model.fit(X_fold_train, y_fold_train)
        fold_probs = fold_model.predict_proba(X_fold_val)[:, 1]

        threshold, threshold_metrics = find_optimal_threshold_by_cost(
            y_fold_val, fold_probs, cost_fn, cost_fp
        )

        cv_thresholds.append(threshold)
        cv_medical_costs.append(threshold_metrics['medical_cost'])

    print(f" готово")

    # Усредняем порог по фолдам
    best_threshold_cv = np.mean(cv_thresholds)
    medical_cost_cv_mean = np.mean(cv_medical_costs)
    medical_cost_cv_std = np.std(cv_medical_costs)
    medical_cost_cv_coef = medical_cost_cv_std / medical_cost_cv_mean if medical_cost_cv_mean > 0 else 1.0

    # =========================================================================
    # 2. ОБУЧЕНИЕ НА ВСЁМ TRAIN
    # =========================================================================
    model.fit(X_train, y_train)
    probs_val = model.predict_proba(X_val)[:, 1]

    # =========================================================================
    # 3. МЕТРИКИ НА VAL С CV-ПОРОГОМ
    # =========================================================================
    val_metrics = get_metrics_at_threshold(y_val, probs_val, best_threshold_cv, cost_fn, cost_fp)

    # =========================================================================
    # 4. МЕТРИКИ КАЛИБРОВКИ ДЛЯ DUMMY
    # =========================================================================
    bss = brier_skill_score(y_val, probs_val)
    reliability_slope = get_reliability_slope(y_val, probs_val)
    ece, mce = calculate_ece_mce(y_val, probs_val, n_bins=10)

    return {
        'Experiment': model_name,
        'Converged': 'N/A',
        'PR_AUC_mean': 0.0,
        'PR_AUC_std': 0.0,
        'Stability_Gap': 0.0,
        'CV_Stability_std': 0.0,
        'Top_Feature_OR': 'N/A',
        'Bottom_Feature_OR': 'N/A',
        'BSS_Before': bss,
        'BSS_After': bss,
        'Calibration_Gain': 0.0,
        'Reliability_Slope': reliability_slope,
        'ECE': ece,
        'MCE': mce,
        'Threshold_CV': best_threshold_cv,
        'Medical_Cost_CV': medical_cost_cv_mean,
        'Medical_Cost_CV_std': medical_cost_cv_std,
        'Medical_Cost_CV_coef': medical_cost_cv_coef,
        'Val_Recall': val_metrics['recall'],
        'Val_Precision': val_metrics['precision'],
        'Val_FN': val_metrics['fn'],
        'Val_FP': val_metrics['fp'],
        'Val_TP': val_metrics['tp'],
        'Val_TN': val_metrics['tn'],
        'Val_NNI': val_metrics['nni'],
        'Medical_Cost': val_metrics['medical_cost'],
        'Selection_Rate': val_metrics['selection_rate'] * 100,
        'Recall_Lower_CI': val_metrics['recall'],
        'Recall_Upper_CI': val_metrics['recall'],
        'base_model': model,
        'calibrated_model': None,
        'threshold_cv_std': np.std(cv_thresholds),
        '_grid_cost': cost_fn if cost_fn > cost_fp else cost_fp
    }


def run_all_experiments(X_train, y_train, X_val, y_val, feature_names=None,
                        random_state=42, cost_fn=None, cost_fp=None):
    """
    Запускает все эксперименты из сетки с CV-оптимизацией порога по Medical Cost.

    ВАЖНО: Не изменяет глобальные переменные COST_FN, COST_FP.
    Каждый эксперимент использует свои значения стоимости.

    Параметры:
    ----------
    X_train, y_train : pd.DataFrame, pd.Series
        Train данные
    X_val, y_val : pd.DataFrame, pd.Series
        Val данные
    feature_names : list, optional
        Список признаков
    random_state : int, default=42
        random_state для воспроизводимости
    cost_fn : float, optional
        Базовый штраф за FN (по умолчанию COST_FN)
    cost_fp : float, optional
        Базовый штраф за FP (по умолчанию COST_FP)

    Возвращает:
    -----------
    df_results : pd.DataFrame
        DataFrame со всеми результатами
    """
    if cost_fn is None:
        cost_fn = COST_FN
    if cost_fp is None:
        cost_fp = COST_FP

    experiments = generate_grid_experiments(cost_fn, cost_fp)

    print("=" * 70)
    print("🚀 ЗАПУСК ЭКСПЕРИМЕНТОВ (CV + Medical Cost оптимизация)")
    print("=" * 70)
    print(f"Всего экспериментов: {len(experiments)}")
    print(f"Базовый COST_FN: {cost_fn}, COST_FP: {cost_fp}")
    print(f"Сетка FN ∈ {GRID_COST_VALUES}, penalty ∈ {GRID_PENALTIES}, C ∈ {GRID_C_VALUES}")
    print(f"Калибровка: {'✅ ВКЛ' if CALIBRATION_ENABLED else '❌ ВЫКЛ'}")
    print(f"Random state: {random_state}")
    print("\n⏳ Выполнение...\n")

    results = []
    failed_experiments = []
    start_time = time.time()

    for i, exp in enumerate(experiments, 1):
        print(f"   [{i}/{len(experiments)}] {exp['name']}...", end='', flush=True)

        try:
            exp_cost_fn = exp.get('_cost_fn', cost_fn)
            exp_cost_fp = exp.get('_cost_fp', cost_fp)

            model = build_model(exp, BASE_CONFIG)

            if exp.get('model_type') == 'dummy':
                result = run_dummy_experiment_cv(
                    model, exp['name'], X_train, y_train, X_val, y_val,
                    cost_fn=exp_cost_fn, cost_fp=exp_cost_fp,
                    cv_folds=CV_FOLDS, random_state=random_state
                )
            else:
                result = run_single_experiment_cv(
                    model=model,
                    model_name=exp['name'],
                    X_train=X_train, y_train=y_train,
                    X_val=X_val, y_val=y_val,
                    exp_config=exp,
                    feature_names=feature_names,
                    cv_folds=CV_FOLDS,
                    random_state=random_state,
                    cost_fn=exp_cost_fn,
                    cost_fp=exp_cost_fp
                )

            result['_cost_fn_used'] = exp_cost_fn
            result['_cost_fp_used'] = exp_cost_fp
            result['_grid_cost'] = exp.get('_cost_used', exp_cost_fn)

            results.append(result)
            print(" ✓", flush=True)

        except Exception as e:
            print(f" ❌ ОШИБКА: {e}", flush=True)
            print(f"      Эксперимент {exp['name']} пропущен", flush=True)
            failed_experiments.append({  # <-- ДОБАВИТЬ ЭТИ 3 СТРОКИ
                'experiment': exp['name'],
                'error': str(e)
            })
            continue

    elapsed_time = time.time() - start_time
    print(f"\n⏱️ Затраченное время: {elapsed_time:.1f} секунд")
    print(f"   Успешно завершено: {len(results)} из {len(experiments)} экспериментов")

    df_results = pd.DataFrame(results)

    # Сохраняем информацию о неудачных экспериментах (только для отладки, без метрик)
    if failed_experiments:
        print(f"\n⚠️ НЕУДАЧНЫЕ ЭКСПЕРИМЕНТЫ ({len(failed_experiments)}):")
        for fail in failed_experiments[:5]:
            print(f"   • {fail['experiment']}: {fail['error'][:80]}")
        if len(failed_experiments) > 5:
            print(f"   ... и ещё {len(failed_experiments) - 5} экспериментов")

        # Сохраняем в файл (не выводим в ноутбук, чтобы не подглядывать)
        failed_path = os.path.join(MODELLING_REPORTS_PATH, 'failed_experiments.json')
        with open(failed_path, 'w') as f:
            json.dump(failed_experiments, f, indent=2)
        print(f"   💾 Подробности сохранены в: {failed_path}")

    print("\n" + "=" * 70)
    print("✅ ЭКСПЕРИМЕНТЫ ЗАВЕРШЕНЫ")
    print("=" * 70)

    return df_results


print("✅ run_all_experiments() загружена (исправленная версия)")

✅ run_all_experiments() загружена (исправленная версия)


In [20]:
# =============================================================================
# 3.7 ФОРМАТИРОВАНИЕ ВЫВОДА
# =============================================================================

def format_feature_name_for_table(feature_name, max_len=25, preserve_arrow=True):
    """
    Сокращение названия признака для таблиц.

    Параметры:
    ----------
    feature_name : str
        Название признака (может содержать "↑", "↓", ":", числа)
    max_len : int, default=25
        Максимальная длина строки
    preserve_arrow : bool, default=True
        Сохранять ли стрелки ↑/↓ в конце

    Возвращает:
    -----------
    str : Отформатированное название
    """
    if pd.isna(feature_name) or feature_name == 'N/A' or feature_name == '':
        return 'N/A'

    feature_name = str(feature_name)

    if len(feature_name) <= max_len:
        return feature_name

    arrow = ''
    if preserve_arrow:
        if '↑' in feature_name:
            arrow = ' ↑'
            feature_name = feature_name.replace('↑', '').strip()
        elif '↓' in feature_name:
            arrow = ' ↓'
            feature_name = feature_name.replace('↓', '').strip()

    if ': ' in feature_name:
        parts = feature_name.split(': ', 1)
        feature_part = parts[0]
        value_part = parts[1] if len(parts) > 1 else ''

        # Извлекаем числа из value_part
        numbers = re.findall(r'[\d.]+', value_part)
        value_short = numbers[0] if numbers else ''

        if len(feature_part) > 18:
            feature_short = feature_part[:10] + "..." + feature_part[-6:]
        else:
            feature_short = feature_part

        result = f"{feature_short}: {value_short}{arrow}"
        return result[:max_len]
    else:
        head_len = max_len // 2 - 2
        tail_len = max_len // 2 - 2
        if head_len > 0 and tail_len > 0:
            return feature_name[:head_len] + "..." + feature_name[-tail_len:]
        else:
            return feature_name[:max_len - 3] + "..."


def print_pretty_table(df, title="", max_col_width=35):
    """
    Красивая печать таблицы в консоль.

    Параметры:
    ----------
    df : pd.DataFrame
        Данные для вывода
    title : str, default=""
        Заголовок таблицы
    max_col_width : int, default=35
        Максимальная ширина колонки
    """
    if title:
        print("\n" + "=" * 100)
        print(title)
        print("=" * 100)

    df_str = df.astype(str)

    col_widths = {}
    for col in df_str.columns:
        max_len = max(len(col), df_str[col].map(len).max())
        col_widths[col] = min(max_len + 2, max_col_width)

    # Печатаем заголовок
    header_parts = [col.center(col_widths[col]) for col in df_str.columns]
    print("| " + " | ".join(header_parts) + " |")

    # Печатаем разделитель
    sep_parts = ["-" * col_widths[col] for col in df_str.columns]
    print("|-" + "-|-".join(sep_parts) + "-|")

    # Печатаем строки
    for _, row in df_str.iterrows():
        row_parts = []
        for col in df_str.columns:
            cell = str(row[col])
            # Обрезаем слишком длинные ячейки
            if len(cell) > col_widths[col] - 2:
                cell = cell[:col_widths[col] - 5] + "..."
            row_parts.append(cell.ljust(col_widths[col]))
        print("| " + " | ".join(row_parts) + " |")

    print("")


def display_winner_summary(best_row, selection_details):
    """
    Выводит сводку по победителю.
    """
    print("\n" + "=" * 70)
    print("🏆 ФИНАЛЬНАЯ МОДЕЛЬ (ВЫБРАНА АВТОМАТИЧЕСКИ)")
    print("=" * 70)

    # Основные метрики
    summary = pd.DataFrame({
        'Параметр': [
            'Модель',
            'COST_FN (штраф за пропуск)',
            'Medical Cost (VAL)',
            'Medical Cost (CV)',
            'Medical Cost CV Coef (стабильность)',
            'Medical Cost CV Std (стабильность)',
            'Recall (VAL)',
            'Precision (VAL)',
            'NNI (VAL)',
            'Selection Rate',
            'Threshold',
            'Прошло фильтры',
        ],
        'Значение': [
            best_row['Experiment'],
            f"{best_row.get('_grid_cost', 'N/A')}",
            f"{best_row['Medical_Cost']:.0f}",
            f"{best_row['Medical_Cost_CV']:.0f}",
            f"{best_row.get('Medical_Cost_CV_std', 0):.1f}",
            f"{best_row.get('Medical_Cost_CV_coef', 1.0):.3f}",
            f"{best_row['Val_Recall']:.3f}",
            f"{best_row['Val_Precision']:.3f}",
            f"{best_row['Val_NNI']:.1f}",
            f"{best_row['Selection_Rate']:.1f}%",
            f"{best_row['Threshold_CV']:.4f}",
            f"{selection_details['filters_passed']} / {selection_details['total_experiments']}",
        ]
    })
    print_pretty_table(summary, "")

    # Технические метрики
    tech_summary = pd.DataFrame({
        'Метрика': [
            'PR_AUC (CV mean)',
            'PR_AUC (CV std)',
            'BSS (After Calibration)',
            'ECE',
            'MCE',
            'Stability Gap',
            'CV Threshold Stability (std)',
        ],
        'Значение': [
            f"{best_row['PR_AUC_mean']:.4f}",
            f"{best_row['PR_AUC_std']:.4f}",
            f"{best_row['BSS_After']:.4f}",
            f"{best_row['ECE']:.4f}",
            f"{best_row['MCE']:.4f}",
            f"{best_row['Stability_Gap']:.4f}",
            f"{best_row['CV_Stability_std']:.4f}",
        ]
    })
    print_pretty_table(tech_summary, "📊 ТЕХНИЧЕСКИЕ МЕТРИКИ")

    # Калибровка
    calib_summary = pd.DataFrame({
        'Метрика': [
            'BSS Before Calibration',
            'BSS After Calibration',
            'Calibration Gain',
            'Reliability Slope'
        ],
        'Значение': [
            f"{best_row['BSS_Before']:.4f}",
            f"{best_row['BSS_After']:.4f}",
            f"{best_row['Calibration_Gain']:.4f}",
            f"{best_row['Reliability_Slope']:.3f}",
        ]
    })
    print_pretty_table(calib_summary, "🔧 КАЛИБРОВКА")

    # Важные признаки
    if 'Top_Feature_OR' in best_row and 'Bottom_Feature_OR' in best_row:
        feature_summary = pd.DataFrame({
            'Признак': ['Самый важный (|OR-1| max)', 'Самый слабый (|OR-1| min)'],
            'Odds Ratio': [
                format_feature_name_for_table(str(best_row['Top_Feature_OR']), max_len=50),
                format_feature_name_for_table(str(best_row['Bottom_Feature_OR']), max_len=50),
            ]
        })
        print_pretty_table(feature_summary, "🔬 ВАЖНОСТЬ ПРИЗНАКОВ (ODDS RATIO)")

    # Дополнительная информация
    print("\n" + "=" * 70)
    print("📋 ДОПОЛНИТЕЛЬНАЯ ИНФОРМАЦИЯ")
    print("=" * 70)
    print(f"   Val FN: {best_row['Val_FN']} (пропущенных пациентов)")
    print(f"   Val FP: {best_row['Val_FP']} (ложных тревог)")
    print(f"   Val TP: {best_row['Val_TP']} (пойманных пациентов)")
    print(f"   Val TN: {best_row['Val_TN'] if 'Val_TN' in best_row else 'N/A'}")
    print(f"   Recall 95% CI: [{best_row['Recall_Lower_CI']:.3f}, {best_row.get('Recall_Upper_CI', 1.0):.3f}]")
    print("=" * 70)


def print_experiments_summary(df_results, top_n=5):
    """
    Выводит сводку по всем экспериментам (топ-N по Medical Cost).

    Параметры:
    ----------
    df_results : pd.DataFrame
        DataFrame с результатами экспериментов
    top_n : int, default=5
        Количество лучших моделей для отображения
    """
    print("\n" + "=" * 70)
    print("📊 СВОДКА ЭКСПЕРИМЕНТОВ (топ-{} по Medical Cost)".format(top_n))
    print("=" * 70)

    # Сортируем по Medical Cost
    df_sorted = df_results.sort_values('Medical_Cost_CV').head(top_n)

    # Выбираем колонки для отображения
    display_cols = ['Experiment', 'Medical_Cost', 'Val_Recall', 'Val_Precision',
                    'Selection_Rate', 'Threshold_CV', 'Converged']

    available_cols = [col for col in display_cols if col in df_sorted.columns]
    df_display = df_sorted[available_cols].copy()

    # Форматируем
    df_display['Medical_Cost'] = df_display['Medical_Cost'].apply(lambda x: f"{x:.0f}")
    df_display['Val_Recall'] = df_display['Val_Recall'].apply(lambda x: f"{x:.3f}")
    df_display['Val_Precision'] = df_display['Val_Precision'].apply(lambda x: f"{x:.3f}")
    df_display['Selection_Rate'] = df_display['Selection_Rate'].apply(lambda x: f"{x:.1f}%")
    df_display['Threshold_CV'] = df_display['Threshold_CV'].apply(lambda x: f"{x:.4f}")

    print_pretty_table(df_display, "")


print("✅ Функции форматирования загружены")

✅ Функции форматирования загружены


# БЛОК 4. ЗАПУСК ЭКСПЕРИМЕНТОВ

In [21]:
# =============================================================================
# 4.1 ЗАПУСК ЭКСПЕРИМЕНТОВ
# =============================================================================

print("\n" + "=" * 70)
print("🚀 ЗАПУСК ЭКСПЕРИМЕНТОВ")
print("=" * 70)

# Проверка, что данные загружены
if 'X_train' not in globals() or X_train is None:
    raise NameError("❌ X_train не определён. Выполните блоки 1.8 и 1.9")

if 'X_val' not in globals() or X_val is None:
    raise NameError("❌ X_val не определён. Выполните блоки 1.8 и 1.9")

print(f"\n📊 Данные для экспериментов:")
print(f"   Train: {X_train.shape[0]} сэмплов, {X_train.shape[1]} признаков")
print(f"   Val:   {X_val.shape[0]} сэмплов")
print(f"   Положительных в Train: {y_train.sum()} ({y_train.mean()*100:.2f}%)")
print(f"   Положительных в Val:   {y_val.sum()} ({y_val.mean()*100:.2f}%)")
print(f"   Признаки: {list(X_train.columns[:5])}...")

print(f"\n🔬 Сетка экспериментов:")
print(f"   COST_FN (базовый): {COST_FN}, COST_FP: {COST_FP}")
print(f"   GRID_COST_VALUES (перебор FN): {GRID_COST_VALUES}")
print(f"   Penalty: {GRID_PENALTIES}")
print(f"   C: {GRID_C_VALUES}")
print(f"   Random state: {RANDOM_STATE}")
print(f"   CV folds: {CV_FOLDS}")
print(f"   Калибровка: {CALIBRATION_METHOD if CALIBRATION_ENABLED else 'OFF'}")

# Замер времени
start_time = time.time()

# Запуск всех экспериментов из сетки
df_results = run_all_experiments(
    X_train=X_train, y_train=y_train,
    X_val=X_val, y_val=y_val,
    feature_names=feature_names,
    random_state=RANDOM_STATE,
    cost_fn=COST_FN,
    cost_fp=COST_FP
)

elapsed_time = time.time() - start_time

# Проверка, что эксперименты завершились успешно
if len(df_results) == 0:
    raise RuntimeError(
        "❌ НЕ ОДИН ЭКСПЕРИМЕНТ НЕ ЗАВЕРШИЛСЯ УСПЕШНО!\n"
        "   Проверьте ошибки выше. Возможные причины:\n"
        "   • Проблемы с данными (NaN, Inf)\n"
        "   • Несовместимость параметров модели\n"
        "   • Недостаточно памяти"
    )

print(f"\n✅ Завершено экспериментов: {len(df_results)}")
print(f"⏱️ Общее время выполнения: {elapsed_time:.1f} секунд")
print(f"   Среднее время на эксперимент: {elapsed_time/len(df_results):.1f} сек")

# Сохраняем сырые результаты experiments_comparison.csv
experiments_csv_path = os.path.join(MODELLING_REPORTS_PATH, 'experiments_comparison.csv')

# Убираем объекты моделей из CSV (только метрики)
df_to_save = df_results.drop(
    columns=[col for col in ['probs_calib', 'y_val', 'base_model', 'calibrated_model']
             if col in df_results.columns],
    errors='ignore'
)
df_to_save.to_csv(experiments_csv_path, index=False)
print(f"\n💾 Результаты экспериментов сохранены: {experiments_csv_path}")

print("=" * 70)


🚀 ЗАПУСК ЭКСПЕРИМЕНТОВ

📊 Данные для экспериментов:
   Train: 3031 сэмплов, 10 признаков
   Val:   1011 сэмплов
   Положительных в Train: 149 (4.92%)
   Положительных в Val:   50 (4.95%)
   Признаки: ['age', 'avg_glucose_level', 'cardio_risk', 'stable_old_age', 'smoking_age_impact']...

🔬 Сетка экспериментов:
   COST_FN (базовый): 18, COST_FP: 1
   GRID_COST_VALUES (перебор FN): [16, 17, 18, 19, 20]
   Penalty: ['l2']
   C: [0.01, 0.1, 1]
   Random state: 99
   CV folds: 5
   Калибровка: sigmoid
🚀 ЗАПУСК ЭКСПЕРИМЕНТОВ (CV + Medical Cost оптимизация)
Всего экспериментов: 16
Базовый COST_FN: 18, COST_FP: 1
Сетка FN ∈ [16, 17, 18, 19, 20], penalty ∈ ['l2'], C ∈ [0.01, 0.1, 1]
Калибровка: ✅ ВКЛ
Random state: 99

⏳ Выполнение...

   [1/16] FN16_L2_C=0.01...   CV поиск порога... готово
 ✓
   [2/16] FN16_L2_C=0.1...   CV поиск порога... готово
 ✓
   [3/16] FN16_L2_C=1...   CV поиск порога... готово
 ✓
   [4/16] FN17_L2_C=0.01...   CV поиск порога... готово
 ✓
   [5/16] FN17_L2_C=0.1...   CV 

In [22]:
# =============================================================================
# 4.2 АВТОМАТИЧЕСКИЙ ВЫБОР ЛУЧШЕЙ МОДЕЛИ (С ЗАЩИТОЙ ОТ ПУСТОГО РЕЗУЛЬТАТА)
# =============================================================================

best_row, selection_details = auto_select_best_model(df_results)

# Флаг: есть ли победитель
HAS_WINNER = best_row is not None

if not HAS_WINNER:
    print("\n" + "=" * 70)
    print("⚠️ ВНИМАНИЕ: НИ ОДНА МОДЕЛЬ НЕ ПРОШЛА ФИЛЬТРЫ")
    print("=" * 70)

    # Показываем лучшие модели до отсева (уровень 2)
    print("\n📊 ТОП-5 МОДЕЛЕЙ ПО MEDICAL COST (после фильтров Уровня 1 и 2):")

    df_level2 = df_results.copy()
    df_level2 = df_level2[df_level2['BSS_After'] > FILTER_MIN_BSS]
    if FILTER_REQUIRE_CONVERGED:
        df_level2 = df_level2[df_level2['Converged'] == '✅']
    df_level2 = df_level2[df_level2['Stability_Gap'] <= FILTER_MAX_STABILITY_GAP]
    df_level2 = df_level2[df_level2['CV_Stability_std'] <= FILTER_MAX_CV_STABILITY_STD]
    df_level2 = df_level2[df_level2['MCE'] <= FILTER_MAX_MCE]
    df_level2 = df_level2[df_level2['ECE'] <= FILTER_MAX_ECE]
    df_level2 = df_level2[df_level2['Reliability_Slope'] >= FILTER_RELIABILITY_SLOPE_MIN]
    df_level2 = df_level2[df_level2['Reliability_Slope'] <= FILTER_RELIABILITY_SLOPE_MAX]
    df_level2 = df_level2[df_level2['PR_AUC_mean'] >= FILTER_MIN_PR_AUC]

    if len(df_level2) > 0:
        df_level2_sorted = df_level2.sort_values('Medical_Cost').head(5)

        display_df = df_level2_sorted[[
            'Experiment', 'Medical_Cost', 'Val_Recall', 'Val_Precision',
            'Selection_Rate', 'Val_NNI', 'Val_TP', 'Recall_Lower_CI'
        ]].copy()

        for col in ['Medical_Cost', 'Val_Recall', 'Val_Precision', 'Selection_Rate', 'Val_NNI', 'Recall_Lower_CI']:
            if col in display_df.columns:
                display_df[col] = display_df[col].apply(lambda x: f"{x:.3f}" if pd.notna(x) else "N/A")
        if 'Val_TP' in display_df.columns:
            display_df['Val_TP'] = display_df['Val_TP'].apply(lambda x: f"{int(x)}" if pd.notna(x) else "N/A")

        print_pretty_table(display_df, "")

        print("\n📋 ПРИЧИНЫ ОТСЕВА (фильтры Уровня 3, топ-1 модель уровня 2):")
        best_level2 = df_level2_sorted.iloc[0]

        checks = [
            ('Recall', best_level2['Val_Recall'], FILTER_MIN_RECALL_VAL, 'ge'),
            ('Recall Lower CI', best_level2['Recall_Lower_CI'], FILTER_MIN_RECALL_LOWER_CI, 'ge'),
            ('Precision', best_level2['Val_Precision'], FILTER_MIN_PRECISION_VAL, 'ge'),
            ('Selection Rate', best_level2['Selection_Rate'], FILTER_MAX_SELECTION_RATE_VAL * 100, 'le'),
            ('NNI', best_level2['Val_NNI'], FILTER_MAX_NNI_VAL, 'le'),
            ('TP', best_level2['Val_TP'], FILTER_MIN_TP_VAL, 'ge'),
        ]

        for name, value, threshold, op in checks:
            if op == 'ge':
                passed = value >= threshold
                symbol = "✅" if passed else "❌"
            else:
                passed = value <= threshold
                symbol = "✅" if passed else "❌"

            if not passed:
                if op == 'ge':
                    print(f"   {symbol} {name}: {value:.3f} < {threshold}")
                else:
                    print(f"   {symbol} {name}: {value:.3f} > {threshold}")
    else:
        print("   Нет моделей, прошедших даже фильтры Уровня 1-2")
        print("   Попробуйте ослабить FILTER_MIN_BSS, FILTER_MIN_PR_AUC или другие параметры")

    print("\n💡 РЕКОМЕНДАЦИИ:")
    print("   1. Ослабьте фильтры Уровня 3 в блоке 1.4")
    print("   2. Увеличьте STABILITY_MARGIN (сейчас {:.0f}%)".format(STABILITY_MARGIN * 100))
    print("   3. Проверьте качество признаков в EDA")
    print("   4. Попробуйте другие значения GRID_COST_VALUES")
    print("   5. Увеличьте CV_FOLDS для более стабильной оценки")

    print("\n" + "=" * 70)
    print("⏸️ НОУТБУК ПРОДОЛЖАЕТ РАБОТУ (модель не выбрана, но отчёт сохранён)")
    print("=" * 70)

    # Создаём пустые переменные для дальнейших ячеек
    best_row = None
    best_model_name = None
    best_threshold = None
    best_base_model = None
    best_calibrated_model = None
    final_threshold = 0.5
    best_y_val = y_val.values if hasattr(y_val, 'values') else np.array(y_val)
    best_probs_calib = None

# =============================================================================
# ЕСЛИ МОДЕЛЬ ЕСТЬ — ПРОДОЛЖАЕМ КАК ОБЫЧНО
# =============================================================================

if HAS_WINNER:
    # Если модель найдена — продолжаем
    best_model_name = best_row['Experiment']
    best_threshold = best_row['Threshold_CV']
    best_base_model = best_row['base_model']
    best_calibrated_model = best_row.get('calibrated_model', None)

    # =========================================================================
    # КЛИНИЧЕСКИЙ ЗАПАС (Threshold Margin)
    # =========================================================================
    THRESHOLD_MARGIN = 0.95
    best_threshold_safe = best_threshold * THRESHOLD_MARGIN

    print(f"\n🔒 Клинический запас порога:")
    print(f"   Исходный порог: {best_threshold:.4f}")
    print(f"   С запасом ({THRESHOLD_MARGIN:.0%}): {best_threshold_safe:.4f}")
    print(f"   → Более консервативный (выше Recall, больше FP)")

    final_threshold = best_threshold_safe

    # Восстанавливаем probs_calib
    if best_calibrated_model is not None:
        best_probs_calib = best_calibrated_model.predict_proba(X_val)[:, 1]
    else:
        best_probs_calib = best_base_model.predict_proba(X_val)[:, 1]
    best_y_val = y_val.values if hasattr(y_val, 'values') else np.array(y_val)

    # Доверительный интервал Recall
    print("\n" + "=" * 70)
    print("📊 ДОВЕРИТЕЛЬНЫЙ ИНТЕРВАЛ RECALL (95%):")
    print("=" * 70)

    recall_lower = best_row.get('Recall_Lower_CI', 0.0)
    recall_upper = best_row.get('Recall_Upper_CI', 1.0)

    print(f"   Recall на валидации:    {best_row['Val_Recall']:.3f}")
    print(f"   95% доверительный интервал: [{recall_lower:.3f} — {recall_upper:.3f}]")
    print(f"   Худший сценарий: {recall_lower:.1%}")
    print(f"   Лучший сценарий: {recall_upper:.1%}")

    # Выводим сводку по победителю
    display_winner_summary(best_row, selection_details)

    print(f"\n✅ Порог фиксирован: {best_threshold:.4f}")
    print(f"   Этот порог будет использован для:")
    print(f"   • Визуализации (блок 4.3)")
    print(f"   • Финальной модели (блок 4.4)")

    # Сохраняем метаданные о победителе
    WINNER_METADATA = {
        'has_winner': True,
        'model_name': best_model_name,
        'threshold': best_threshold,
        'final_threshold': final_threshold,
        'recall': float(best_row['Val_Recall']),
        'recall_lower_ci': float(recall_lower),
        'recall_upper_ci': float(recall_upper),
        'medical_cost_cv': float(best_row['Medical_Cost_CV']),
        'selection_rate': float(best_row['Selection_Rate'])
    }
else:
    # Нет победителя — сохраняем информацию об отчёте
    WINNER_METADATA = {
        'has_winner': False,
        'reason': 'Ни одна модель не прошла фильтры',
        'top_candidates': df_level2_sorted[['Experiment', 'Medical_Cost', 'Val_Recall']].to_dict('records') if 'df_level2_sorted' in dir() else []
    }
    print("\n⚠️ Модель не выбрана. Блоки визуализации и сохранения будут пропущены.")

print("\n✅ Выбор модели завершён")


🤖 АВТОМАТИЧЕСКИЙ ВЫБОР МОДЕЛИ

📊 СТАТИСТИКА ФИЛЬТРАЦИИ:
   Всего экспериментов: 16
   После Уровня 1 (технический): 15
   После Уровня 2 (расширенный): 7
   После Уровня 3 (клинический): 5

📋 ФИЛЬТРЫ, ОТСЕКШИЕ БОЛЬШЕ ВСЕГО:
   • Medical_Cost_CV_coef: отсечено 6 моделей
   • CV_Stability_std: отсечено 2 моделей
   • BSS_After: отсечено 1 моделей
   • Val_Recall: отсечено 1 моделей
   • Recall_Lower_CI: отсечено 1 моделей

🏆 ПОБЕДИТЕЛЬ:
   Модель: FN17_L2_C=1
   COST_FN: 17 (штраф за FN)
   Medical Cost (CV): 229
   Medical Cost (VAL): 404
   Recall (VAL): 0.860
   Precision (VAL): 0.131
   Selection Rate: 32.4%
   Threshold: 0.0488

📋 Топ-3 моделей (для справки):
   1. FN17_L2_C=1: Cost_CV=229, Recall=0.860
   2. FN18_L2_C=1: Cost_CV=232, Recall=0.860
   3. FN19_L2_C=1: Cost_CV=239, Recall=0.860

🔒 Клинический запас порога:
   Исходный порог: 0.0488
   С запасом (95%): 0.0463
   → Более консервативный (выше Recall, больше FP)

📊 ДОВЕРИТЕЛЬНЫЙ ИНТЕРВАЛ RECALL (95%):
   Recall на валидац

# Лучшая модель

In [23]:
# =============================================================================
# 4.3 СРАВНЕНИЕ ПОБЕДИТЕЛЯ С DUMMY БАЗОВОЙ ЛИНИЕЙ
# =============================================================================

if not HAS_WINNER:
    print("\n⚠️ Нет победителя, пропускаем сравнение с Dummy")
else:
    print("\n" + "=" * 70)
    print("📊 СРАВНЕНИЕ ПОБЕДИТЕЛЯ С DUMMY БАЗОВОЙ ЛИНИЕЙ")
    print("=" * 70)

    # Находим Dummy модель в результатах
    dummy_rows = df_results[df_results['Experiment'].str.contains('Dummy', case=False, na=False)]

    if len(dummy_rows) > 0:
        dummy_row = dummy_rows.iloc[0]

        print("\n📋 СРАВНЕНИЕ МЕТРИК (VAL):")
        print("-" * 70)

        comparison_data = []

        metrics_to_compare = [
            ('Medical Cost (CV)', 'Medical_Cost_CV', '{:.0f}', '↓ (меньше лучше)'),
            ('Medical Cost (VAL)', 'Medical_Cost', '{:.0f}', '↓ (меньше лучше)'),
            ('Recall', 'Val_Recall', '{:.3f}', '↑ (больше лучше)'),
            ('Recall (нижн. граница)', 'Recall_Lower_CI', '{:.3f}', '↑ (больше лучше)'),
            ('Precision', 'Val_Precision', '{:.3f}', '↑ (больше лучше)'),
            ('NNI (Number Needed to Inspect)', 'Val_NNI', '{:.1f}', '↓ (меньше лучше)'),
            ('Selection Rate', 'Selection_Rate', '{:.1f}%', '↓ (меньше лучше)'),
            ('False Negatives (FN)', 'Val_FN', '{:.0f}', '↓ (меньше лучше)'),
            ('False Positives (FP)', 'Val_FP', '{:.0f}', '↓ (меньше лучше)'),
            ('True Positives (TP)', 'Val_TP', '{:.0f}', '↑ (больше лучше)'),
        ]

        for metric_name, col_name, fmt, direction in metrics_to_compare:
            if col_name in best_row and col_name in dummy_row:
                best_val = best_row[col_name]
                dummy_val = dummy_row[col_name]

                if direction == '↑ (больше лучше)':
                    improvement = (best_val - dummy_val) / dummy_val * 100 if dummy_val != 0 else float('inf')
                    improvement_sign = '+' if improvement > 0 else ''
                else:
                    improvement = (dummy_val - best_val) / dummy_val * 100 if dummy_val != 0 else float('inf')
                    improvement_sign = '+' if improvement > 0 else ''

                comparison_data.append({
                    'Метрика': metric_name,
                    'Победитель': fmt.format(best_val),
                    'Dummy': fmt.format(dummy_val),
                    'Δ': f"{improvement_sign}{improvement:.1f}%"
                })

        df_comparison = pd.DataFrame(comparison_data)
        print_pretty_table(df_comparison, "🏆 ПОБЕДИТЕЛЬ vs DUMMY (baseline)")

        # Дополнительные метрики калибровки
        print("\n📋 СРАВНЕНИЕ КАЛИБРОВКИ:")
        print("-" * 70)

        calib_data = []

        calib_metrics = [
            ('Brier Skill Score (BSS)', 'BSS_After', '{:.4f}', '↑ (больше лучше)'),
            ('ECE (Expected Calibration Error)', 'ECE', '{:.4f}', '↓ (меньше лучше)'),
            ('MCE (Maximum Calibration Error)', 'MCE', '{:.4f}', '↓ (меньше лучше)'),
            ('Reliability Slope', 'Reliability_Slope', '{:.3f}', '→ к 1 (идеал)'),
        ]

        for metric_name, col_name, fmt, direction in calib_metrics:
            if col_name in best_row and col_name in dummy_row:
                best_val = best_row[col_name]
                dummy_val = dummy_row[col_name]

                if direction == '→ к 1 (идеал)':
                    best_dist = abs(best_val - 1)
                    dummy_dist = abs(dummy_val - 1)
                    improvement = (dummy_dist - best_dist) / dummy_dist * 100 if dummy_dist != 0 else float('inf')
                    improvement_sign = '+' if improvement > 0 else ''
                elif direction == '↑ (больше лучше)':
                    improvement = (best_val - dummy_val) / dummy_val * 100 if dummy_val != 0 else float('inf')
                    improvement_sign = '+' if improvement > 0 else ''
                else:
                    improvement = (dummy_val - best_val) / dummy_val * 100 if dummy_val != 0 else float('inf')
                    improvement_sign = '+' if improvement > 0 else ''

                calib_data.append({
                    'Метрика': metric_name,
                    'Победитель': fmt.format(best_val),
                    'Dummy': fmt.format(dummy_val),
                    'Δ': f"{improvement_sign}{improvement:.1f}%"
                })

        if calib_data:
            df_calib = pd.DataFrame(calib_data)
            print_pretty_table(df_calib, "")

        # Итоговая оценка
        print("\n" + "-" * 70)
        print("📊 ИТОГОВАЯ ОЦЕНКА:")
        print("-" * 70)

        improvements = []
        for _, row in df_comparison.iterrows():
            delta_str = row['Δ']
            if delta_str != 'inf%' and delta_str != 'nan%':
                try:
                    delta_val = float(delta_str.replace('%', '').replace('+', ''))
                    if delta_val > 0:
                        improvements.append(True)
                    elif delta_val < 0:
                        improvements.append(False)
                except:
                    pass

        if improvements:
            better_count = sum(improvements)
            total_count = len(improvements)

            if better_count > total_count / 2:
                print(f"   ✅ Модель лучше Dummy по {better_count}/{total_count} метрикам")
            elif better_count == total_count:
                print(f"   ✅✅ Модель превосходит Dummy по ВСЕМ метрикам!")
            else:
                print(f"   ⚠️ Модель хуже Dummy по {total_count - better_count}/{total_count} метрикам")

            # Сравнение Medical Cost CV
            best_cost_cv = best_row['Medical_Cost_CV']
            dummy_cost_cv = dummy_row['Medical_Cost_CV']
            cost_reduction = (dummy_cost_cv - best_cost_cv) / dummy_cost_cv * 100 if dummy_cost_cv > 0 else 0

            if cost_reduction > 0:
                print(f"   💰 Medical Cost (CV) снижен на {cost_reduction:.1f}% относительно Dummy")
            else:
                print(f"   ⚠️ Medical Cost (CV) НЕ снижен относительно Dummy")

        # Визуализация сравнения
        print("\n📈 Построение графика сравнения...")

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        # График 1: Сравнение метрик (нормализованных)
        metrics_for_plot = ['Medical_Cost_CV', 'Val_Recall', 'Recall_Lower_CI', 'Val_Precision', 'Val_NNI']
        metric_labels = ['Medical Cost (CV)', 'Recall', 'Recall (нижн.)', 'Precision', 'NNI']

        best_norm = []
        dummy_norm = []

        for metric in metrics_for_plot:
            if metric in best_row and metric in dummy_row:
                best_val = best_row[metric]
                dummy_val = dummy_row[metric]
                if dummy_val != 0:
                    best_norm.append(best_val / dummy_val)
                    dummy_norm.append(1.0)
                else:
                    best_norm.append(0)
                    dummy_norm.append(0)

        x = np.arange(len(metric_labels[:len(best_norm)]))
        width = 0.35

        axes[0].bar(x - width/2, dummy_norm, width, label='Dummy (baseline)', color='gray', alpha=0.7)
        axes[0].bar(x + width/2, best_norm, width, label='Победитель', color='crimson', alpha=0.7)
        axes[0].set_xticks(x)
        axes[0].set_xticklabels(metric_labels[:len(best_norm)], rotation=45, ha='right')
        axes[0].set_ylabel('Относительное значение (Dummy = 1)')
        axes[0].set_title('Сравнение метрик (нормализовано)')
        axes[0].axhline(y=1, color='gray', linestyle='--', alpha=0.5)
        axes[0].legend()

        # График 2: Confusion Matrix для Dummy
        if 'base_model' in dummy_row:
            dummy_probs = dummy_row['base_model'].predict_proba(X_val)[:, 1]
            dummy_threshold = dummy_row['Threshold_CV']

            plot_confusion_matrix_at_threshold(
                y_true=best_y_val,
                y_probs=dummy_probs,
                threshold=dummy_threshold,
                title=f"Dummy - Validation (threshold={dummy_threshold:.3f})",
                ax=axes[1]
            )
            axes[1].set_title(f'Dummy Model\nThreshold={dummy_threshold:.3f}')
        else:
            axes[1].text(0.5, 0.5, 'Dummy model not available',
                         ha='center', va='center', transform=axes[1].transAxes)
            axes[1].set_title('Dummy - No Data')

        plt.tight_layout()
        plt.savefig(os.path.join(MODELLING_PLOTS_PATH, 'comparison_with_dummy.png'), dpi=150, bbox_inches='tight')
        plt.close(fig)

        print(f"   ✅ График сохранён: {MODELLING_PLOTS_PATH}/comparison_with_dummy.png")

    else:
        print("⚠️ Dummy модель не найдена в результатах экспериментов")
        print("   Убедитесь, что Dummy включён в generate_grid_experiments()")

    print("=" * 70)


📊 СРАВНЕНИЕ ПОБЕДИТЕЛЯ С DUMMY БАЗОВОЙ ЛИНИЕЙ

📋 СРАВНЕНИЕ МЕТРИК (VAL):
----------------------------------------------------------------------

🏆 ПОБЕДИТЕЛЬ vs DUMMY (baseline)
|             Метрика              |  Победитель  |  Dummy  |     Δ      |
|----------------------------------|--------------|---------|------------|
| Medical Cost (CV)                | 229          | 556     | +58.7%     |
| Medical Cost (VAL)               | 404          | 923     | +56.2%     |
| Recall                           | 0.860        | 0.040   | +2050.0%   |
| Recall (нижн. граница)           | 0.737        | 0.040   | +1743.6%   |
| Precision                        | 0.131        | 0.033   | +299.8%    |
| NNI (Number Needed to Inspect)   | 7.6          | 30.5    | +75.0%     |
| Selection Rate                   | 32.4%        | 6.0%    | -437.7%    |
| False Negatives (FN)             | 7            | 48      | +85.4%     |
| False Positives (FP)             | 285          | 59      | -383.1%  

In [24]:
# =============================================================================
# 4.4 ВИЗУАЛИЗАЦИЯ ПОБЕДИТЕЛЯ (ВАЛИДАЦИЯ)
# =============================================================================

if not HAS_WINNER:
    print("\n⚠️ Нет победителя, пропускаем визуализацию")
else:
    print("\n" + "=" * 70)
    print("📊 ВИЗУАЛИЗАЦИЯ ПОБЕДИТЕЛЯ (ВАЛИДАЦИЯ)")
    print("=" * 70)

    # Убеждаемся, что папка для графиков существует
    os.makedirs(MODELLING_PLOTS_PATH, exist_ok=True)

    # -----------------------------------------------------------------------------
    # 1. Confusion Matrix
    # -----------------------------------------------------------------------------
    print("📈 Построение Confusion Matrix...")
    fig, ax = plot_confusion_matrix_at_threshold(
        y_true=best_y_val,
        y_probs=best_probs_calib,
        threshold=best_threshold,
        title=f"{best_model_name} - Validation"
    )
    plt.savefig(os.path.join(MODELLING_PLOTS_PATH, 'confusion_matrix.png'), dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"   ✅ Сохранено: {MODELLING_PLOTS_PATH}/confusion_matrix.png")

    # -----------------------------------------------------------------------------
    # 2. Medical Cost Curve
    # -----------------------------------------------------------------------------
    print("📈 Построение Medical Cost Curve...")
    fig, best_th, best_cost, best_fn, best_fp = plot_medical_cost_curve(
        y_true=best_y_val,
        y_probs=best_probs_calib,
        cost_fn=COST_FN,
        cost_fp=COST_FP,
        model_name=best_model_name,
        save_path=os.path.join(MODELLING_PLOTS_PATH, 'medical_cost_curve.png')
    )
    plt.close(fig)
    print(f"   ✅ Сохранено: {MODELLING_PLOTS_PATH}/medical_cost_curve.png")
    print(f"   Оптимальный порог: {best_th:.4f}, Cost: {best_cost:.0f}, FN={best_fn}, FP={best_fp}")

    # -----------------------------------------------------------------------------
    # 3. Recall vs Threshold
    # -----------------------------------------------------------------------------
    print("📈 Построение Recall vs Threshold...")
    fig = plot_recall_vs_threshold(
        y_true=best_y_val,
        y_probs=best_probs_calib,
        model_name=best_model_name,
        target_recall=FILTER_MIN_RECALL_VAL,
        save_path=os.path.join(MODELLING_PLOTS_PATH, 'recall_vs_threshold.png')
    )
    plt.close(fig)
    print(f"   ✅ Сохранено: {MODELLING_PLOTS_PATH}/recall_vs_threshold.png")

    # -----------------------------------------------------------------------------
    # 4. Calibration Curve
    # -----------------------------------------------------------------------------
    print("📈 Построение Calibration Curve...")
    fig = plot_calibration_curve(
        y_true=best_y_val,
        y_probs=best_probs_calib,
        model_name=best_model_name,
        n_bins=10,
        save_path=os.path.join(MODELLING_PLOTS_PATH, 'calibration_curve.png')
    )
    plt.close(fig)
    print(f"   ✅ Сохранено: {MODELLING_PLOTS_PATH}/calibration_curve.png")

    # -----------------------------------------------------------------------------
    # 5. Risk Stratification
    # -----------------------------------------------------------------------------
    print("📈 Построение Risk Stratification...")
    strat_table = get_risk_stratification_table(
        y_true=best_y_val,
        y_probs=best_probs_calib,
        model_name=best_model_name
    )

    # Выводим таблицу в консоль
    print_risk_stratification_table(strat_table, best_model_name)

    # Сохраняем таблицу в CSV
    strat_table_path = os.path.join(MODELLING_REPORTS_PATH, 'risk_stratification.csv')
    strat_table.to_csv(strat_table_path, index=False)
    print(f"   💾 Таблица стратификации сохранена: {strat_table_path}")

    # Сохраняем график
    fig = plot_risk_stratification(strat_table, model_name=best_model_name,
                                   save_path=os.path.join(MODELLING_PLOTS_PATH, 'risk_stratification.png'))
    plt.close(fig)
    print(f"   ✅ График сохранён: {MODELLING_PLOTS_PATH}/risk_stratification.png")

    # -----------------------------------------------------------------------------
    # 6. Probability Histogram
    # -----------------------------------------------------------------------------
    print("📈 Построение Probability Histogram...")
    fig = plot_probability_histogram(
        y_true=best_y_val,
        y_probs=best_probs_calib,
        model_name=best_model_name,
        save_path=os.path.join(MODELLING_PLOTS_PATH, 'probability_histogram.png')
    )
    plt.close(fig)
    print(f"   ✅ Сохранено: {MODELLING_PLOTS_PATH}/probability_histogram.png")

    print("\n✅ Визуализация завершена")
    print(f"   Все графики сохранены в: {MODELLING_PLOTS_PATH}")


📊 ВИЗУАЛИЗАЦИЯ ПОБЕДИТЕЛЯ (ВАЛИДАЦИЯ)
📈 Построение Confusion Matrix...
   ✅ Сохранено: /content/drive/MyDrive/ml_learning/datasets/stroke/cycle_5/modelling/plots/confusion_matrix.png
📈 Построение Medical Cost Curve...
   ✅ Сохранено: /content/drive/MyDrive/ml_learning/datasets/stroke/cycle_5/modelling/plots/medical_cost_curve.png
   Оптимальный порог: 0.0558, Cost: 389, FN=8, FP=245
📈 Построение Recall vs Threshold...
   ✅ Сохранено: /content/drive/MyDrive/ml_learning/datasets/stroke/cycle_5/modelling/plots/recall_vs_threshold.png
📈 Построение Calibration Curve...
   ✅ Сохранено: /content/drive/MyDrive/ml_learning/datasets/stroke/cycle_5/modelling/plots/calibration_curve.png
📈 Построение Risk Stratification...

📊 СТРАТИФИКАЦИЯ РИСКА - FN17_L2_C=1
        Risk Group  Total_Patients  Actual_Target Actual_Risk_Rate  %_of_All_Patients  %_of_All_Target  Lift
Очень низкий (<2%)             470              2            0.43%               46.5              4.0   0.1
     Низкий (2-5%)      

In [25]:
# =============================================================================
# 4.6 ПРОВЕРКА СТАБИЛЬНОСТИ ФИНАЛИСТА НА РАЗНЫХ RANDOM_STATE
# =============================================================================

if not HAS_WINNER:
    print("\n⚠️ Нет победителя, пропускаем проверку стабильности")
else:
    print("\n" + "=" * 70)
    print("🔬 ПРОВЕРКА СТАБИЛЬНОСТИ ФИНАЛИСТА")
    print("=" * 70)

    # Берём финалиста
    final_model_name = best_row['Experiment']
    final_config = {
        'penalty': best_row['Experiment'].split('_')[1].lower(),
        'C': float(best_row['Experiment'].split('_C=')[1]),
        'model_type': 'logreg'
    }

    # Извлекаем FN
    import re
    fn_match = re.search(r'FN(\d+)', final_model_name)
    cost_fn = int(fn_match.group(1)) if fn_match else COST_FN

    print(f"\n📋 Проверяемая модель: {final_model_name}")
    print(f"   FN = {cost_fn}, FP = {COST_FP}")
    print(f"   Фиксированный порог: {best_threshold:.4f}")

    # Список random_state для проверки
    VALIDATION_RANDOM_STATES = [42, 123, 456, 789, 999]
    stability_results = []

    print("\n🔄 Запуск проверки на разных random_state...")
    print("-" * 50)

    from scipy.stats import beta

    def get_recall_ci(tp, fn, confidence=0.95):
        """Доверительный интервал для Recall через бета-распределение"""
        n = tp + fn
        if n == 0:
            return 0.0, 0.0
        alpha = tp + 1
        beta_param = fn + 1
        lower = beta.ppf((1 - confidence) / 2, alpha, beta_param)
        upper = beta.ppf(1 - (1 - confidence) / 2, alpha, beta_param)
        return lower, upper

    for rs in VALIDATION_RANDOM_STATES:
        print(f"\n   random_state = {rs}...", end=' ', flush=True)

        # Создаём новые сплиты с этим random_state
        from sklearn.model_selection import train_test_split
        X_full = pd.concat([X_train, X_val])
        y_full = pd.concat([y_train, y_val])

        X_train_new, X_val_new, y_train_new, y_val_new = train_test_split(
            X_full, y_full,
            test_size=len(X_val) / len(X_full),
            stratify=y_full,
            random_state=rs
        )

        # Обучаем финалиста
        model = build_model(final_config)
        model.fit(X_train_new, y_train_new)

        # Калибровка
        if CALIBRATION_ENABLED:
            calibrated = CalibratedClassifierCV(model, cv=CALIBRATION_CV, method=CALIBRATION_METHOD)
            calibrated.fit(X_train_new, y_train_new)
            probs_val = calibrated.predict_proba(X_val_new)[:, 1]
        else:
            probs_val = model.predict_proba(X_val_new)[:, 1]

        # Метрики с фиксированным порогом
        metrics = get_metrics_at_threshold(y_val_new, probs_val, best_threshold, cost_fn, COST_FP)

        # Доверительный интервал для Recall
        recall_lower, recall_upper = get_recall_ci(metrics['tp'], metrics['fn'])

        # Дополнительные метрики
        from sklearn.metrics import average_precision_score, brier_score_loss
        pr_auc = average_precision_score(y_val_new, probs_val)
        bss = brier_skill_score(y_val_new, probs_val)
        ece, mce = calculate_ece_mce(y_val_new, probs_val, n_bins=10)

        stability_results.append({
            'random_state': rs,
            'recall': metrics['recall'],
            'recall_lower': recall_lower,
            'recall_upper': recall_upper,
            'precision': metrics['precision'],
            'medical_cost': metrics['medical_cost'],
            'fn_count': metrics['fn'],
            'fp_count': metrics['fp'],
            'tp_count': metrics['tp'],
            'tn_count': metrics['tn'],
            'selection_rate': metrics['selection_rate'] * 100,
            'nni': metrics['nni'],
            'pr_auc': pr_auc,
            'bss': bss,
            'ece': ece,
            'mce': mce,
            'n_val': len(X_val_new),
            'n_positives': y_val_new.sum()
        })

        print(f"Recall={metrics['recall']:.3f} (CI: {recall_lower:.3f}-{recall_upper:.3f}), Cost={metrics['medical_cost']:.0f}, FN={metrics['fn']}, FP={metrics['fp']}")

    # =========================================================================
    # АНАЛИЗ РЕЗУЛЬТАТОВ
    # =========================================================================
    df_stability = pd.DataFrame(stability_results)

    print("\n" + "=" * 70)
    print("📊 РЕЗУЛЬТАТЫ СТАБИЛЬНОСТИ ПО ВСЕМ МЕТРИКАМ")
    print("=" * 70)

    # Детальная таблица
    print("\n📋 ДЕТАЛЬНЫЕ РЕЗУЛЬТАТЫ ПО КАЖДОМУ RANDOM_STATE:")
    print("-" * 100)
    print(f"   rs    | Recall | Rec_lower | Rec_upper | Cost | FN | FP | TP | SelRate | NNI")
    print(f"   ------|--------|-----------|-----------|------|----|----|----|---------|-----")

    for _, row in df_stability.iterrows():
        fn_val = int(row['fn_count']) if pd.notna(row['fn_count']) else 0
        fp_val = int(row['fp_count']) if pd.notna(row['fp_count']) else 0
        tp_val = int(row['tp_count']) if pd.notna(row['tp_count']) else 0

        print(f"   {int(row['random_state']):4d} | {row['recall']:.3f}  | {row['recall_lower']:.3f}    | {row['recall_upper']:.3f}   | {row['medical_cost']:.0f} | {fn_val:2d} | {fp_val:3d} | {tp_val:2d} | {row['selection_rate']:5.1f}% | {row['nni']:4.1f}")

    # =========================================================================
    # УСРЕДНЁННЫЕ ХАРАКТЕРИСТИКИ
    # =========================================================================
    print("\n" + "=" * 70)
    print("📊 УСРЕДНЁННЫЕ ХАРАКТЕРИСТИКИ (по 5 random_state)")
    print("=" * 70)

    avg_recall = df_stability['recall'].mean()
    std_recall = df_stability['recall'].std()
    median_recall = df_stability['recall'].median()
    min_recall = df_stability['recall'].min()
    max_recall = df_stability['recall'].max()

    avg_recall_lower = df_stability['recall_lower'].mean()
    min_recall_lower = df_stability['recall_lower'].min()

    avg_cost = df_stability['medical_cost'].mean()
    std_cost = df_stability['medical_cost'].std()
    median_cost = df_stability['medical_cost'].median()

    avg_fn = df_stability['fn_count'].mean()
    median_fn = df_stability['fn_count'].median()
    min_fn = df_stability['fn_count'].min()
    max_fn = df_stability['fn_count'].max()

    avg_fp = df_stability['fp_count'].mean()
    median_fp = df_stability['fp_count'].median()

    avg_selection = df_stability['selection_rate'].mean()
    avg_nni = df_stability['nni'].mean()
    avg_pr_auc = df_stability['pr_auc'].mean()
    avg_bss = df_stability['bss'].mean()
    avg_ece = df_stability['ece'].mean()

    print(f"\n🎯 ЦЕЛЕВЫЕ МЕТРИКИ:")
    print(f"   • Recall:              {avg_recall:.3f} ± {std_recall:.3f}  (медиана: {median_recall:.3f})")
    print(f"   • Recall (нижн. граница): {avg_recall_lower:.3f} ± {df_stability['recall_lower'].std():.3f}")
    print(f"   • Recall (min/max):     {min_recall:.3f} — {max_recall:.3f}")
    print(f"   • Худший сценарий (средняя нижняя): {avg_recall_lower:.1%}")
    print(f"   • Medical Cost:        {avg_cost:.0f} ± {std_cost:.0f} (медиана: {median_cost:.0f})")

    print(f"\n📊 КЛИНИЧЕСКИЕ МЕТРИКИ:")
    print(f"   • Precision:           {df_stability['precision'].mean():.3f} ± {df_stability['precision'].std():.3f}")
    print(f"   • Selection Rate:      {avg_selection:.1f}% ± {df_stability['selection_rate'].std():.1f}%")
    print(f"   • NNI:                 {avg_nni:.1f} ± {df_stability['nni'].std():.1f}")

    print(f"\n🔬 ОШИБКИ:")
    print(f"   • FN (пропуски):       {avg_fn:.1f} ± {df_stability['fn_count'].std():.1f}   (min: {min_fn:.0f}, max: {max_fn:.0f})")
    print(f"   • FP (ложные):         {avg_fp:.1f} ± {df_stability['fp_count'].std():.1f}")
    print(f"   • TP (поймано):        {df_stability['tp_count'].mean():.1f} ± {df_stability['tp_count'].std():.1f}")

    print(f"\n📐 ТЕХНИЧЕСКИЕ МЕТРИКИ:")
    print(f"   • PR-AUC:              {avg_pr_auc:.3f} ± {df_stability['pr_auc'].std():.3f}")
    print(f"   • BSS:                 {avg_bss:.3f} ± {df_stability['bss'].std():.3f}")
    print(f"   • ECE:                 {avg_ece:.4f} ± {df_stability['ece'].std():.4f}")

    # =========================================================================
    # КРИТЕРИИ СТАБИЛЬНОСТИ
    # =========================================================================
    print("\n" + "=" * 70)
    print("🎯 ВЕРДИКТ ПО СТАБИЛЬНОСТИ")
    print("=" * 70)

    STABILITY_RECALL_MIN = 0.75
    STABILITY_RECALL_STD_MAX = 0.05
    STABILITY_COST_STD_MAX = 50
    STABILITY_RECALL_LOWER_MIN = 0.70

    stability_issues = []

    if min_recall < STABILITY_RECALL_MIN:
        stability_issues.append(f"❌ Recall падает до {min_recall:.1%} (< {STABILITY_RECALL_MIN:.0%})")
    else:
        print(f"✅ Recall не падает ниже {STABILITY_RECALL_MIN:.0%}")

    if std_recall > STABILITY_RECALL_STD_MAX:
        stability_issues.append(f"❌ Разброс Recall слишком высок: std={std_recall:.3f} (> {STABILITY_RECALL_STD_MAX})")
    else:
        print(f"✅ Разброс Recall в норме: std={std_recall:.3f}")

    if df_stability['medical_cost'].std() > STABILITY_COST_STD_MAX:
        stability_issues.append(f"❌ Разброс Cost высок: std={df_stability['medical_cost'].std():.0f} (> {STABILITY_COST_STD_MAX})")
    else:
        print(f"✅ Разброс Cost в норме: std={df_stability['medical_cost'].std():.0f}")

    if avg_recall_lower < STABILITY_RECALL_LOWER_MIN:
        stability_issues.append(f"❌ Средняя нижняя граница Recall: {avg_recall_lower:.1%} (< {STABILITY_RECALL_LOWER_MIN:.0%})")
    else:
        print(f"✅ Средняя нижняя граница Recall: {avg_recall_lower:.1%} (≥ {STABILITY_RECALL_LOWER_MIN:.0%})")

    print("\n" + "-" * 70)

    if len(stability_issues) == 0:
        print("\n✅✅✅ ФИНАЛИСТ СТАБИЛЕН!")
        print(f"\n   Ожидаемые характеристики на новых данных:")
        print(f"   • Recall: {avg_recall:.1%} ± {std_recall:.1%}")
        print(f"   • Recall (нижняя граница): {avg_recall_lower:.1%}")
        print(f"   • FN: {avg_fn:.0f} ± {df_stability['fn_count'].std():.0f}")
        print(f"   • Medical Cost: {avg_cost:.0f} ± {std_cost:.0f}")
        print(f"   • Selection Rate: {avg_selection:.1f}%")
        print("\n   ✅ Можно доверять модели. Переходите к тесту.")
    else:
        print("\n⚠️⚠️⚠️ ФИНАЛИСТ НЕСТАБИЛЕН!")
        print("\n   Проблемы:")
        for issue in stability_issues:
            print(f"   {issue}")
        print("\n   📌 РЕКОМЕНДАЦИИ:")
        print("      1. Собрать больше данных (нужно 5000-10000 пациентов)")
        print("      2. Упростить модель (сильнее регуляризация, меньше признаков)")
        print("      3. Увеличить долю валидации до 40%")
        print("      4. Попробовать другого финалиста из топа")
        print("\n   ⏸️ Тест пока не рекомендуется — модель может развалиться.")

    # =========================================================================
    # ВИЗУАЛИЗАЦИЯ
    # =========================================================================
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))

    # График 1: Recall по random_state с доверительными интервалами
    axes[0,0].errorbar(df_stability['random_state'], df_stability['recall'],
                       yerr=[df_stability['recall'] - df_stability['recall_lower'],
                             df_stability['recall_upper'] - df_stability['recall']],
                       fmt='bo-', capsize=5, linewidth=2, markersize=8)
    axes[0,0].axhline(y=median_recall, color='r', linestyle='--', label=f'Медиана: {median_recall:.3f}')
    axes[0,0].axhline(y=STABILITY_RECALL_MIN, color='orange', linestyle=':', label=f'Min допустимый: {STABILITY_RECALL_MIN}')
    axes[0,0].set_xlabel('Random State')
    axes[0,0].set_ylabel('Recall')
    axes[0,0].set_title('Стабильность Recall (95% CI)')
    axes[0,0].legend()
    axes[0,0].grid(alpha=0.3)

    # График 2: Medical Cost по random_state
    axes[0,1].plot(df_stability['random_state'], df_stability['medical_cost'], 'ro-', linewidth=2, markersize=8)
    axes[0,1].axhline(y=median_cost, color='b', linestyle='--', label=f'Медиана: {median_cost:.0f}')
    axes[0,1].set_xlabel('Random State')
    axes[0,1].set_ylabel('Medical Cost')
    axes[0,1].set_title('Стабильность Medical Cost')
    axes[0,1].legend()
    axes[0,1].grid(alpha=0.3)

    # График 3: FN и FP по random_state
    axes[1,0].plot(df_stability['random_state'], df_stability['fn_count'], 'r-', linewidth=2, marker='o', label='FN (пропуски)')
    axes[1,0].plot(df_stability['random_state'], df_stability['fp_count'], 'orange', linewidth=2, marker='s', label='FP (ложные)')
    axes[1,0].set_xlabel('Random State')
    axes[1,0].set_ylabel('Count')
    axes[1,0].set_title('Стабильность ошибок')
    axes[1,0].legend()
    axes[1,0].grid(alpha=0.3)

    # График 4: Нижняя граница Recall по random_state
    axes[1,1].plot(df_stability['random_state'], df_stability['recall_lower'], 'g-', linewidth=2, marker='^', markersize=8)
    axes[1,1].axhline(y=STABILITY_RECALL_LOWER_MIN, color='r', linestyle='--', label=f'Min допустимый: {STABILITY_RECALL_LOWER_MIN}')
    axes[1,1].set_xlabel('Random State')
    axes[1,1].set_ylabel('Recall Lower Bound (95% CI)')
    axes[1,1].set_title('Нижняя граница Recall (худший сценарий)')
    axes[1,1].legend()
    axes[1,1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(MODELLING_PLOTS_PATH, 'finalist_stability.png'), dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"\n💾 График сохранён: {MODELLING_PLOTS_PATH}/finalist_stability.png")

    # Сохраняем CSV
    stability_path = os.path.join(MODELLING_REPORTS_PATH, 'finalist_stability.csv')
    df_stability.to_csv(stability_path, index=False)
    print(f"💾 Данные сохранены: {stability_path}")

    print("\n" + "=" * 70)


🔬 ПРОВЕРКА СТАБИЛЬНОСТИ ФИНАЛИСТА

📋 Проверяемая модель: FN17_L2_C=1
   FN = 17, FP = 1
   Фиксированный порог: 0.0488

🔄 Запуск проверки на разных random_state...
--------------------------------------------------

   random_state = 42... Recall=0.860 (CI: 0.737-0.930), Cost=392, FN=7, FP=273

   random_state = 123... Recall=0.840 (CI: 0.714-0.916), Cost=401, FN=8, FP=265

   random_state = 456... Recall=0.880 (CI: 0.761-0.943), Cost=390, FN=6, FP=288

   random_state = 789... Recall=0.820 (CI: 0.691-0.902), Cost=411, FN=9, FP=258

   random_state = 999... Recall=0.820 (CI: 0.691-0.902), Cost=400, FN=9, FP=247

📊 РЕЗУЛЬТАТЫ СТАБИЛЬНОСТИ ПО ВСЕМ МЕТРИКАМ

📋 ДЕТАЛЬНЫЕ РЕЗУЛЬТАТЫ ПО КАЖДОМУ RANDOM_STATE:
----------------------------------------------------------------------------------------------------
   rs    | Recall | Rec_lower | Rec_upper | Cost | FN | FP | TP | SelRate | NNI
   ------|--------|-----------|-----------|------|----|----|----|---------|-----
     42 | 0.860  | 0.737 

In [26]:
# =============================================================================
# 4.6 СОХРАНЕНИЕ МОДЕЛИ И МЕТАДАННЫХ ДЛЯ CYCLE_COMPARISON
# =============================================================================

print("\n" + "=" * 70)
print("💾 СОХРАНЕНИЕ АРТЕФАКТОВ")
print("=" * 70)

if not HAS_WINNER:
    print("\n⚠️ Нет победителя. Сохраняем только отчёт.")

    # Сохраняем метаданные без модели
    metadata = {
        'project_name': PROJECT_NAME,
        'cycle_number': CYCLE_NUMBER,
        'has_winner': False,
        'created_at': datetime.now().isoformat(),
        'failure_report': WINNER_METADATA,
        'cycle_status': 'NO_MODEL',
        'total_experiments': len(df_results)
    }

    # Сохраняем метаданные
    metadata_path = os.path.join(MODELLING_PATH, 'final_metadata.json')
    with open(metadata_path, 'w', encoding='utf-8') as f:
        json.dump(metadata, f, indent=2, ensure_ascii=False)

    print(f"\n💾 Метаданные сохранены: {metadata_path}")
    print("\n⚠️ Цикл завершён без модели (NO_MODEL)")

else:
    print("\n✅ Есть победитель. Сохраняем модель и метаданные.")

    # 1. ОБУЧЕНИЕ ФИНАЛЬНОЙ МОДЕЛИ НА TRAIN+VAL
    print("\n🔧 Обучение финальной модели на TRAIN+VAL...")

    X_final = pd.concat([X_train, X_val], axis=0)
    y_final = pd.concat([y_train, y_val], axis=0)

    print(f"   Train+Val: {X_final.shape[0]} сэмплов, {X_final.shape[1]} признаков")
    print(f"   Положительных: {y_final.sum()} ({y_final.mean()*100:.2f}%)")

    from sklearn.base import clone

    if best_calibrated_model is not None and CALIBRATION_ENABLED:
        print("   Переобучение калиброванной модели на TRAIN+VAL...")
        final_model = clone(best_base_model)
        final_model.fit(X_final, y_final)
        final_calibrated = CalibratedClassifierCV(final_model, cv=CALIBRATION_CV, method=CALIBRATION_METHOD)
        final_calibrated.fit(X_final, y_final)
        final_model = final_calibrated
        print("   ✅ Калиброванная модель переобучена")
    else:
        final_model = clone(best_base_model)
        final_model.fit(X_final, y_final)
        print("   ✅ Базовая модель обучена")

    # 2. СОХРАНЕНИЕ МОДЕЛИ
    model_path = os.path.join(MODELLING_PATH, 'final_model.joblib')
    joblib.dump(final_model, model_path)

    with open(model_path, 'rb') as f:
        model_checksum = hashlib.md5(f.read()).hexdigest()

    print(f"\n💾 Модель сохранена: {model_path}")
    final_model.threshold_ = final_threshold
    print(f"   Порог {final_threshold:.4f} добавлен как атрибут model.threshold_")

    # 3. ФОРМИРОВАНИЕ МЕТАДАННЫХ
    metadata = {
        'project_name': PROJECT_NAME,
        'cycle_number': CYCLE_NUMBER,
        'has_winner': True,
        'random_state': RANDOM_STATE,
        'created_at': datetime.now().isoformat(),
        'model_name': best_model_name,
        'model_checksum': model_checksum,
        'feature_names': feature_names,
        'n_features': len(feature_names),
        'threshold': float(best_threshold),
        'final_threshold': float(final_threshold),
        'cost_fn': cost_fn,
        'cost_fp': COST_FP,
        'val_metrics': {
            'recall': float(best_row['Val_Recall']),
            'recall_lower_ci': float(best_row.get('Recall_Lower_CI', 0.0)),
            'recall_upper_ci': float(best_row.get('Recall_Upper_CI', 1.0)),
            'precision': float(best_row['Val_Precision']),
            'medical_cost': float(best_row['Medical_Cost']),
            'medical_cost_cv': float(best_row['Medical_Cost_CV']),
            'selection_rate': float(best_row['Selection_Rate']),
            'nni': float(best_row['Val_NNI']),
            'fn': int(best_row['Val_FN']),
            'fp': int(best_row['Val_FP']),
            'tp': int(best_row['Val_TP']),
        },
        'tech_metrics': {
            'pr_auc_mean': float(best_row['PR_AUC_mean']),
            'bss_after': float(best_row['BSS_After']),
            'ece': float(best_row['ECE']),
            'mce': float(best_row['MCE']),
            'stability_gap': float(best_row['Stability_Gap']),
            'medical_cost_cv_coef': float(best_row.get('Medical_Cost_CV_coef', 1.0)),
        },
        'cycle_status': 'VALID',
        'total_experiments': len(df_results)
    }

    # Добавляем результаты проверки стабильности, если есть
    if 'STABILITY_RESULTS' in globals() and STABILITY_RESULTS:
        metadata['stability_check'] = {
            'enabled': True,
            'n_random_states': len(STABILITY_RESULTS['df']),
            'avg_recall': STABILITY_RESULTS['avg_recall'],
            'std_recall': STABILITY_RESULTS['std_recall'],
            'median_recall': STABILITY_RESULTS['median_recall'],
            'min_recall': STABILITY_RESULTS['min_recall'],
            'max_recall': STABILITY_RESULTS['max_recall'],
            'is_stable': STABILITY_RESULTS['is_stable']
        }

    # 4. СОХРАНЕНИЕ МЕТАДАННЫХ
    metadata_path = os.path.join(MODELLING_PATH, 'final_metadata.json')
    with open(metadata_path, 'w', encoding='utf-8') as f:
        json.dump(metadata, f, indent=2, ensure_ascii=False)

    print(f"\n💾 Метаданные сохранены: {metadata_path}")

    # 5. КОПИРОВАНИЕ ТРАНСФОРМЕРОВ И ДАННЫХ (опционально)
    SCALERS_PATH = os.path.join(EDA_TRANSFORMERS_PATH, 'scalers.pkl')
    if os.path.exists(SCALERS_PATH):
        shutil.copy(SCALERS_PATH, os.path.join(MODELLING_PATH, 'scalers.pkl'))
        print(f"\n💾 Scalers сохранены")

    # 6. ОБНОВЛЕНИЕ latest_cycle.txt
    latest_link_path = os.path.join(PROJECT_PATH, 'latest_cycle.txt')
    with open(latest_link_path, 'w') as f:
        f.write(str(CYCLE_NUMBER))
    print(f"\n📁 latest_cycle.txt обновлён: {CYCLE_NUMBER}")

    # 7. ИТОГ
    print("\n" + "=" * 70)
    print("✅ АРТЕФАКТЫ СОХРАНЕНЫ")
    print("=" * 70)
    print(f"\n📂 Папка цикла: {MODELLING_PATH}")
    print(f"\n📊 Метрики на VAL:")
    print(f"   • Recall:    {metadata['val_metrics']['recall']:.3f}")
    print(f"   • Cost (CV): {metadata['val_metrics']['medical_cost_cv']:.0f}")
    print(f"\n🎯 Порог для TEST: {final_threshold:.4f}")

print("\n" + "=" * 70)


💾 СОХРАНЕНИЕ АРТЕФАКТОВ

✅ Есть победитель. Сохраняем модель и метаданные.

🔧 Обучение финальной модели на TRAIN+VAL...
   Train+Val: 4042 сэмплов, 10 признаков
   Положительных: 199 (4.92%)
   Переобучение калиброванной модели на TRAIN+VAL...
   ✅ Калиброванная модель переобучена

💾 Модель сохранена: /content/drive/MyDrive/ml_learning/datasets/stroke/cycle_5/modelling/final_model.joblib
   Порог 0.0463 добавлен как атрибут model.threshold_

💾 Метаданные сохранены: /content/drive/MyDrive/ml_learning/datasets/stroke/cycle_5/modelling/final_metadata.json

💾 Scalers сохранены

📁 latest_cycle.txt обновлён: 5

✅ АРТЕФАКТЫ СОХРАНЕНЫ

📂 Папка цикла: /content/drive/MyDrive/ml_learning/datasets/stroke/cycle_5/modelling

📊 Метрики на VAL:
   • Recall:    0.860
   • Cost (CV): 229

🎯 Порог для TEST: 0.0463



# БЛОК 5. АНАЛИЗ ЛУЧШЕЙ МОДЕЛИ

In [27]:
# =============================================================================
# 5. УГЛУБЛЁННЫЙ АНАЛИЗ ЛУЧШЕЙ МОДЕЛИ (ТОЛЬКО НА VAL И TRAIN)
# =============================================================================

if not HAS_WINNER:
    print("\n⚠️ Нет победителя, пропускаем углублённый анализ")
else:
    print("\n" + "=" * 70)
    print("🔬 УГЛУБЛЁННЫЙ АНАЛИЗ ПОБЕДИТЕЛЯ")
    print("=" * 70)
    print(f"Модель: {best_model_name}")
    print("=" * 70)

    # -------------------------------------------------------------------------
    # 5.1 СТАТИСТИЧЕСКАЯ ЗНАЧИМОСТЬ КОЭФФИЦИЕНТОВ
    # -------------------------------------------------------------------------
    print("\n" + "-" * 70)
    print("5.1 СТАТИСТИЧЕСКАЯ ЗНАЧИМОСТЬ КОЭФФИЦИЕНТОВ")
    print("-" * 70)

    df_coef = None

    if hasattr(best_base_model, 'coef_'):
        try:
            df_coef = get_coefficient_significance(
                model=best_base_model,
                X_train=X_train,
                y_train=y_train,
                feature_names=feature_names,
                alpha=0.05
            )

            # Для L1-модели p-values нет, выводим только коэффициенты
            if 'p_value' in df_coef.columns:
                print("\n📋 ЗНАЧИМЫЕ ПРИЗНАКИ (p < 0.05):")
                significant = df_coef[df_coef['p_value'] < 0.05]
                print(f"   Всего значимых: {len(significant)} из {len(df_coef)}")
                for _, row in significant.head(10).iterrows():
                    print(f"   • {row['Feature']}: OR = {row['OR']:.3f} [{row['CI_lower_95']:.2f}-{row['CI_upper_95']:.2f}] {row['Signif']}")
                if len(significant) > 10:
                    print(f"   ... и ещё {len(significant) - 10} признаков")
            else:
                print("\n📋 ВАЖНОСТЬ ПРИЗНАКОВ (L1-модель, p-values не определены):")
                print(f"   Всего признаков: {len(df_coef)}")
                for _, row in df_coef.head(10).iterrows():
                    print(f"   • {row['Feature']}: OR = {row['OR']:.3f} — {row['Interpretation']}")

            coef_path = os.path.join(MODELLING_REPORTS_PATH, 'coefficients_stats.csv')
            df_coef.to_csv(coef_path, index=False)
            print(f"\n💾 Коэффициенты сохранены: {coef_path}")

            forest_path = os.path.join(MODELLING_PLOTS_PATH, 'forest_plot.png')
            fig = plot_forest_plot(df_coef, title=f"Odds Ratios - {best_model_name}", save_path=forest_path)
            plt.close(fig)
            print(f"   ✅ Forest plot сохранён: {forest_path}")

            coef_imp_path = os.path.join(MODELLING_PLOTS_PATH, 'coefficient_importance.png')
            fig = plot_coefficient_importance(df_coef, top_n=15, save_path=coef_imp_path)
            plt.close(fig)
            print(f"   ✅ Coefficient importance сохранён: {coef_imp_path}")

        except Exception as e:
            print(f"⚠️ Ошибка при расчёте статистической значимости: {e}")
    else:
        print("⚠️ Модель не поддерживает коэффициенты (возможно Dummy)")

    # -------------------------------------------------------------------------
    # 5.2 SHAP-АНАЛИЗ
    # -------------------------------------------------------------------------
    # При автоматизации закомментируйте этот блок.

    print("\n" + "-" * 70)
    print("5.2 SHAP-АНАЛИЗ")
    print("-" * 70)
    print("ℹ️ SHAP применяется к базовой (некалиброванной) модели")
    print("ℹ️ SHAP значения в пространстве log-odds (LinearExplainer)")
    print("   Положительное значение → увеличение log-odds риска")
    print("   Отрицательное значение → снижение log-odds риска")

    try:
        import shap

        if hasattr(best_base_model, 'coef_'):
            sample_size = min(500, len(X_train))
            X_sample = X_train.sample(sample_size, random_state=RANDOM_STATE)

            explainer = shap.LinearExplainer(best_base_model, X_sample)
            shap_values = explainer.shap_values(X_val)

            # SHAP Summary Plot (beeswarm)
            shap.summary_plot(shap_values, X_val, feature_names=feature_names, show=False)
            fig = plt.gcf()
            fig.set_size_inches(10, 8)
            plt.title(f"SHAP Summary - {best_model_name}")
            plt.tight_layout()
            plt.savefig(os.path.join(MODELLING_PLOTS_PATH, 'shap_summary.png'), dpi=150, bbox_inches='tight')
            plt.close(fig)
            print(f"   ✅ SHAP summary сохранён: {MODELLING_PLOTS_PATH}/shap_summary.png")

            # SHAP Bar Plot (feature importance)
            shap.summary_plot(shap_values, X_val, feature_names=feature_names, plot_type="bar", show=False)
            fig = plt.gcf()
            fig.set_size_inches(10, 8)
            plt.title(f"SHAP Feature Importance - {best_model_name}")
            plt.tight_layout()
            plt.savefig(os.path.join(MODELLING_PLOTS_PATH, 'shap_importance.png'), dpi=150, bbox_inches='tight')
            plt.close(fig)
            print(f"   ✅ SHAP importance сохранён: {MODELLING_PLOTS_PATH}/shap_importance.png")

            # Числовой вывод важности признаков
            shap_importance = pd.DataFrame({
                'Feature': feature_names,
                'Mean_|SHAP|': np.abs(shap_values).mean(axis=0),
                'Mean_SHAP': shap_values.mean(axis=0),
                'Std_SHAP': shap_values.std(axis=0)
            }).sort_values('Mean_|SHAP|', ascending=False)

            print("\n📋 ТОП-10 ПРИЗНАКОВ ПО SHAP:")
            for _, row in shap_importance.head(10).iterrows():
                direction = "↑ риск" if row['Mean_SHAP'] > 0 else "↓ риск"
                print(f"   • {row['Feature']}: |SHAP|={row['Mean_|SHAP|']:.4f}, {direction}")

            shap_path = os.path.join(MODELLING_REPORTS_PATH, 'shap_importance.csv')
            shap_importance.to_csv(shap_path, index=False)
            print(f"\n💾 SHAP важность сохранена: {shap_path}")

            print("\n✅ SHAP анализ завершён")
        else:
            print("⚠️ SHAP: модель не имеет coef_ (Dummy)")

    except ImportError:
        print("⚠️ SHAP не установлен. Установите shap для расширенного анализа")
    except Exception as e:
        print(f"⚠️ Ошибка SHAP: {e}")

    # -------------------------------------------------------------------------
    # 5.3 АНАЛИЗ ОШИБОК (FN vs TP и FP)
    # -------------------------------------------------------------------------
    print("\n" + "-" * 70)
    print("5.3 АНАЛИЗ ОШИБОК (FN vs TP и FP)")
    print("-" * 70)

    error_profiles = analyze_error_profiles(
        X_val=X_val,
        y_true=best_y_val,
        y_probs=best_probs_calib,
        threshold=best_threshold,
        feature_names=feature_names,
        model_name=best_model_name,
        target_label=TARGET_LABEL,
        save_path=os.path.join(MODELLING_REPORTS_PATH, 'error_profiles.csv') if 'MODELLING_REPORTS_PATH' in globals() else None
    )

    # -------------------------------------------------------------------------
    # 5.4 РАСПРЕДЕЛЕНИЕ ВЕРОЯТНОСТЕЙ
    # -------------------------------------------------------------------------
    print("\n" + "-" * 70)
    print("5.4 РАСПРЕДЕЛЕНИЕ ВЕРОЯТНОСТЕЙ")
    print("-" * 70)

    fig = plot_probability_histogram(
        y_true=best_y_val,
        y_probs=best_probs_calib,
        model_name=best_model_name,
        save_path=os.path.join(MODELLING_PLOTS_PATH, 'probability_histogram.png')
    )
    plt.close(fig)
    print(f"   ✅ Гистограмма сохранена: {MODELLING_PLOTS_PATH}/probability_histogram.png")

    # -------------------------------------------------------------------------
    # 5.5 РЕКОМЕНДАЦИИ ДЛЯ СЛЕДУЮЩЕГО ЦИКЛА
    # -------------------------------------------------------------------------
    print("\n" + "-" * 70)
    print("5.5 РЕКОМЕНДАЦИИ ДЛЯ EDA")
    print("-" * 70)

    recommendations = []

    # Слабые признаки (только если есть p-values)
    if df_coef is not None and 'p_value' in df_coef.columns:
        weak_features = df_coef[df_coef['p_value'] >= 0.05]['Feature'].tolist()
        if weak_features:
            recommendations.append(f"Рассмотреть удаление признаков: {', '.join(weak_features[:5])} (p ≥ 0.05)")

    # Анализ ошибок (FN vs TP)
    if error_profiles is not None and len(error_profiles) > 0:
        top_diff = error_profiles.iloc[0]
        recommendations.append(f"FN имеют более высокий '{top_diff['feature']}' — возможно, нужен interaction или дополнительный признак")

    # Анализ FP (если есть данные)
    if error_profiles is not None and 'Difference_FP_TP' in error_profiles.columns:
        fp_analysis = error_profiles.nlargest(3, 'Abs_Difference_FP_TP')
        for _, row in fp_analysis.iterrows():
            if pd.notna(row.get('Difference_FP_TP', None)):
                recommendations.append(f"FP имеют отклонение по '{row['feature']}' — возможно, стоит добавить признак для различения")

    # Сохраняем рекомендации
    recommendations_path = os.path.join(MODELLING_PATH, 'eda_recommendations.json')
    with open(recommendations_path, 'w', encoding='utf-8') as f:
        json.dump({
            'cycle': CYCLE_NUMBER,
            'model': best_model_name,
            'random_state': RANDOM_STATE,
            'recommendations': recommendations
        }, f, indent=2, ensure_ascii=False)

    print("\n📌 РЕКОМЕНДАЦИИ:")
    if recommendations:
        for rec in recommendations:
            print(f"   • {rec}")
    else:
        print("   • Нет конкретных рекомендаций. Модель работает хорошо.")

    print(f"\n💾 Сохранено: {recommendations_path}")

    print("\n" + "=" * 70)
    print("✅ УГЛУБЛЁННЫЙ АНАЛИЗ ЗАВЕРШЁН")
    print("=" * 70)


🔬 УГЛУБЛЁННЫЙ АНАЛИЗ ПОБЕДИТЕЛЯ
Модель: FN17_L2_C=1

----------------------------------------------------------------------
5.1 СТАТИСТИЧЕСКАЯ ЗНАЧИМОСТЬ КОЭФФИЦИЕНТОВ
----------------------------------------------------------------------

ℹ️ Penalty='l2'. p-values рассчитываются через Fisher Information.
   ВНИМАНИЕ: Это ПРИБЛИЖЁННЫЕ значения. Для регуляризованных моделей
   p-values НЕ являются строго статистически корректными.
   Используйте их только для ОРИЕНТИРОВОЧНОЙ оценки важности признаков.


📋 ЗНАЧИМЫЕ ПРИЗНАКИ (p < 0.05):
   Всего значимых: 9 из 10
   • work_type_self-employed: OR = 0.103 [0.05-0.20] ***
   • age: OR = 7.533 [5.97-9.50] ***
   • work_type_govt_job: OR = 0.156 [0.08-0.30] ***
   • work_type_private: OR = 0.171 [0.09-0.31] ***
   • bmi_missing_flag: OR = 5.807 [3.39-9.95] ***
   • stable_old_age: OR = 0.513 [0.38-0.70] ***
   • cardio_risk: OR = 1.393 [1.07-1.80] *
   • smoking_age_impact: OR = 1.266 [1.11-1.44] ***
   • avg_glucose_level: OR = 1.196 [1.11-1

In [28]:
# =============================================================================
# 5.6 SENSITIVITY ANALYSIS (устойчивость к удалению данных)
# =============================================================================

if not HAS_WINNER:
    print("\n⚠️ Нет победителя, пропускаем Sensitivity Analysis")
else:
    print("\n" + "=" * 70)
    print("🔍 SENSITIVITY ANALYSIS: проверка устойчивости модели")
    print("=" * 70)

    SENSITIVITY_ITERATIONS = 10      # Сколько раз проверяем
    SENSITIVITY_DROP_PERCENT = 5     # Сколько процентов данных выкидываем

    print(f"   Параметры: {SENSITIVITY_ITERATIONS} итераций, удаляем {SENSITIVITY_DROP_PERCENT}% данных")
    print(f"   ⏳ Выполняется...")

    recalls = []
    precisions = []
    costs = []

    for i in range(SENSITIVITY_ITERATIONS):
        # Случайно удаляем строки из валидации
        n_drop = int(len(X_val) * SENSITIVITY_DROP_PERCENT / 100)
        drop_idx = np.random.choice(len(X_val), size=n_drop, replace=False)

        X_val_subset = X_val.drop(index=X_val.index[drop_idx])
        y_val_subset = y_val.drop(index=y_val.index[drop_idx])

        # Предсказания на подвыборке
        if best_calibrated_model is not None:
            probs_subset = best_calibrated_model.predict_proba(X_val_subset)[:, 1]
        else:
            probs_subset = best_base_model.predict_proba(X_val_subset)[:, 1]

        # Метрики
        metrics = get_metrics_at_threshold(y_val_subset, probs_subset, final_threshold, COST_FN, COST_FP)

        recalls.append(metrics['recall'])
        precisions.append(metrics['precision'])
        costs.append(metrics['medical_cost'])

    # Результаты
    recalls = np.array(recalls)
    precisions = np.array(precisions)
    costs = np.array(costs)

    print(f"\n📊 РЕЗУЛЬТАТЫ SENSITIVITY ANALYSIS:")
    print(f"   Recall:    среднее = {recalls.mean():.4f}, мин = {recalls.min():.4f}, макс = {recalls.max():.4f}")
    print(f"   Precision: среднее = {precisions.mean():.4f}, мин = {precisions.min():.4f}, макс = {precisions.max():.4f}")
    print(f"   Cost:      среднее = {costs.mean():.0f}, мин = {costs.min():.0f}, макс = {costs.max():.0f}")

    recall_drop = (best_row['Val_Recall'] - recalls.min()) / best_row['Val_Recall'] * 100
    print(f"\n   📉 Худший сценарий для Recall: падение на {recall_drop:.1f}%")

    if recall_drop < 10:
        print(f"   ✅ Модель устойчива! Recall падает менее чем на 10% при удалении {SENSITIVITY_DROP_PERCENT}% данных.")
    elif recall_drop < 20:
        print(f"   ⚠️ Умеренная чувствительность. Рекомендуется собрать больше данных.")
    else:
        print(f"   🔴 Высокая чувствительность! Модель сильно зависит от отдельных строк данных.")

    # Сохраняем результаты
    sensitivity_path = os.path.join(MODELLING_REPORTS_PATH, 'sensitivity_analysis.csv')
    pd.DataFrame({
        'iteration': range(1, SENSITIVITY_ITERATIONS + 1),
        'recall': recalls,
        'precision': precisions,
        'medical_cost': costs
    }).to_csv(sensitivity_path, index=False)
    print(f"\n💾 Результаты сохранены: {sensitivity_path}")


🔍 SENSITIVITY ANALYSIS: проверка устойчивости модели
   Параметры: 10 итераций, удаляем 5% данных
   ⏳ Выполняется...

📊 РЕЗУЛЬТАТЫ SENSITIVITY ANALYSIS:
   Recall:    среднее = 0.8608, мин = 0.8542, макс = 0.8750
   Precision: среднее = 0.1274, мин = 0.1254, макс = 0.1292
   Cost:      среднее = 404, мин = 389, макс = 412

   📉 Худший сценарий для Recall: падение на 0.7%
   ✅ Модель устойчива! Recall падает менее чем на 10% при удалении 5% данных.

💾 Результаты сохранены: /content/drive/MyDrive/ml_learning/datasets/stroke/cycle_5/modelling/reports/sensitivity_analysis.csv


In [29]:
# =============================================================================
# 5.7 PERMUTATION IMPORTANCE (проверка синтетических признаков)
# =============================================================================

if not HAS_WINNER:
    print("\n⚠️ Нет победителя, пропускаем Permutation Importance")
else:
    print("\n" + "=" * 70)
    print("🔄 PERMUTATION IMPORTANCE: проверка значимости признаков")
    print("=" * 70)

    print("   Суть: перемешиваем признак и смотрим, как изменилась Medical Cost")
    print("   Если Cost почти не изменился → признак бесполезен (шум)")
    print("   ⏳ Выполняется...")

    from sklearn.metrics import confusion_matrix

    def calculate_cost_for_model(model, X, y_true, threshold, cost_fn=COST_FN, cost_fp=COST_FP):
        """Быстрый расчёт Medical Cost для модели на данных"""
        probs = model.predict_proba(X)[:, 1]
        y_pred = (probs >= threshold).astype(int)
        cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
        if cm.shape != (2, 2):
            return float('inf')
        tn, fp, fn, tp = cm.ravel()
        return fn * cost_fn + fp * cost_fp

    # Базовая стоимость
    baseline_cost = calculate_cost_for_model(
        best_base_model, X_val, best_y_val, final_threshold
    )

    print(f"\n📊 Базовая Medical Cost (без перемешивания): {baseline_cost:.0f}")

    # Перебираем признаки
    permutation_results = []

    for feature in feature_names:
        X_val_permuted = X_val.copy()
        X_val_permuted[feature] = np.random.permutation(X_val_permuted[feature].values)

        permuted_cost = calculate_cost_for_model(
            best_base_model, X_val_permuted, best_y_val, final_threshold
        )

        cost_increase = permuted_cost - baseline_cost
        cost_increase_pct = (cost_increase / baseline_cost) * 100

        permutation_results.append({
            'feature': feature,
            'cost_increase': cost_increase,
            'cost_increase_pct': cost_increase_pct,
            'importance': 'high' if cost_increase_pct > 10 else 'medium' if cost_increase_pct > 3 else 'low'
        })

    df_permutation = pd.DataFrame(permutation_results).sort_values('cost_increase_pct', ascending=False)

    print(f"\n📋 ВАЖНОСТЬ ПРИЗНАКОВ (по влиянию на Medical Cost):")
    print("-" * 60)

    for _, row in df_permutation.head(10).iterrows():
        if row['importance'] == 'high':
            emoji = "🔴"
        elif row['importance'] == 'medium':
            emoji = "🟡"
        else:
            emoji = "🟢"
        print(f"   {emoji} {row['feature']}: +{row['cost_increase_pct']:.1f}% к Cost")

    low_importance_features = df_permutation[df_permutation['importance'] == 'low']['feature'].tolist()
    if low_importance_features:
        print(f"\n⚠️ ПРИЗНАКИ С НИЗКОЙ ВАЖНОСТЬЮ (<3% влияния):")
        for f in low_importance_features[:5]:
            print(f"   • {f}")
        if len(low_importance_features) > 5:
            print(f"   ... и ещё {len(low_importance_features) - 5}")

    # Сохраняем
    permutation_path = os.path.join(MODELLING_REPORTS_PATH, 'permutation_importance.csv')
    df_permutation.to_csv(permutation_path, index=False)
    print(f"\n💾 Результаты сохранены: {permutation_path}")


🔄 PERMUTATION IMPORTANCE: проверка значимости признаков
   Суть: перемешиваем признак и смотрим, как изменилась Medical Cost
   Если Cost почти не изменился → признак бесполезен (шум)
   ⏳ Выполняется...

📊 Базовая Medical Cost (без перемешивания): 804

📋 ВАЖНОСТЬ ПРИЗНАКОВ (по влиянию на Medical Cost):
------------------------------------------------------------
   🔴 age: +23.1% к Cost
   🟢 smoking_age_impact: +2.7% к Cost
   🟢 avg_glucose_level: +2.2% к Cost
   🟢 cardio_risk: +1.1% к Cost
   🟢 bmi_missing_flag: +-0.1% к Cost
   🟢 work_type_never_worked: +-0.2% к Cost
   🟢 work_type_govt_job: +-0.6% к Cost
   🟢 work_type_self-employed: +-4.1% к Cost
   🟢 stable_old_age: +-4.7% к Cost
   🟢 work_type_private: +-5.6% к Cost

⚠️ ПРИЗНАКИ С НИЗКОЙ ВАЖНОСТЬЮ (<3% влияния):
   • smoking_age_impact
   • avg_glucose_level
   • cardio_risk
   • bmi_missing_flag
   • work_type_never_worked
   ... и ещё 4

💾 Результаты сохранены: /content/drive/MyDrive/ml_learning/datasets/stroke/cycle_5/modelli

In [30]:
# =============================================================================
# 5.9 LEARNING CURVE (эффект обучения)
# =============================================================================

if not HAS_WINNER:
    print("\n⚠️ Нет победителя, пропускаем Learning Curve")
else:
    print("\n" + "=" * 70)
    print("📈 LEARNING CURVE: хватает ли данных?")
    print("=" * 70)

    print("   Суть: обучаем финалиста на 50%, 75%, 100% данных")
    print("   Если метрики всё ещё растут → нужно больше данных")
    print("   ⏳ Выполняется...")

    from sklearn.base import clone

    fractions = [0.5, 0.75, 1.0]
    learning_results = []

    for frac in fractions:
        n_samples = int(len(X_train) * frac)
        X_subset = X_train.iloc[:n_samples]
        y_subset = y_train.iloc[:n_samples]

        # Клонируем и обучаем
        temp_model = clone(best_base_model)
        temp_model.fit(X_subset, y_subset)

        # Калибруем (если нужно)
        if CALIBRATION_ENABLED and is_calibration_needed(temp_model):
            temp_calib = CalibratedClassifierCV(temp_model, cv=CALIBRATION_CV, method=CALIBRATION_METHOD)
            temp_calib.fit(X_subset, y_subset)
            val_probs = temp_calib.predict_proba(X_val)[:, 1]
        else:
            val_probs = temp_model.predict_proba(X_val)[:, 1]

        # Метрики
        metrics = get_metrics_at_threshold(y_val, val_probs, final_threshold, COST_FN, COST_FP)

        learning_results.append({
            'fraction': frac,
            'n_samples': n_samples,
            'recall': metrics['recall'],
            'precision': metrics['precision'],
            'medical_cost': metrics['medical_cost']
        })

    df_learning = pd.DataFrame(learning_results)

    print(f"\n📊 РЕЗУЛЬТАТЫ LEARNING CURVE:")
    print("-" * 70)
    print(f"   Доля данных | N_samples | Recall  | Precision | Medical Cost")
    print(f"   ------------|-----------|---------|-----------|-------------")

    for _, row in df_learning.iterrows():
        n_samples_val = int(row['n_samples']) if pd.notna(row['n_samples']) else 0
        recall_val = row['recall'] if pd.notna(row['recall']) else 0.0
        precision_val = row['precision'] if pd.notna(row['precision']) else 0.0
        cost_val = row['medical_cost'] if pd.notna(row['medical_cost']) else 0.0

        print(f"   {row['fraction']*100:5.0f}%        | {n_samples_val:6d}   | {recall_val:.4f} | {precision_val:.4f}    | {cost_val:.0f}")

    # Анализ
    if len(df_learning) >= 2:
        recall_50 = df_learning[df_learning['fraction'] == 0.5]['recall'].values[0]
        recall_100 = df_learning[df_learning['fraction'] == 1.0]['recall'].values[0]

        cost_50 = df_learning[df_learning['fraction'] == 0.5]['medical_cost'].values[0]
        cost_100 = df_learning[df_learning['fraction'] == 1.0]['medical_cost'].values[0]

        if recall_50 > 0:
            recall_growth = (recall_100 - recall_50) / recall_50 * 100
        else:
            recall_growth = 0

        print(f"\n📈 Рост Recall при увеличении данных с 50% до 100%: +{recall_growth:.1f}%")

        if recall_growth > 10:
            print(f"   🔴 Модель всё ещё учится! Рекомендуется собрать больше данных.")
        elif recall_growth > 5:
            print(f"   ⚠️ Умеренный рост. Дополнительные данные могут помочь.")
        else:
            print(f"   ✅ Модель вышла на плато. Увеличение данных вряд ли сильно улучшит качество.")

        # Анализ по Medical Cost
        cost_reduction = (cost_50 - cost_100) / cost_50 * 100 if cost_50 > 0 else 0
        print(f"📉 Снижение Medical Cost: {cost_reduction:.1f}%")
    else:
        print(f"\n⚠️ Недостаточно данных для анализа learning curve")

    # График Learning Curve
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(df_learning['n_samples'], df_learning['recall'], 'bo-', linewidth=2, markersize=8)
    axes[0].set_xlabel('Number of training samples')
    axes[0].set_ylabel('Recall')
    axes[0].set_title('Learning Curve: Recall')
    axes[0].grid(alpha=0.3)

    axes[1].plot(df_learning['n_samples'], df_learning['medical_cost'], 'ro-', linewidth=2, markersize=8)
    axes[1].set_xlabel('Number of training samples')
    axes[1].set_ylabel('Medical Cost')
    axes[1].set_title('Learning Curve: Medical Cost')
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(MODELLING_PLOTS_PATH, 'learning_curve.png'), dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"\n💾 График сохранён: {MODELLING_PLOTS_PATH}/learning_curve.png")

    # Сохраняем данные
    df_learning.to_csv(os.path.join(MODELLING_REPORTS_PATH, 'learning_curve.csv'), index=False)
    print(f"💾 Данные сохранены: {MODELLING_REPORTS_PATH}/learning_curve.csv")

    print("\n" + "=" * 70)

    # =========================================================================
    # ДИАГНОСТИКА LEARNING CURVE: доверительные интервалы
    # =========================================================================
    print("\n" + "=" * 70)
    print("🔍 ПРОВЕРКА СТАБИЛЬНОСТИ LEARNING CURVE")
    print("=" * 70)

    from scipy import stats

    # Данные из вашей таблицы
    data = {
        0.50: {'recall': 0.94, 'n_samples': 1515, 'n_positives': int(1515 * 0.0492) if 'y_train' in globals() else 75},
        0.75: {'recall': 0.92, 'n_samples': 2273, 'n_positives': int(2273 * 0.0492) if 'y_train' in globals() else 112},
        1.00: {'recall': 0.92, 'n_samples': 3031, 'n_positives': int(3031 * 0.0492) if 'y_train' in globals() else 149},
    }

    # Рассчитываем доверительные интервалы для Recall
    print("\n📊 ДОВЕРИТЕЛЬНЫЕ ИНТЕРВАЛЫ ДЛЯ RECALL (95%):")
    print("-" * 60)
    print(f"   Доля данных | Recall | 95% CI нижн. | 95% CI верхн. | Размах")
    print(f"   ------------|--------|--------------|---------------|-------")

    for frac, vals in data.items():
        n = vals['n_positives']
        recall = vals['recall']

        if n > 0:
            se = np.sqrt(recall * (1 - recall) / n)
            ci_lower = recall - 1.96 * se
            ci_upper = recall + 1.96 * se
            ci_lower = max(0, ci_lower)
            ci_upper = min(1, ci_upper)
            ci_range = ci_upper - ci_lower

            print(f"   {frac*100:5.0f}%        | {recall:.3f}   | {ci_lower:.3f}        | {ci_upper:.3f}         | {ci_range:.3f}")
        else:
            print(f"   {frac*100:5.0f}%        | {recall:.3f}   | N/A          | N/A           | N/A")

    # Статистический тест: отличается ли Recall 50% от 100%?
    recall_50 = data[0.50]['recall']
    recall_100 = data[1.00]['recall']
    n_50 = data[0.50]['n_positives']
    n_100 = data[1.00]['n_positives']

    if n_50 > 0 and n_100 > 0:
        p_pooled = (recall_50 * n_50 + recall_100 * n_100) / (n_50 + n_100)
        se_diff = np.sqrt(p_pooled * (1 - p_pooled) * (1/n_50 + 1/n_100))
        z_score = (recall_50 - recall_100) / se_diff if se_diff > 0 else 0
        p_value = 2 * (1 - stats.norm.cdf(abs(z_score)))

        print(f"\n📊 СТАТИСТИЧЕСКИЙ ТЕСТ (Recall 50% vs 100%):")
        print(f"   Z-оценка: {z_score:.3f}")
        print(f"   P-значение: {p_value:.4f}")

        if p_value < 0.05:
            print(f"   ✅ Разница СТАТИСТИЧЕСКИ ЗНАЧИМА (p < 0.05)")
            print(f"      → Модель действительно улучшилась/ухудшилась")
        else:
            print(f"   ⚠️ Разница НЕ СТАТИСТИЧЕСКИ ЗНАЧИМА (p >= 0.05)")
            print(f"      → Recall 0.94 и 0.92 — это В ПРЕДЕЛАХ СТАТИСТИЧЕСКОЙ ПОГРЕШНОСТИ")
            print(f"      → Модель достигла плато")


📈 LEARNING CURVE: хватает ли данных?
   Суть: обучаем финалиста на 50%, 75%, 100% данных
   Если метрики всё ещё растут → нужно больше данных
   ⏳ Выполняется...

📊 РЕЗУЛЬТАТЫ LEARNING CURVE:
----------------------------------------------------------------------
   Доля данных | N_samples | Recall  | Precision | Medical Cost
   ------------|-----------|---------|-----------|-------------
      50%        |   1515   | 0.8600 | 0.1225    | 434
      75%        |   2273   | 0.8600 | 0.1225    | 434
     100%        |   3031   | 0.8600 | 0.1261    | 424

📈 Рост Recall при увеличении данных с 50% до 100%: +0.0%
   ✅ Модель вышла на плато. Увеличение данных вряд ли сильно улучшит качество.
📉 Снижение Medical Cost: 2.3%

💾 График сохранён: /content/drive/MyDrive/ml_learning/datasets/stroke/cycle_5/modelling/plots/learning_curve.png
💾 Данные сохранены: /content/drive/MyDrive/ml_learning/datasets/stroke/cycle_5/modelling/reports/learning_curve.csv


🔍 ПРОВЕРКА СТАБИЛЬНОСТИ LEARNING CURVE

📊 ДО